# Token-chunk head with original CE/metric tiles — COMPONENT ONLY

Previous target-passed native-CE token chunking saved69.79% isolated allocated component memory at B4/T512 (5515.90→1666.45MiB), but slowed warm head+required-metrics+adjoint224.99→271.90ms (+20.85%). This successor addresses duplicate vocabulary work, not a blind chunk-size search: token chunk remains128, and the original eight CE/mixed/summary kernel bodies remain byte-identical.

The candidate retains only small per-row forward max/logsum/route tensors, not logits or gradients. Backward recomputes the SAME half head tile from owned half snapshots, then applies the original CE backward after the REAL global-normalized upstream scalar arrives. Mixed near-certain rows use native replay and disjoint insertion after FP16 narrowing. Global loss uses the full-row native NLL SUM order followed by ONE original denominator, never per-chunk normalization. Frozen half snapshots, target/denominator/saved-state mutation guards, full-V adjoints and original common consumer are preserved. No duplicate consumer or model/Trainer installer is installed.

Predeclared numerical gates for this new original-CE integration match that producer's prior contract: entropy rtol0.01/atol1e-6; main loss, token NLL and Rewind rows rtol1e-4/atol1e-6; accuracy/first argmax/counts/NaN/Inf classes exact. Hidden gradient retains the predecessor gate relativeL2<=0.001 and maxabs<=0.005*reference max+1e-7. These are engineering gates, NOT official task-quality checks. The native-only predecessor used tighter native-op metric gates; no prior target result is being relabeled.

A bounded direct tile gate compares10 actual-vocabulary cases against the frozen CE producer, plus a full-V scale-before-half negative. Seven complete nativeAMP full-H/V cases include ordinary T133/T512, arbitrary upstream, all-ignored labels, mixed near-certain rows, a masked nonfinite final row and actual GradScaler overflow. Every repeated raw tile is checked bitwise between forward autocast FP16 and backward outside autocast before accepting saved-normalizer reuse. Native fallback still supplies all metrics, including masked-row NaN propagation.32 owner CPU tests and40 tile CPU/interpreter tests passed; those are NOT target CUDA execution.

Native, previous native-CE token chunks, and new original-CE token chunks each measure in separate fresh processes with unchanged80s caps. Both B4/T133 andB4/T512 report cold synchronized wall, three warm stream-event samples, first-call and all-observed allocator peaks. Fixed method order and short samples are disclosed. No task switch/precision flag changes, downloads, installs, prewarmed cache or retries. Runtime/source/lock/pidfd/subreaper/quarantine/export helpers remain unchanged.

These are synthetic head COMPONENT measurements, not full model/training memory,29s/3GB evidence, score, task-speed improvement, useful-owned-kernel share or admission. No production integration is bundled. Original useful full-task share and unchanged64-step quality/output/peak/time gates remain future requirements. Parent alone submits after independent review.


In [ ]:
import os,base64,hashlib,tempfile
from pathlib import Path
assert os.environ.get('SOUP_REPO') and os.environ.get('SOUP_RESULTS')
assert Path('/kaggle/working').is_dir()
PAYLOAD = {'solution/__init__.py': 'IiIiT3JpZ2luYWwgdG9rZW4tY2h1bmsgQ0UgY29tcG9uZW50LCBubyBhdXRvbWF0aWMgaW5zdGFsbGF0aW9uLiIiIgo=', 'solution/ce_tile.py': 'IiIiT3JpZ2luYWwgQ0Uvc3VtbWFyeSB0aWxlIGFkYXB0ZXIgZm9yIGFuIG93bmVkIHRva2VuLWF4aXMgaGVhZCByZWNvbXB1dGF0aW9uLgoKTm8gY2F1c2FsIHNoaWZ0aW5nLCBzY2FsYXIgcmVkdWN0aW9uLCBkaXZpc2lvbiwgbW9kZWwgaW5zdGFsbGF0aW9uIG9yIGhpZGRlbgphdXRvZ3JhZCBncmFwaC4gT3JpZ2luYWwga2VybmVscyBhcmUgaW1wb3J0ZWQgYnl0ZS1pZGVudGljYWxseS4gVGhlIG93bmVyIG11c3QKcHJvdmUgaWRlbnRpY2FsIHJlY29tcHV0ZWQgRlAxNiBsb2dpdHMgYW5kIHNhdmUgZXZlcnkgc3RhdGUgdGVuc29yIHRocm91Z2gKY3R4LnNhdmVfZm9yX2JhY2t3YXJkLiBObyBsb2dpdHMgb3IgZ3JhZGllbnRzIGFyZSByZXRhaW5lZCBpbiBzdGF0ZS4KIiIiCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIC5vcmlnaW5hbCBpbXBvcnQgbWF0ZXJpYWxpemVkX2NlIGFzIGNlCmZyb20gLm9yaWdpbmFsIGltcG9ydCBtaXhlZF9yb3dzIGFzIG1peGVkCmZyb20gLm9yaWdpbmFsLm1ldHJpY19zdW1tYXJ5IGltcG9ydCBkZXRhY2hlZF9tZXRyaWNzCgpWT0NBQlVMQVJZID0gMTUxOTM2CgoKZGVmIHN0YW1wKHZhbHVlKToKICAgIHJldHVybiAodHVwbGUodmFsdWUuc2hhcGUpLHR1cGxlKHZhbHVlLnN0cmlkZSgpKSx2YWx1ZS5kdHlwZSx2YWx1ZS5kZXZpY2UsCiAgICAgICAgICAgIHZhbHVlLnN0b3JhZ2Vfb2Zmc2V0KCksdmFsdWUuX3ZlcnNpb24pCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgVGlsZVBsYW46CiAgICByb3V0ZTogc3RyCiAgICByZWFzb246IHN0ciB8IE5vbmUKICAgIHJvd3M6IGludAogICAgdm9jYWJ1bGFyeTogaW50CiAgICB0YXJnZXRfc3RhbXA6IHR1cGxlCiAgICBzYXZlZF9zdGFtcHM6IHR1cGxlCiAgICBwYXJ0aXRpb25fY291bnRzOiB0dXBsZQoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIFRpbGVTdGF0ZToKICAgIG1ldGFkYXRhOiBUaWxlUGxhbgogICAgdGVuc29yczogdHVwbGUKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBUaWxlRm9yd2FyZDoKICAgIHRva2VuX25sbDogdG9yY2guVGVuc29yCiAgICB0b2tlbl9lbnRyb3B5OiB0b3JjaC5UZW5zb3IKICAgIGZpcnN0X2FyZ21heDogdG9yY2guVGVuc29yCiAgICBzdGF0ZTogVGlsZVN0YXRlCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgVXBzdHJlYW06CiAgICB0ZW5zb3I6IHRvcmNoLlRlbnNvcgogICAgY3VzdG9tX2VsaWdpYmxlOiBib29sCiAgICB0ZW5zb3Jfc3RhbXA6IHR1cGxlCgoKZGVmIF9jaGVjayhsb2dpdHMsdGFyZ2V0KToKICAgIGlmICh0eXBlKGxvZ2l0cykgaXMgbm90IHRvcmNoLlRlbnNvciBvciB0eXBlKHRhcmdldCkgaXMgbm90IHRvcmNoLlRlbnNvcgogICAgICAgICAgICBvciBub3QgbG9naXRzLmlzX2N1ZGEgb3IgbG9naXRzLmR0eXBlIT10b3JjaC5mbG9hdDE2CiAgICAgICAgICAgIG9yIGxvZ2l0cy5uZGltIT0yIG9yIGxvZ2l0cy5zaGFwZVsxXSE9Vk9DQUJVTEFSWQogICAgICAgICAgICBvciBub3QgMTw9bG9naXRzLnNoYXBlWzBdPD0yMDQ4CiAgICAgICAgICAgIG9yIHRhcmdldC5zaGFwZSE9KGxvZ2l0cy5zaGFwZVswXSwpIG9yIHRhcmdldC5kdHlwZSE9dG9yY2guaW50NjQKICAgICAgICAgICAgb3IgdGFyZ2V0LmRldmljZSE9bG9naXRzLmRldmljZQogICAgICAgICAgICBvciBhbnkobm90IHguaXNfY29udGlndW91cygpIG9yIHguaXNfY29uaigpIG9yIHguaXNfbmVnKCkgZm9yIHggaW4gKGxvZ2l0cyx0YXJnZXQpKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVGlsZSByZXF1aXJlcyBjb250aWd1b3VzIEZQMTZbTSxWMTUxOTM2XSwgc2hpZnRlZCBpbnQ2NFtNXSwgb25lIG93bmVkIENVREEgZGV2aWNlJykKCgpkZWYgX3N0YXRlKHJvdXRlLHJlYXNvbixsb2dpdHMsdGFyZ2V0LHRlbnNvcnM9KCksY291bnRzPSgpKToKICAgIHBsYW49VGlsZVBsYW4ocm91dGUscmVhc29uLGxvZ2l0cy5zaGFwZVswXSxsb2dpdHMuc2hhcGVbMV0sc3RhbXAodGFyZ2V0KSwKICAgICAgICAgICAgICAgICAgdHVwbGUoc3RhbXAoeCkgZm9yIHggaW4gdGVuc29ycyksdHVwbGUoY291bnRzKSkKICAgIHJldHVybiBUaWxlU3RhdGUocGxhbix0dXBsZSh0ZW5zb3JzKSkKCgpkZWYgX25hdGl2ZV9mb3J3YXJkKGxvZ2l0cyx0YXJnZXQscmVhc29uKToKICAgICMgU3RyZWFtaW5nIGNhbm5vdCBvbWl0IG1ldHJpY3Mgb24gYSBuYXRpdmUgZmFsbGJhY2s6IHByZXNlcnZlIG5hdGl2ZSBOYU4sCiAgICAjIGlnbm9yZWQtdGFyZ2V0IGFuZCBmaXJzdC1pbmRleCBhcmdtYXggYmVoYXZpb3Igb24gZXZlcnkgcm93LgogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgcHVibGljPWxvZ2l0cy5mbG9hdCgpCiAgICAgICAgbmxsPUYuY3Jvc3NfZW50cm9weShwdWJsaWMsdGFyZ2V0LGlnbm9yZV9pbmRleD0tMTAwLHJlZHVjdGlvbj0nbm9uZScpCiAgICAgICAgbG9ncD1GLmxvZ19zb2Z0bWF4KHB1YmxpYywtMSkKICAgICAgICBlbnRyb3B5PS0obG9ncC5leHAoKSpsb2dwKS5zdW0oLTEpCiAgICAgICAgcHJlZGljdGlvbnM9cHVibGljLmFyZ21heCgtMSkKICAgIHJldHVybiBUaWxlRm9yd2FyZChubGwsZW50cm9weSxwcmVkaWN0aW9ucyxfc3RhdGUoJ25hdGl2ZScscmVhc29uLGxvZ2l0cyx0YXJnZXQpKQoKCmRlZiBmb3J3YXJkX3RpbGUobG9naXRzLHRhcmdldCk6CiAgICAiIiJBbHJlYWR5IHNoaWZ0ZWQgZmxhdCB0YXJnZXRzOyByZXR1cm5zIGRldGFjaGVkIHBlci1yb3cgdmFsdWVzIGFuZCBzdGF0ZS4KClR3byBzbWFsbCBvcmlnaW5hbCBjbGFzc2lmaWVyIGhvc3QgcmVhZHMgcmVtYWluIHJlYWwgY29zdHMuIE5vIGRlbm9taW5hdG9yCnJlYWQsIGhhcmR3YXJlIHF1ZXJ5LCBkdXBsaWNhdGUgZm9yd2FyZCBzb2Z0bWF4LCBvciBzY2FsYXIgdGlsZSBsb3NzIG9jY3Vycy4KVGhlIGVuY2xvc2luZyBvd25lciBjaGVja3MgcGlubmVkIHJ1bnRpbWUvZGV2aWNlIG9uY2UgYmVmb3JlIGFsbCB0aWxlcy4KIiIiCiAgICBfY2hlY2sobG9naXRzLHRhcmdldCkKICAgIG4sdj1sb2dpdHMuc2hhcGU7bnQ9Y2UudHJpdG9uLmNkaXYodixjZS5USUxFKQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgcGFydGlhbD10b3JjaC5lbXB0eSgobixudCksZGV2aWNlPWxvZ2l0cy5kZXZpY2UsZHR5cGU9dG9yY2guZmxvYXQzMikKICAgICAgICBiYWQ9dG9yY2guZW1wdHkoKG4sbnQpLGRldmljZT1sb2dpdHMuZGV2aWNlLGR0eXBlPXRvcmNoLmludDMyKQogICAgICAgIG1heGltdW09dG9yY2guZW1wdHkobixkZXZpY2U9bG9naXRzLmRldmljZSxkdHlwZT10b3JjaC5mbG9hdDMyKQogICAgICAgIGNlLnNvdXBfY2VfZXhwX3RpbGVfbWF4WyhuLG50KV0obG9naXRzLHRhcmdldCxwYXJ0aWFsLGJhZCwKICAgICAgICAgICAgVj12LE5UPW50LEJMT0NLPWNlLlRJTEUsKipjZS5fbGF1bmNoX2NvbmZpZygpKQogICAgICAgIGludmFsaWQsYW55X3N1cGVydmlzZWQ9dG9yY2guc3RhY2soKHRvcmNoLmFueShiYWQpLHRvcmNoLmFueSh0YXJnZXQhPS0xMDApKSkudG9saXN0KCkKICAgICAgICBpZiBpbnZhbGlkOnJldHVybiBfbmF0aXZlX2ZvcndhcmQobG9naXRzLHRhcmdldCwnbm9uZmluaXRlX2xvZ2l0c19vcl9pbnZhbGlkX3RhcmdldCcpCiAgICAgICAgaWYgbm90IGFueV9zdXBlcnZpc2VkOnJldHVybiBfbmF0aXZlX2ZvcndhcmQobG9naXRzLHRhcmdldCwnbm9fc3VwZXJ2aXNlZF9yb3dzJykKICAgICAgICBjZS5zb3VwX2NlX2V4cF9yb3dfbWF4WyhuLCldKHBhcnRpYWwsbWF4aW11bSxOVD1udCwKICAgICAgICAgICAgTUVSR0U9Y2UudHJpdG9uLm5leHRfcG93ZXJfb2ZfMihudCksKipjZS5fbGF1bmNoX2NvbmZpZygpKQogICAgICAgIGxvZ3N1bT10b3JjaC5lbXB0eV9saWtlKG1heGltdW0pO3NlbGVjdGVkPXRvcmNoLmVtcHR5X2xpa2UobWF4aW11bSkKICAgICAgICBndWFyZD10b3JjaC5lbXB0eShuLGRldmljZT1sb2dpdHMuZGV2aWNlLGR0eXBlPXRvcmNoLmludDMyKQogICAgICAgIGNlLnNvdXBfY2VfZXhwX3RpbGVfc3VtWyhuLG50KV0obG9naXRzLG1heGltdW0scGFydGlhbCwKICAgICAgICAgICAgVj12LE5UPW50LEJMT0NLPWNlLlRJTEUsKipjZS5fbGF1bmNoX2NvbmZpZygpKQogICAgICAgIGNlLnNvdXBfY2VfZXhwX3NlbGVjdGVkWyhuLCldKGxvZ2l0cyx0YXJnZXQsbWF4aW11bSxwYXJ0aWFsLGxvZ3N1bSxzZWxlY3RlZCxndWFyZCwKICAgICAgICAgICAgVj12LE5UPW50LE1FUkdFPWNlLnRyaXRvbi5uZXh0X3Bvd2VyX29mXzIobnQpLE5FQVI9Y2UuTkVBUl9DRVJUQUlOX05MTCwKICAgICAgICAgICAgKipjZS5fbGF1bmNoX2NvbmZpZygpKQogICAgICAgIHBhcnRpdGlvbixyZWFzb249bWl4ZWQucGFydGl0aW9uX3Jvd3Moc2VsZWN0ZWQsdGFyZ2V0LGd1YXJkKQogICAgICAgIGlmIHJlYXNvbiBpcyBub3QgTm9uZTpyZXR1cm4gX25hdGl2ZV9mb3J3YXJkKGxvZ2l0cyx0YXJnZXQscmVhc29uKQogICAgICAgIGlmIHBhcnRpdGlvbi5uYXRpdmVfcm93cyBpcyBOb25lOgogICAgICAgICAgICB2YWx1ZXM9c2VsZWN0ZWQ7dGVuc29ycz0obWF4aW11bSxsb2dzdW0pO3JvdXRlPSdhbGxfc2FmZScKICAgICAgICBlbHNlOgogICAgICAgICAgICB2YWx1ZXM9bWl4ZWQubWl4ZWRfc2VsZWN0ZWRfdmFsdWVzKGxvZ2l0cyx0YXJnZXQsc2VsZWN0ZWQscGFydGl0aW9uLm5hdGl2ZV9yb3dzKQogICAgICAgICAgICB0ZW5zb3JzPShtYXhpbXVtLGxvZ3N1bSxndWFyZCxwYXJ0aXRpb24ubmF0aXZlX3Jvd3MpO3JvdXRlPSdtaXhlZF9yb3dzJwogICAgICAgIG5sbD10b3JjaC53aGVyZSh0YXJnZXQhPS0xMDAsLXZhbHVlcywwLjApCiAgICAgICAgZW50cm9weSxwcmVkaWN0aW9ucz1kZXRhY2hlZF9tZXRyaWNzKGxvZ2l0cyxtYXhpbXVtLGxvZ3N1bSkKICAgICAgICBjb3VudHM9KHBhcnRpdGlvbi5uYXRpdmVfc3VwZXJ2aXNlZF9yb3dzLHBhcnRpdGlvbi5zYWZlX3N1cGVydmlzZWRfcm93cyxwYXJ0aXRpb24uaWdub3JlZF9yb3dzKQogICAgICAgIHJldHVybiBUaWxlRm9yd2FyZChubGwsZW50cm9weSxwcmVkaWN0aW9ucyxfc3RhdGUocm91dGUsTm9uZSxsb2dpdHMsdGFyZ2V0LHRlbnNvcnMsY291bnRzKSkKCgpkZWYgcHJlcGFyZV91cHN0cmVhbShub3JtYWxpemVkX3Vwc3RyZWFtKToKICAgICIiIkNoZWNrIHRydWUgcG9zdC1nbG9iYWwtbm9ybWFsaXphdGlvbiB1cHN0cmVhbSBPTkNFIHBlciB3aG9sZSBiYWNrd2FyZC4KCk5vIHByZXNjYWxpbmcgYmVmb3JlIENFLCBubyBoYWxmIGdyYWRpZW50IGNhY2hlLCBubyBleHRyYSBkaXZpc2lvbi4gTm9uZmluaXRlCnVwc3RyZWFtIGRlbGVnYXRlcyB0byBuYXRpdmUgQ0UgYXV0b2dyYWQ7IGhpZ2hlci1vcmRlciBpcyB1bnN1cHBvcnRlZCBieSB0aGUKZW5jbG9zaW5nIHJlY29tcHV0aW5nLWhlYWQgY29tcG9uZW50IGFuZCByZWplY3RlZCBoZXJlIHRvby4KIiIiCiAgICBpZiB0b3JjaC5pc19ncmFkX2VuYWJsZWQoKTpyYWlzZSBSdW50aW1lRXJyb3IoJ0hpZ2hlci1vcmRlciB0aWxlIGJhY2t3YXJkIHVuc3VwcG9ydGVkJykKICAgIHE9bm9ybWFsaXplZF91cHN0cmVhbQogICAgaWYgdHlwZShxKSBpcyBub3QgdG9yY2guVGVuc29yIG9yIHEubmRpbSE9MDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdBY3R1YWwgc2NhbGFyIG5vcm1hbGl6ZWQgdXBzdHJlYW0gcmVxdWlyZWQnKQogICAgZmluaXRlPWJvb2wodG9yY2guaXNmaW5pdGUocSkuYWxsKCkuaXRlbSgpKQogICAgZWxpZ2libGU9ZmluaXRlIGFuZCBxLmR0eXBlPT10b3JjaC5mbG9hdDMyIGFuZCBub3QgcS5pc19uZWcoKSBhbmQgbm90IHEuaXNfY29uaigpCiAgICBxPXEuY29udGlndW91cygpCiAgICByZXR1cm4gVXBzdHJlYW0ocSxlbGlnaWJsZSxzdGFtcChxKSkKCgpkZWYgX25hdGl2ZV9iYWNrd2FyZChsb2dpdHMsdGFyZ2V0LHEpOgogICAgd2l0aCB0b3JjaC5lbmFibGVfZ3JhZCgpOgogICAgICAgIHdvcms9bG9naXRzLmRldGFjaCgpLnJlcXVpcmVzX2dyYWRfKFRydWUpCiAgICAgICAgdG90YWw9Ri5jcm9zc19lbnRyb3B5KHdvcmsuZmxvYXQoKSx0YXJnZXQsaWdub3JlX2luZGV4PS0xMDAscmVkdWN0aW9uPSdzdW0nKQogICAgICAgIGdyYWRpZW50LD10b3JjaC5hdXRvZ3JhZC5ncmFkKHRvdGFsLHdvcmssZ3JhZF9vdXRwdXRzPXEpCiAgICByZXR1cm4gZ3JhZGllbnQKCgpkZWYgYmFja3dhcmRfdGlsZShsb2dpdHMsdGFyZ2V0LHN0YXRlLHVwc3RyZWFtKToKICAgICIiIlVzZSBzYXZlZCBub3JtYWxpemVycyBvbmx5IHdpdGggYW4gb3duZXItcHJvdmVuIGlkZW50aWNhbCByYXcxNiB0aWxlLgoKQ2FsbGVyIHJlY29uc3RydWN0cyBUaWxlU3RhdGUocGxhbiwgc2F2ZWRfdGVuc29yc19zbGljZSkgYWZ0ZXIgcmV0cmlldmluZyBpdHMKYXV0b2dyYWQgY3R4LnNhdmVkX3RlbnNvcnM7IHRoYXQgcHJvdmlkZXMgbmF0aXZlIHNhdmVkLXZlcnNpb24gcHJvdGVjdGlvbi4KTWV0YWRhdGEgc3RhbXBzIGFkZGl0aW9uYWxseSByZWplY3QgZGlyZWN0IG11dGF0aW9uIG9mIHRhcmdldHMvc3RhdGUvdXBzdHJlYW0uCiIiIgogICAgX2NoZWNrKGxvZ2l0cyx0YXJnZXQpCiAgICBpZiB0b3JjaC5pc19ncmFkX2VuYWJsZWQoKTpyYWlzZSBSdW50aW1lRXJyb3IoJ0hpZ2hlci1vcmRlciB0aWxlIGJhY2t3YXJkIHVuc3VwcG9ydGVkJykKICAgIGlmIHR5cGUoc3RhdGUpIGlzIG5vdCBUaWxlU3RhdGUgb3IgdHlwZShzdGF0ZS5tZXRhZGF0YSkgaXMgbm90IFRpbGVQbGFuOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ093bmVkIHRpbGUgc3RhdGUgcmVxdWlyZWQnKQogICAgcD1zdGF0ZS5tZXRhZGF0YQogICAgaWYgKHAucm93cyE9bG9naXRzLnNoYXBlWzBdIG9yIHAudm9jYWJ1bGFyeSE9bG9naXRzLnNoYXBlWzFdCiAgICAgICAgICAgIG9yIHN0YW1wKHRhcmdldCkhPXAudGFyZ2V0X3N0YW1wCiAgICAgICAgICAgIG9yIHR1cGxlKHN0YW1wKHgpIGZvciB4IGluIHN0YXRlLnRlbnNvcnMpIT1wLnNhdmVkX3N0YW1wcyk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdUaWxlIHRhcmdldC9zdGF0ZSBtZXRhZGF0YSBvciB2ZXJzaW9uIGNoYW5nZWQnKQogICAgaWYgKHR5cGUodXBzdHJlYW0pIGlzIG5vdCBVcHN0cmVhbSBvciBzdGFtcCh1cHN0cmVhbS50ZW5zb3IpIT11cHN0cmVhbS50ZW5zb3Jfc3RhbXAKICAgICAgICAgICAgb3IgdXBzdHJlYW0udGVuc29yLmRldmljZSE9bG9naXRzLmRldmljZSk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdBY3R1YWwgbm9ybWFsaXplZCB1cHN0cmVhbSBjaGFuZ2VkIG9yIGhhcyB3cm9uZyBkZXZpY2UnKQogICAgcT11cHN0cmVhbS50ZW5zb3IKICAgIGlmIHAucm91dGU9PSduYXRpdmUnIG9yIG5vdCB1cHN0cmVhbS5jdXN0b21fZWxpZ2libGU6CiAgICAgICAgcmV0dXJuIF9uYXRpdmVfYmFja3dhcmQobG9naXRzLHRhcmdldCxxKQogICAgaWYgcC5yb3V0ZSBub3QgaW4gKCdhbGxfc2FmZScsJ21peGVkX3Jvd3MnKSBvciBsZW4oc3RhdGUudGVuc29ycykhPSgyIGlmIHAucm91dGU9PSdhbGxfc2FmZScgZWxzZSA0KToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0ludmFsaWQgdGlsZSByb3V0ZS9zdGF0ZSBzaGFwZScpCiAgICBtYXhpbXVtLGxvZ3N1bT1zdGF0ZS50ZW5zb3JzWzoyXTtuLHY9bG9naXRzLnNoYXBlCiAgICBncmFkaWVudD10b3JjaC5lbXB0eV9saWtlKGxvZ2l0cykKICAgIGlmIHAucm91dGU9PSdhbGxfc2FmZSc6CiAgICAgICAgY2Uuc291cF9jZV9leHBfYmFja3dhcmRbKG4sY2UudHJpdG9uLmNkaXYodixjZS5USUxFKSldKAogICAgICAgICAgICBsb2dpdHMsdGFyZ2V0LG1heGltdW0sbG9nc3VtLHEsZ3JhZGllbnQsVj12LEJMT0NLPWNlLlRJTEUsKipjZS5fbGF1bmNoX2NvbmZpZygpKQogICAgZWxzZToKICAgICAgICBndWFyZCxuYXRpdmVfcm93cz1zdGF0ZS50ZW5zb3JzWzI6XQogICAgICAgIG1peGVkLnNvdXBfY2VfbWl4ZWRfYmFja3dhcmRbKG4sY2UudHJpdG9uLmNkaXYodixjZS5USUxFKSldKAogICAgICAgICAgICBsb2dpdHMsdGFyZ2V0LG1heGltdW0sbG9nc3VtLHEsZ3VhcmQsZ3JhZGllbnQsVj12LEJMT0NLPWNlLlRJTEUsKipjZS5fbGF1bmNoX2NvbmZpZygpKQogICAgICAgICMgTmF0aXZlIG5lYXItcm93IENFIHNlZXMgYWN0dWFsIHEsIHRoZW4gLmZsb2F0KCkgYmFja3dhcmQgbmFycm93cyBvbmNlLgogICAgICAgIHNlbGVjdGVkPWxvZ2l0cy5pbmRleF9zZWxlY3QoMCxuYXRpdmVfcm93cykKICAgICAgICBzZWxlY3RlZF90YXJnZXRzPXRhcmdldC5pbmRleF9zZWxlY3QoMCxuYXRpdmVfcm93cykKICAgICAgICBuYXRpdmVfZ3JhZGllbnQ9X25hdGl2ZV9iYWNrd2FyZChzZWxlY3RlZCxzZWxlY3RlZF90YXJnZXRzLHEpCiAgICAgICAgZ3JhZGllbnQuaW5kZXhfY29weV8oMCxuYXRpdmVfcm93cyxuYXRpdmVfZ3JhZGllbnQpCiAgICByZXR1cm4gZ3JhZGllbnQK', 'solution/contracts.py': 'ZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgpAZGF0YWNsYXNzCmNsYXNzIFN1bW1hcmllczoKICAgIGVudHJvcHk6IHRvcmNoLlRlbnNvcgogICAgdG9rZW5fbmxsOiB0b3JjaC5UZW5zb3IKICAgIHByZWRpY3Rpb25zOiB0b3JjaC5UZW5zb3IKCmRlZiBfY2hlY2soaCx3LGxhYmVscyxjaHVuayxkZW5vbWluYXRvcik6CiAgICBhc3NlcnQgdHlwZShoKSBpcyB0b3JjaC5UZW5zb3IgYW5kIHR5cGUodykgaW4gKHRvcmNoLlRlbnNvcix0b3JjaC5ubi5QYXJhbWV0ZXIpIGFuZCB0eXBlKGxhYmVscykgaXMgdG9yY2guVGVuc29yCiAgICBhc3NlcnQgaC5kZXZpY2U9PXcuZGV2aWNlPT1sYWJlbHMuZGV2aWNlIGFuZCBoLmRldmljZS50eXBlIGluICgnY3B1JywnY3VkYScpCiAgICBpZiBoLmRldmljZS50eXBlPT0nY3VkYSc6CiAgICAgICAgYXNzZXJ0IGguc2hhcGVbLTFdPT0xNTM2IGFuZCB3LnNoYXBlPT0oMTUxOTM2LDE1MzYpLCAnVGFzay1zcGVjaWZpYyBDVURBIGdlb21ldHJ5JwogICAgICAgIGFzc2VydCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfY2FwYWJpbGl0eShoLmRldmljZSk9PSg3LDUpLCAnVDQgY29tcG9uZW50IG9ubHknCiAgICAgICAgYXNzZXJ0IHRvcmNoLmlzX2F1dG9jYXN0X2VuYWJsZWQoJ2N1ZGEnKSBhbmQgdG9yY2guZ2V0X2F1dG9jYXN0X2R0eXBlKCdjdWRhJyk9PXRvcmNoLmZsb2F0MTYsICdOYXRpdmUgQU1QIEZQMTYgYm91bmRhcnkgb25seScKICAgIGFzc2VydCBoLmR0eXBlPT13LmR0eXBlPT10b3JjaC5mbG9hdDMyIGFuZCBsYWJlbHMuZHR5cGU9PXRvcmNoLmludDY0CiAgICBhc3NlcnQgaC5uZGltPT0zIGFuZCB3Lm5kaW09PTIgYW5kIGxhYmVscy5zaGFwZT09aC5zaGFwZVs6Ml0gYW5kIGguc2hhcGVbLTFdPT13LnNoYXBlWzFdCiAgICBhc3NlcnQgaC5pc19jb250aWd1b3VzKCkgYW5kIHcuaXNfY29udGlndW91cygpIGFuZCBsYWJlbHMuaXNfY29udGlndW91cygpCiAgICBhc3NlcnQgbm90IGFueSh4LmlzX25lZygpIG9yIHguaXNfY29uaigpIGZvciB4IGluIChoLHcsbGFiZWxzKSkKICAgIGFzc2VydCBub3Qgdy5yZXF1aXJlc19ncmFkLCdGcm96ZW4gaGVhZCBvbmx5OiBubyB2b2NhYnVsYXJ5L3dlaWdodC1ncmFkaWVudCBhY2N1bXVsYXRpb24gcHJvb2YnCiAgICBhc3NlcnQgaC51bnR5cGVkX3N0b3JhZ2UoKS5kYXRhX3B0cigpIT13LnVudHlwZWRfc3RvcmFnZSgpLmRhdGFfcHRyKCkKICAgIGFzc2VydCB0eXBlKGNodW5rKSBpcyBpbnQgYW5kIGNodW5rPjAgYW5kIGgubnVtZWwoKT4wIGFuZCB3LnNoYXBlWzBdPjAKICAgIGFzc2VydCBkZW5vbWluYXRvciBpcyBOb25lIG9yICh0eXBlKGRlbm9taW5hdG9yKSBpcyBpbnQgYW5kIGRlbm9taW5hdG9yPj0wKSBvciAodHlwZShkZW5vbWluYXRvcikgaXMgdG9yY2guVGVuc29yIGFuZCBkZW5vbWluYXRvci5kZXZpY2U9PWguZGV2aWNlIGFuZCBkZW5vbWluYXRvci5kdHlwZT09dG9yY2guaW50NjQgYW5kIGRlbm9taW5hdG9yLm5kaW09PTAgYW5kIG5vdCBkZW5vbWluYXRvci5yZXF1aXJlc19ncmFkKQoKZGVmIHNoaWZ0X3RhcmdldHMobGFiZWxzKToKICAgIHJldHVybiBGLnBhZChsYWJlbHMsKDAsMSksdmFsdWU9LTEwMClbLi4uLDE6XS5jb250aWd1b3VzKCkK', 'solution/original/__init__.py': '', 'solution/original/materialized_ce.py': 'IiIiT3JpZ2luYWwgZXhwZXJpbWVudGFsIFRyaXRvbiBtYXRlcmlhbGl6ZWQgY2F1c2FsIENFLCBndWFyZGVkIHJvdy1taXhlZCByZXZpc2lvbi4KCkRlc2lnbiBmb2xsb3dzIHRoaXMgcHJvamVjdCdzIGluZGVwZW5kZW50bHkgYXV0aG9yZWQgQ1BVIHByb3RvdHlwZSwgbm90IGFueQp0aGlyZC1wYXJ0eSBvcHRpbWl6ZWQga2VybmVsLiBTb3VyY2Ugc2VtYW50aWNzOiB0cmFuc2Zvcm1lcnMgOTNjOGI3YjQKbG9zc191dGlscy5weTozMi03MTsgcHl0b3JjaCA3MGQ5OWU5OSBTb2Z0TWF4LmN1OjM5LTYzLCBMb3NzLmN1OjIzNy00NzUuClByZXBhcmF0aW9uIGlzIENQVS9zdGF0aWMgb25seS4gUnVubmluZyB0aGUgc3VwcGxpZWQgbm90ZWJvb2sgbGF0ZXIgbGF1bmNoZXMgQ1VEQS4KVGhlIGZpdmUgZXhpc3RpbmcgbnVtZXJpY2FsIGtlcm5lbHMgYXJlIHVuY2hhbmdlZDsgZ3VhcmRlZCByb3dzIHVzZSBuYXRpdmUgQ0UuCiIiIgoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKdHJ5OgogICAgaW1wb3J0IHRyaXRvbgogICAgaW1wb3J0IHRyaXRvbi5sYW5ndWFnZSBhcyB0bAogICAgX0lNUE9SVF9FUlJPUiA9IE5vbmUKZXhjZXB0IChJbXBvcnRFcnJvciwgUnVudGltZUVycm9yLCBPU0Vycm9yKSBhcyBlcnJvcjoKICAgIHRyaXRvbiA9IE5vbmUKICAgIHRsID0gTm9uZQogICAgX0lNUE9SVF9FUlJPUiA9IGYie3R5cGUoZXJyb3IpLl9fbmFtZV9ffToge2Vycm9yfSIKCgpWT0NBQlVMQVJZID0gMTUxOTM2ClRJTEUgPSAyMDQ4CiMgQ29uc2VydmF0aXZlIG51bWVyaWNhbC1yb3V0ZSBndWFyZCwgbm90IGFuIGFjY2VwdGFuY2UgdG9sZXJhbmNlLiBUaW55IG5hdGl2ZQojIHRhcmdldCBsb3NzZXMgYXJlIGNhbmNlbGxhdGlvbi1zZW5zaXRpdmU7IG1peGVkIHJvdXRpbmcgcHJlc2VydmVzIG5hdGl2ZSByb3dzLgpORUFSX0NFUlRBSU5fTkxMID0gMWUtMwoKCkBkYXRhY2xhc3MKY2xhc3MgRGlhZ25vc3RpY3M6CiAgICBldmVudHM6IGxpc3RbZGljdFtzdHIsIEFueV1dID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCgogICAgZGVmIHJlY29yZChzZWxmLCBldmVudDogc3RyLCAqKnZhbHVlczogQW55KSAtPiBOb25lOgogICAgICAgIHNlbGYuZXZlbnRzLmFwcGVuZCh7ImV2ZW50IjogZXZlbnQsICoqdmFsdWVzfSkKCgpkZWYgX3JlY29yZChkaWFnbm9zdGljcywgZXZlbnQsICoqdmFsdWVzKToKICAgIGlmIGRpYWdub3N0aWNzIGlzIG5vdCBOb25lOgogICAgICAgIGRpYWdub3N0aWNzLnJlY29yZChldmVudCwgKip2YWx1ZXMpCgoKZGVmIGJhY2tlbmRfaW5mbygpOgogICAgIiIiSW1wb3J0IG1ldGFkYXRhIG9ubHk6IGRvZXMgbm90IHF1ZXJ5IENVREEgb3IgaW5pdGlhbGl6ZSBhIGRyaXZlci4iIiIKICAgIHJldHVybiB7InRyaXRvbl9pbXBvcnRlZCI6IHRyaXRvbiBpcyBub3QgTm9uZSwKICAgICAgICAgICAgInRyaXRvbl92ZXJzaW9uIjogZ2V0YXR0cih0cml0b24sICJfX3ZlcnNpb25fXyIsIE5vbmUpLAogICAgICAgICAgICAidG9yY2hfdmVyc2lvbiI6IHRvcmNoLl9fdmVyc2lvbl9fLCAiaW1wb3J0X2Vycm9yIjogX0lNUE9SVF9FUlJPUn0KCgppZiB0cml0b24gaXMgbm90IE5vbmU6CiAgICBAdHJpdG9uLmppdAogICAgZGVmIHNvdXBfY2VfZXhwX3RpbGVfbWF4KFosIFRhcmdldCwgUGFydGlhbCwgQmFkLCBWOiB0bC5jb25zdGV4cHIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgTlQ6IHRsLmNvbnN0ZXhwciwgQkxPQ0s6IHRsLmNvbnN0ZXhwcik6CiAgICAgICAgcm93ID0gdGwucHJvZ3JhbV9pZCgwKQogICAgICAgIHRpbGUgPSB0bC5wcm9ncmFtX2lkKDEpCiAgICAgICAgY29sdW1uID0gdGlsZSAqIEJMT0NLICsgdGwuYXJhbmdlKDAsIEJMT0NLKQogICAgICAgIHJlYWwgPSBjb2x1bW4gPCBWCiAgICAgICAgdmFsdWUgPSB0bC5sb2FkKFogKyByb3cgKiBWICsgY29sdW1uLCBtYXNrPXJlYWwsIG90aGVyPS1mbG9hdCgiaW5mIikpLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgZmluaXRlID0gKHZhbHVlID09IHZhbHVlKSAmICh0bC5hYnModmFsdWUpICE9IGZsb2F0KCJpbmYiKSkKICAgICAgICBpbnZhbGlkX3ZhbHVlcyA9IHRsLnN1bSgocmVhbCAmIH5maW5pdGUpLnRvKHRsLmludDMyKSwgYXhpcz0wKQogICAgICAgIHRhcmdldCA9IHRsLmxvYWQoVGFyZ2V0ICsgcm93KQogICAgICAgIGludmFsaWRfdGFyZ2V0ID0gKHRhcmdldCAhPSAtMTAwKSAmICgodGFyZ2V0IDwgMCkgfCAodGFyZ2V0ID49IFYpKQogICAgICAgIGJhZCA9IGludmFsaWRfdmFsdWVzICsgdGwud2hlcmUodGlsZSA9PSAwLCBpbnZhbGlkX3RhcmdldC50byh0bC5pbnQzMiksIDApCiAgICAgICAgdGwuc3RvcmUoUGFydGlhbCArIHJvdyAqIE5UICsgdGlsZSwgdGwubWF4KHZhbHVlLCBheGlzPTApKQogICAgICAgIHRsLnN0b3JlKEJhZCArIHJvdyAqIE5UICsgdGlsZSwgYmFkKQoKICAgIEB0cml0b24uaml0CiAgICBkZWYgc291cF9jZV9leHBfcm93X21heChQYXJ0aWFsLCBNYXhpbXVtLCBOVDogdGwuY29uc3RleHByLCBNRVJHRTogdGwuY29uc3RleHByKToKICAgICAgICByb3cgPSB0bC5wcm9ncmFtX2lkKDApCiAgICAgICAgaW5kZXggPSB0bC5hcmFuZ2UoMCwgTUVSR0UpCiAgICAgICAgdmFsdWVzID0gdGwubG9hZChQYXJ0aWFsICsgcm93ICogTlQgKyBpbmRleCwgbWFzaz1pbmRleCA8IE5ULCBvdGhlcj0tZmxvYXQoImluZiIpKQogICAgICAgIHRsLnN0b3JlKE1heGltdW0gKyByb3csIHRsLm1heCh2YWx1ZXMsIGF4aXM9MCkpCgogICAgQHRyaXRvbi5qaXQKICAgIGRlZiBzb3VwX2NlX2V4cF90aWxlX3N1bShaLCBNYXhpbXVtLCBQYXJ0aWFsLCBWOiB0bC5jb25zdGV4cHIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgTlQ6IHRsLmNvbnN0ZXhwciwgQkxPQ0s6IHRsLmNvbnN0ZXhwcik6CiAgICAgICAgcm93ID0gdGwucHJvZ3JhbV9pZCgwKQogICAgICAgIHRpbGUgPSB0bC5wcm9ncmFtX2lkKDEpCiAgICAgICAgY29sdW1uID0gdGlsZSAqIEJMT0NLICsgdGwuYXJhbmdlKDAsIEJMT0NLKQogICAgICAgIHJlYWwgPSBjb2x1bW4gPCBWCiAgICAgICAgdmFsdWUgPSB0bC5sb2FkKFogKyByb3cgKiBWICsgY29sdW1uLCBtYXNrPXJlYWwsIG90aGVyPS1mbG9hdCgiaW5mIikpLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgbWF4aW11bSA9IHRsLmxvYWQoTWF4aW11bSArIHJvdykKICAgICAgICBleHBvbmVudGlhbCA9IHRsLmV4cCh2YWx1ZSAtIG1heGltdW0pCiAgICAgICAgdGwuc3RvcmUoUGFydGlhbCArIHJvdyAqIE5UICsgdGlsZSwgdGwuc3VtKHRsLndoZXJlKHJlYWwsIGV4cG9uZW50aWFsLCAwLjApLCBheGlzPTApKQoKICAgIEB0cml0b24uaml0CiAgICBkZWYgc291cF9jZV9leHBfc2VsZWN0ZWQoWiwgVGFyZ2V0LCBNYXhpbXVtLCBQYXJ0aWFsLCBMb2dTdW0sIFNlbGVjdGVkLCBHdWFyZCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBWOiB0bC5jb25zdGV4cHIsIE5UOiB0bC5jb25zdGV4cHIsIE1FUkdFOiB0bC5jb25zdGV4cHIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgTkVBUjogdGwuY29uc3RleHByKToKICAgICAgICByb3cgPSB0bC5wcm9ncmFtX2lkKDApCiAgICAgICAgaW5kZXggPSB0bC5hcmFuZ2UoMCwgTUVSR0UpCiAgICAgICAgdmFsdWVzID0gdGwubG9hZChQYXJ0aWFsICsgcm93ICogTlQgKyBpbmRleCwgbWFzaz1pbmRleCA8IE5ULCBvdGhlcj0wLjApCiAgICAgICAgbG9nc3VtID0gdGwubG9nKHRsLnN1bSh2YWx1ZXMsIGF4aXM9MCkpCiAgICAgICAgbWF4aW11bSA9IHRsLmxvYWQoTWF4aW11bSArIHJvdykKICAgICAgICB0YXJnZXQgPSB0bC5sb2FkKFRhcmdldCArIHJvdykKICAgICAgICBzdXBlcnZpc2VkID0gdGFyZ2V0ICE9IC0xMDAKICAgICAgICBzYWZlX3RhcmdldCA9IHRsLndoZXJlKHN1cGVydmlzZWQsIHRhcmdldCwgMCkKICAgICAgICBzZWxlY3RlZF92YWx1ZSA9IHRsLmxvYWQoWiArIHJvdyAqIFYgKyBzYWZlX3RhcmdldCkudG8odGwuZmxvYXQzMikKICAgICAgICBsb2dwID0gKHNlbGVjdGVkX3ZhbHVlIC0gbWF4aW11bSkgLSBsb2dzdW0KICAgICAgICB0bC5zdG9yZShMb2dTdW0gKyByb3csIGxvZ3N1bSkKICAgICAgICB0bC5zdG9yZShTZWxlY3RlZCArIHJvdywgdGwud2hlcmUoc3VwZXJ2aXNlZCwgbG9ncCwgMC4wKSkKICAgICAgICBmaW5pdGUgPSAobG9ncCA9PSBsb2dwKSAmICh0bC5hYnMobG9ncCkgIT0gZmxvYXQoImluZiIpKQogICAgICAgIHVuc2FmZSA9IHN1cGVydmlzZWQgJiAoKGxvZ3AgPj0gLU5FQVIpIHwgfmZpbml0ZSkKICAgICAgICB0bC5zdG9yZShHdWFyZCArIHJvdywgdW5zYWZlLnRvKHRsLmludDMyKSkKCiAgICBAdHJpdG9uLmppdAogICAgZGVmIHNvdXBfY2VfZXhwX2JhY2t3YXJkKFosIFRhcmdldCwgTWF4aW11bSwgTG9nU3VtLCBRLCBHcmFkLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIFY6IHRsLmNvbnN0ZXhwciwgQkxPQ0s6IHRsLmNvbnN0ZXhwcik6CiAgICAgICAgcm93ID0gdGwucHJvZ3JhbV9pZCgwKQogICAgICAgIHRpbGUgPSB0bC5wcm9ncmFtX2lkKDEpCiAgICAgICAgY29sdW1uID0gdGlsZSAqIEJMT0NLICsgdGwuYXJhbmdlKDAsIEJMT0NLKQogICAgICAgIHJlYWwgPSBjb2x1bW4gPCBWCiAgICAgICAgdmFsdWUgPSB0bC5sb2FkKFogKyByb3cgKiBWICsgY29sdW1uLCBtYXNrPXJlYWwsIG90aGVyPTAuMCkudG8odGwuZmxvYXQzMikKICAgICAgICBtYXhpbXVtID0gdGwubG9hZChNYXhpbXVtICsgcm93KQogICAgICAgIGxvZ3N1bSA9IHRsLmxvYWQoTG9nU3VtICsgcm93KQogICAgICAgIHEgPSB0bC5sb2FkKFEpLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgdGFyZ2V0ID0gdGwubG9hZChUYXJnZXQgKyByb3cpCiAgICAgICAgYWRqb2ludCA9IHRsLndoZXJlKHRhcmdldCAhPSAtMTAwLCAtcSwgMC4wKQogICAgICAgIGxvZ3AgPSAodmFsdWUgLSBtYXhpbXVtKSAtIGxvZ3N1bQogICAgICAgIHByb2R1Y3QgPSB0bC5leHAobG9ncCkgKiBhZGpvaW50CiAgICAgICAgc3BhcnNlID0gdGwud2hlcmUoY29sdW1uID09IHRhcmdldCwgYWRqb2ludCwgMC4wKQogICAgICAgIGdyYWRpZW50ID0gc3BhcnNlIC0gcHJvZHVjdAogICAgICAgICMgR3JhZCBpcyBGUDE2OiBuYXJyb3dpbmcgaGFwcGVucyBvbmx5IGhlcmUsIGFmdGVyIG5vcm1hbGl6ZWQgc2NhbGluZy4KICAgICAgICB0bC5zdG9yZShHcmFkICsgcm93ICogViArIGNvbHVtbiwgZ3JhZGllbnQsIG1hc2s9cmVhbCkKCgpkZWYgX3RhcmdldHMobGFiZWxzLCBpZ25vcmVfaW5kZXg9LTEwMCk6CiAgICByZXR1cm4gRi5wYWQobGFiZWxzLCAoMCwgMSksIHZhbHVlPWlnbm9yZV9pbmRleClbLi4uLCAxOl0uY29udGlndW91cygpCgoKZGVmIG5hdGl2ZV9jYXVzYWxfbG9zcyhsb2dpdHMsIGxhYmVscywgbnVtX2l0ZW1zX2luX2JhdGNoPU5vbmUsICosIGlnbm9yZV9pbmRleD0tMTAwLCBzaGlmdF9sYWJlbHM9Tm9uZSk6CiAgICAiIiJVbnBhdGNoZWQgbmF0aXZlIGV4cHJlc3Npb24gdXNlZCBieSB1bnN1cHBvcnRlZC9leGNlcHRpb25hbCBjYWxscy4iIiIKICAgIHRhcmdldCA9IF90YXJnZXRzKGxhYmVscywgaWdub3JlX2luZGV4KSBpZiBzaGlmdF9sYWJlbHMgaXMgTm9uZSBlbHNlIHNoaWZ0X2xhYmVscwogICAgc291cmNlID0gbG9naXRzLmZsb2F0KCkudmlldygtMSwgbG9naXRzLnNoYXBlWy0xXSkKICAgIHRhcmdldCA9IHRhcmdldC52aWV3KC0xKS50byhzb3VyY2UuZGV2aWNlKQogICAgdG90YWwgPSBGLmNyb3NzX2VudHJvcHkoc291cmNlLCB0YXJnZXQsIGlnbm9yZV9pbmRleD1pZ25vcmVfaW5kZXgsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlZHVjdGlvbj0ibWVhbiIgaWYgbnVtX2l0ZW1zX2luX2JhdGNoIGlzIE5vbmUgZWxzZSAic3VtIikKICAgIGlmIG51bV9pdGVtc19pbl9iYXRjaCBpcyBub3QgTm9uZToKICAgICAgICBkZW5vbWluYXRvciA9IG51bV9pdGVtc19pbl9iYXRjaAogICAgICAgIGlmIGlzaW5zdGFuY2UoZGVub21pbmF0b3IsIHRvcmNoLlRlbnNvcik6CiAgICAgICAgICAgIGRlbm9taW5hdG9yID0gZGVub21pbmF0b3IudG8odG90YWwuZGV2aWNlKQogICAgICAgIHRvdGFsID0gdG90YWwgLyBkZW5vbWluYXRvcgogICAgcmV0dXJuIHRvdGFsCgoKZGVmIF9kZW5vbWluYXRvcl9tZXRhZGF0YSh2YWx1ZSwgZGV2aWNlKToKICAgICIiIk9uZSBob3N0IHJlYWQgYXQgbW9zdDsgcHJlc2VydmUgdGhlIG9yaWdpbmFsIG9iamVjdCBmb3IgZXh0ZXJuYWwgU1VNIC8gRC4iIiIKICAgIGlmIHR5cGUodmFsdWUpIGlzIGludDoKICAgICAgICByZXR1cm4gKHZhbHVlLCAicHl0aG9uX2ludCIpIGlmIDAgPCB2YWx1ZSA8PSB0b3JjaC5paW5mbyh0b3JjaC5pbnQ2NCkubWF4IGVsc2UgTm9uZQogICAgaWYgKGlzaW5zdGFuY2UodmFsdWUsIHRvcmNoLlRlbnNvcikgYW5kIHZhbHVlLmR0eXBlID09IHRvcmNoLmludDY0CiAgICAgICAgICAgIGFuZCB2YWx1ZS5uZGltID09IDAgYW5kIG5vdCB2YWx1ZS5yZXF1aXJlc19ncmFkCiAgICAgICAgICAgIGFuZCAodmFsdWUuZGV2aWNlLnR5cGUgPT0gImNwdSIgb3IgdmFsdWUuZGV2aWNlID09IGRldmljZSkpOgogICAgICAgIGNvdW50ID0gaW50KHZhbHVlLml0ZW0oKSkKICAgICAgICByZXR1cm4gKGNvdW50LCAiaW50NjRfIiArIHZhbHVlLmRldmljZS50eXBlKSBpZiBjb3VudCA+IDAgZWxzZSBOb25lCiAgICByZXR1cm4gTm9uZQoKCmRlZiBfbGF1bmNoX2NvbmZpZygpOgogICAgIyBObyBhdXRvdHVuaW5nLCBoaWRkZW4gd2FybXVwLCBwcmVjb21waWxlZCBjYWNoZSBvciBDVURBIGdyYXBoLgogICAgcmV0dXJuIHsibnVtX3dhcnBzIjogNCwgImVuYWJsZV9mcF9mdXNpb24iOiBGYWxzZX0KCgpjbGFzcyBfVHJpdG9uU3VtQ0UodG9yY2guYXV0b2dyYWQuRnVuY3Rpb24pOgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGZvcndhcmQoY3R4LCBsb2dpdHMsIHRhcmdldCwgbWF4aW11bSwgbG9nc3VtLCBzZWxlY3RlZCwgZGlhZ25vc3RpY3MsCiAgICAgICAgICAgICAgICBzdW1tYXJ5X3Npbms9Tm9uZSwgdG9rZW5fZW50cm9weT1Ob25lLCBmaXJzdF9hcmdtYXg9Tm9uZSk6CiAgICAgICAgY3R4LmlucHV0X2NvdW50ID0gbGVuKGN0eC5uZWVkc19pbnB1dF9ncmFkKQogICAgICAgIG4gPSB0YXJnZXQubnVtZWwoKQogICAgICAgIG50ID0gdHJpdG9uLmNkaXYoVk9DQUJVTEFSWSwgVElMRSkKICAgICAgICBmbGF0X3RhcmdldCA9IHRhcmdldC52aWV3KC0xKQogICAgICAgIGludGVybmFsX3RhcmdldCA9IHRvcmNoLndoZXJlKGZsYXRfdGFyZ2V0ICE9IC0xMDAsIHRvcmNoLnplcm9zX2xpa2UoZmxhdF90YXJnZXQpLCBmbGF0X3RhcmdldCkKICAgICAgICAjIFRoZSB0YWJsZSBhbHJlYWR5IGNvbnRhaW5zIHNlbGVjdGVkIGZ1bGwtdm9jYWJ1bGFyeSBsb2cgcHJvYmFiaWxpdGllcy4KICAgICAgICAjIFRoaXMgaXMgTkxMLCBuZXZlciBhIHNlY29uZCBjcm9zcy1lbnRyb3B5L3NvZnRtYXggb3ZlciBvbmUgY2xhc3MuCiAgICAgICAgdG90YWwgPSBGLm5sbF9sb3NzKHNlbGVjdGVkWzosIE5vbmVdLCBpbnRlcm5hbF90YXJnZXQsIHJlZHVjdGlvbj0ic3VtIiwgaWdub3JlX2luZGV4PS0xMDApCiAgICAgICAgaWYgc3VtbWFyeV9zaW5rIGlzIG5vdCBOb25lOgogICAgICAgICAgICBmcm9tIC5tZXRyaWNfc3VtbWFyeSBpbXBvcnQgcHVibGlzaAogICAgICAgICAgICBwdWJsaXNoKHN1bW1hcnlfc2luaywgc2VsZWN0ZWQsIHRhcmdldCwgdG9rZW5fZW50cm9weSwgZmlyc3RfYXJnbWF4LCByb3V0ZT0iYWxsX3NhZmUiKQogICAgICAgIGN0eC5zYXZlX2Zvcl9iYWNrd2FyZChsb2dpdHMsIHRhcmdldCwgbWF4aW11bSwgbG9nc3VtKQogICAgICAgIGN0eC5kaWFnbm9zdGljcyA9IGRpYWdub3N0aWNzCiAgICAgICAgY3R4LnNldF9tYXRlcmlhbGl6ZV9ncmFkcyhGYWxzZSkKICAgICAgICBfcmVjb3JkKGRpYWdub3N0aWNzLCAidHJpdG9uX2ZvcndhcmQiLCByb3dzPW4sIHZvY2FidWxhcnk9Vk9DQUJVTEFSWSwgdGlsZXM9bnQpCiAgICAgICAgcmV0dXJuIHRvdGFsCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGJhY2t3YXJkKGN0eCwgbm9ybWFsaXplZF91cHN0cmVhbSk6CiAgICAgICAgaWYgbm9ybWFsaXplZF91cHN0cmVhbSBpcyBOb25lOgogICAgICAgICAgICBfcmVjb3JkKGN0eC5kaWFnbm9zdGljcywgInVudXNlZF9iYWNrd2FyZCIpCiAgICAgICAgICAgIHJldHVybiAoTm9uZSwpICogY3R4LmlucHV0X2NvdW50CiAgICAgICAgbG9naXRzLCB0YXJnZXQsIG1heGltdW0sIGxvZ3N1bSA9IGN0eC5zYXZlZF90ZW5zb3JzCiAgICAgICAgaGlnaGVyX29yZGVyID0gdG9yY2guaXNfZ3JhZF9lbmFibGVkKCkKICAgICAgICBmaW5pdGUgPSBib29sKHRvcmNoLmlzZmluaXRlKG5vcm1hbGl6ZWRfdXBzdHJlYW0pLmFsbCgpLml0ZW0oKSkKICAgICAgICBpZiAoaGlnaGVyX29yZGVyIG9yIG5vdCBmaW5pdGUgb3Igbm9ybWFsaXplZF91cHN0cmVhbS5kdHlwZSAhPSB0b3JjaC5mbG9hdDMyCiAgICAgICAgICAgICAgICBvciBub3JtYWxpemVkX3Vwc3RyZWFtLmlzX25lZygpIG9yIG5vcm1hbGl6ZWRfdXBzdHJlYW0uaXNfY29uaigpKToKICAgICAgICAgICAgd2l0aCB0b3JjaC5lbmFibGVfZ3JhZCgpOgogICAgICAgICAgICAgICAgd29yayA9IGxvZ2l0cyBpZiBoaWdoZXJfb3JkZXIgZWxzZSBsb2dpdHMuZGV0YWNoKCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkKICAgICAgICAgICAgICAgIG5hdGl2ZV9zdW0gPSBGLmNyb3NzX2VudHJvcHkod29yay5mbG9hdCgpLnZpZXcoLTEsIHdvcmsuc2hhcGVbLTFdKSwgdGFyZ2V0LnZpZXcoLTEpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlZHVjdGlvbj0ic3VtIiwgaWdub3JlX2luZGV4PS0xMDApCiAgICAgICAgICAgICAgICAoZ3JhZGllbnQsKSA9IHRvcmNoLmF1dG9ncmFkLmdyYWQobmF0aXZlX3N1bSwgd29yaywgZ3JhZF9vdXRwdXRzPW5vcm1hbGl6ZWRfdXBzdHJlYW0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY3JlYXRlX2dyYXBoPWhpZ2hlcl9vcmRlcikKICAgICAgICAgICAgX3JlY29yZChjdHguZGlhZ25vc3RpY3MsICJuYXRpdmVfYmFja3dhcmRfZmFsbGJhY2siLAogICAgICAgICAgICAgICAgICAgIHJlYXNvbj0iaGlnaGVyX29yZGVyIiBpZiBoaWdoZXJfb3JkZXIgZWxzZSAibm9uZmluaXRlX29yX3Vuc3VwcG9ydGVkX3Vwc3RyZWFtIikKICAgICAgICAgICAgcmV0dXJuIChncmFkaWVudCwpICsgKE5vbmUsKSAqIChjdHguaW5wdXRfY291bnQgLSAxKQogICAgICAgIGdyYWRpZW50ID0gdG9yY2guZW1wdHlfbGlrZShsb2dpdHMpCiAgICAgICAgcSA9IG5vcm1hbGl6ZWRfdXBzdHJlYW0uY29udGlndW91cygpCiAgICAgICAgbiwgbnQgPSB0YXJnZXQubnVtZWwoKSwgdHJpdG9uLmNkaXYoVk9DQUJVTEFSWSwgVElMRSkKICAgICAgICBzb3VwX2NlX2V4cF9iYWNrd2FyZFsobiwgbnQpXShsb2dpdHMsIHRhcmdldCwgbWF4aW11bSwgbG9nc3VtLCBxLCBncmFkaWVudCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBWPVZPQ0FCVUxBUlksIEJMT0NLPVRJTEUsICoqX2xhdW5jaF9jb25maWcoKSkKICAgICAgICBfcmVjb3JkKGN0eC5kaWFnbm9zdGljcywgInRyaXRvbl9iYWNrd2FyZCIsIHJvd3M9biwgcGhhc2U9ImVucXVldWVkX25vdF9jb21wbGV0aW9uIikKICAgICAgICByZXR1cm4gKGdyYWRpZW50LCkgKyAoTm9uZSwpICogKGN0eC5pbnB1dF9jb3VudCAtIDEpCgoKZGVmIG1hdGVyaWFsaXplZF9jYXVzYWxfbG9zcyhsb2dpdHMsIGxhYmVscywgbnVtX2l0ZW1zX2luX2JhdGNoPU5vbmUsICosCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZ25vcmVfaW5kZXg9LTEwMCwgc2hpZnRfbGFiZWxzPU5vbmUsIGRpYWdub3N0aWNzPU5vbmUsIG5hdGl2ZV9mYWxsYmFjaz1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgc3VtbWFyeV9zaW5rPU5vbmUpOgogICAgIiIiTm8gbW9kZWwvZm9yd2FyZC90cmFpbmVyIHBhdGNoLiBBZGFwdGVyIGZhbGxiYWNrIGRlbGVnYXRlcyB0byBjYXB0dXJlZCBuYXRpdmUgbG9zcy4iIiIKICAgIGRlZiBmYWxsYmFjayhyZWFzb24pOgogICAgICAgIF9yZWNvcmQoZGlhZ25vc3RpY3MsICJuYXRpdmVfZm9yd2FyZF9mYWxsYmFjayIsIHJlYXNvbj1yZWFzb24pCiAgICAgICAgaWYgbmF0aXZlX2ZhbGxiYWNrIGlzIG5vdCBOb25lOgogICAgICAgICAgICByZXR1cm4gbmF0aXZlX2ZhbGxiYWNrKHJlYXNvbikKICAgICAgICByZXR1cm4gbmF0aXZlX2NhdXNhbF9sb3NzKGxvZ2l0cywgbGFiZWxzLCBudW1faXRlbXNfaW5fYmF0Y2gsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZ25vcmVfaW5kZXg9aWdub3JlX2luZGV4LCBzaGlmdF9sYWJlbHM9c2hpZnRfbGFiZWxzKQoKICAgIGlmIG5vdCBsb2dpdHMuaXNfY3VkYToKICAgICAgICByZXR1cm4gZmFsbGJhY2soIm5vdF9jdWRhIikKICAgIGlmIHRyaXRvbiBpcyBOb25lOgogICAgICAgIHJldHVybiBmYWxsYmFjaygiYmFja2VuZF91bmF2YWlsYWJsZSIpCiAgICBpZiBub3QgdG9yY2guaXNfZ3JhZF9lbmFibGVkKCkgb3Igbm90IGxvZ2l0cy5yZXF1aXJlc19ncmFkOgogICAgICAgIHJldHVybiBmYWxsYmFjaygibm9fZ3JhZGllbnRfcGF0aCIpCiAgICBpZiBsb2dpdHMuZHR5cGUgIT0gdG9yY2guZmxvYXQxNiBvciBsb2dpdHMuaXNfbmVnKCkgb3IgbG9naXRzLmlzX2NvbmooKSBvciBub3QgbG9naXRzLmlzX2NvbnRpZ3VvdXMoKToKICAgICAgICByZXR1cm4gZmFsbGJhY2soImR0eXBlX29yX3N0cmlkZSIpCiAgICBpZiAobG9naXRzLm5kaW0gIT0gMyBvciBsb2dpdHMuc2hhcGVbMF0gIT0gNCBvciBub3QgMSA8PSBsb2dpdHMuc2hhcGVbMV0gPD0gNTEyCiAgICAgICAgICAgIG9yIGxvZ2l0cy5zaGFwZVsyXSAhPSBWT0NBQlVMQVJZIG9yIGxhYmVscy5zaGFwZSAhPSBsb2dpdHMuc2hhcGVbOi0xXSk6CiAgICAgICAgcmV0dXJuIGZhbGxiYWNrKCJzaGFwZSIpCiAgICBpZiAobGFiZWxzLmR0eXBlICE9IHRvcmNoLmludDY0IG9yIGxhYmVscy5kZXZpY2UgIT0gbG9naXRzLmRldmljZQogICAgICAgICAgICBvciBpZ25vcmVfaW5kZXggIT0gLTEwMCBvciBzaGlmdF9sYWJlbHMgaXMgbm90IE5vbmUpOgogICAgICAgIHJldHVybiBmYWxsYmFjaygibGFiZWxfY29udHJhY3QiKQogICAgaWYgdG9yY2guY3VkYS5nZXRfZGV2aWNlX2NhcGFiaWxpdHkobG9naXRzLmRldmljZSkgIT0gKDcsIDUpOgogICAgICAgIHJldHVybiBmYWxsYmFjaygibm90X3NtNzUiKQogICAgZGVub21pbmF0b3JfbWV0YWRhdGEgPSBfZGVub21pbmF0b3JfbWV0YWRhdGEobnVtX2l0ZW1zX2luX2JhdGNoLCBsb2dpdHMuZGV2aWNlKQogICAgaWYgZGVub21pbmF0b3JfbWV0YWRhdGEgaXMgTm9uZToKICAgICAgICByZXR1cm4gZmFsbGJhY2soImRlbm9taW5hdG9yIikKICAgIGNvdW50LCBraW5kID0gZGVub21pbmF0b3JfbWV0YWRhdGEKICAgIF9yZWNvcmQoZGlhZ25vc3RpY3MsICJkZW5vbWluYXRvcl92YWxpZGF0ZWQiLCBjb3VudD1jb3VudCwga2luZD1raW5kKQoKICAgIF9yZWNvcmQoZGlhZ25vc3RpY3MsICJ0cml0b25fcHJlcGFzc19zdGFydGVkIiwgcGhhc2U9ImJlZm9yZV9maXJzdF9lbnF1ZXVlIikKICAgIHRhcmdldCA9IF90YXJnZXRzKGxhYmVscykKICAgIG4sIG50ID0gdGFyZ2V0Lm51bWVsKCksIHRyaXRvbi5jZGl2KFZPQ0FCVUxBUlksIFRJTEUpCiAgICBwYXJ0aWFsID0gdG9yY2guZW1wdHkoKG4sIG50KSwgZGV2aWNlPWxvZ2l0cy5kZXZpY2UsIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICBiYWQgPSB0b3JjaC5lbXB0eSgobiwgbnQpLCBkZXZpY2U9bG9naXRzLmRldmljZSwgZHR5cGU9dG9yY2guaW50MzIpCiAgICBtYXhpbXVtID0gdG9yY2guZW1wdHkobiwgZGV2aWNlPWxvZ2l0cy5kZXZpY2UsIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBzb3VwX2NlX2V4cF90aWxlX21heFsobiwgbnQpXShsb2dpdHMsIHRhcmdldCwgcGFydGlhbCwgYmFkLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgVj1WT0NBQlVMQVJZLCBOVD1udCwgQkxPQ0s9VElMRSwgKipfbGF1bmNoX2NvbmZpZygpKQogICAgICAgICMgU21hbGwgZmxhZ3Mgb25seS4gVGhpcyBob3N0IHJlYWQgc3luY2hyb25pemVzIGFuZCBpcyBhIHJlYWwgY29zdC4KICAgICAgICAjIFNoYXJlIHRoZSBleGlzdGluZyBob3N0IHN5bmNocm9uaXphdGlvbjsgZW1wdHkgc3VwZXJ2aXNpb24gaGFzIG5vCiAgICAgICAgIyB1c2VmdWwgY3VzdG9tIENFIHdvcmsuIE5laXRoZXIgZmxhZyBjaGFuZ2VzIHRoZSBnbG9iYWwgaW52YWxpZCBndWFyZC4KICAgICAgICBpbnZhbGlkLCBhbnlfc3VwZXJ2aXNlZCA9IHRvcmNoLnN0YWNrKCh0b3JjaC5hbnkoYmFkKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b3JjaC5hbnkodGFyZ2V0ICE9IC0xMDApKSkudG9saXN0KCkKICAgICAgICBpZiBub3QgaW52YWxpZCBhbmQgYW55X3N1cGVydmlzZWQ6CiAgICAgICAgICAgIHNvdXBfY2VfZXhwX3Jvd19tYXhbKG4sKV0ocGFydGlhbCwgbWF4aW11bSwgTlQ9bnQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgTUVSR0U9dHJpdG9uLm5leHRfcG93ZXJfb2ZfMihudCksICoqX2xhdW5jaF9jb25maWcoKSkKICAgICMgT3V0c2lkZSB0aGUgcHJlcGFzcyBuby1ncmFkIGNvbnRleHQ6IGZhbGxiYWNrIG11c3QgYnVpbGQgdGhlIG5hdGl2ZSBncmFwaC4KICAgIGlmIGludmFsaWQ6CiAgICAgICAgcmV0dXJuIGZhbGxiYWNrKCJub25maW5pdGVfbG9naXRzX29yX2ludmFsaWRfdGFyZ2V0IikKICAgIGlmIG5vdCBhbnlfc3VwZXJ2aXNlZDoKICAgICAgICByZXR1cm4gZmFsbGJhY2soIm5vX3N1cGVydmlzZWRfcm93cyIpCiAgICBsb2dzdW0gPSB0b3JjaC5lbXB0eShuLCBkZXZpY2U9bG9naXRzLmRldmljZSwgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgIHNlbGVjdGVkID0gdG9yY2guZW1wdHlfbGlrZShsb2dzdW0pCiAgICBndWFyZCA9IHRvcmNoLmVtcHR5KG4sIGRldmljZT1sb2dpdHMuZGV2aWNlLCBkdHlwZT10b3JjaC5pbnQzMikKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIHNvdXBfY2VfZXhwX3RpbGVfc3VtWyhuLCBudCldKGxvZ2l0cywgbWF4aW11bSwgcGFydGlhbCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIFY9Vk9DQUJVTEFSWSwgTlQ9bnQsIEJMT0NLPVRJTEUsICoqX2xhdW5jaF9jb25maWcoKSkKICAgICAgICBzb3VwX2NlX2V4cF9zZWxlY3RlZFsobiwpXShsb2dpdHMsIHRhcmdldCwgbWF4aW11bSwgcGFydGlhbCwgbG9nc3VtLCBzZWxlY3RlZCwgZ3VhcmQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgVj1WT0NBQlVMQVJZLCBOVD1udCwgTUVSR0U9dHJpdG9uLm5leHRfcG93ZXJfb2ZfMihudCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgTkVBUj1ORUFSX0NFUlRBSU5fTkxMLCAqKl9sYXVuY2hfY29uZmlnKCkpCiAgICAjIFByZXNlcnZlIGFsbCBjbGFzc2lmaWNhdGlvbiBndWFyZHMsIHdpdGggb25lIGNvbWJpbmVkIGhvc3QgcmVhZCBpbnN0ZWFkCiAgICAjIG9mIHNlcGFyYXRlIG5lYXIvZmluaXRlL3NhZmUvaWdub3JlZCByZWFkcy4gS2VybmVsIGFyaXRobWV0aWMgaXMgdW5jaGFuZ2VkLgogICAgZnJvbSAubWl4ZWRfcm93cyBpbXBvcnQgTWl4ZWRSb3dTdW0sIHBhcnRpdGlvbl9yb3dzCiAgICBwYXJ0aXRpb24sIHJlYXNvbiA9IHBhcnRpdGlvbl9yb3dzKHNlbGVjdGVkLCB0YXJnZXQsIGd1YXJkKQogICAgaWYgcmVhc29uIGlzIG5vdCBOb25lOgogICAgICAgIHJldHVybiBmYWxsYmFjayhyZWFzb24pCiAgICB0b2tlbl9lbnRyb3B5ID0gZmlyc3RfYXJnbWF4ID0gTm9uZQogICAgaWYgc3VtbWFyeV9zaW5rIGlzIG5vdCBOb25lOgogICAgICAgIGZyb20gLm1ldHJpY19zdW1tYXJ5IGltcG9ydCBkZXRhY2hlZF9tZXRyaWNzCiAgICAgICAgdG9rZW5fZW50cm9weSwgZmlyc3RfYXJnbWF4ID0gZGV0YWNoZWRfbWV0cmljcyhsb2dpdHMsIG1heGltdW0sIGxvZ3N1bSkKICAgIGlmIHBhcnRpdGlvbi5uYXRpdmVfcm93cyBpcyBub3QgTm9uZToKICAgICAgICB0b3RhbCA9IE1peGVkUm93U3VtLmFwcGx5KGxvZ2l0cywgdGFyZ2V0LCBtYXhpbXVtLCBsb2dzdW0sIHNlbGVjdGVkLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZ3VhcmQsIHBhcnRpdGlvbiwgZGlhZ25vc3RpY3MsIHN1bW1hcnlfc2luaywgdG9rZW5fZW50cm9weSwgZmlyc3RfYXJnbWF4KQogICAgZWxzZToKICAgICAgICB0b3RhbCA9IF9Ucml0b25TdW1DRS5hcHBseShsb2dpdHMsIHRhcmdldCwgbWF4aW11bSwgbG9nc3VtLCBzZWxlY3RlZCwgZGlhZ25vc3RpY3MsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3VtbWFyeV9zaW5rLCB0b2tlbl9lbnRyb3B5LCBmaXJzdF9hcmdtYXgpCiAgICBkZW5vbWluYXRvciA9IG51bV9pdGVtc19pbl9iYXRjaAogICAgaWYgaXNpbnN0YW5jZShkZW5vbWluYXRvciwgdG9yY2guVGVuc29yKToKICAgICAgICBkZW5vbWluYXRvciA9IGRlbm9taW5hdG9yLnRvKHRvdGFsLmRldmljZSkKICAgIHJlc3VsdCA9IHRvdGFsIC8gZGVub21pbmF0b3IKICAgIGlmIHN1bW1hcnlfc2luayBpcyBub3QgTm9uZToKICAgICAgICBzdW1tYXJ5X3NpbmsuY29tcGxldGUocmVzdWx0KQogICAgcmV0dXJuIHJlc3VsdAo=', 'solution/original/metric_summary.py': 'IiIiT3JpZ2luYWwgZGV0YWNoZWQgbWF0ZXJpYWxpemVkIENFIG1ldHJpYyBzdW1tYXJpZXMsIHJlc2VhcmNoIHByb3Bvc2FsLgoKT25lIGV4dHJhIEZQMTYgdm9jYWJ1bGFyeSByZWFkIHVzZXMgdGhlIGFscmVhZHkgY29tcHV0ZWQgQ0UgbWF4L2xvZ3N1bS4gSXQKcmVwbGFjZXMgbmF0aXZlIGZ1bGwtdm9jYWJ1bGFyeSBlbnRyb3B5IGFuZCBmaXJzdC13aW5zIGFyZ21heCBPTkxZIGFmdGVyIGEKY29uc3VtZXItb3duZWQgcGVyLWNhbGwgc2luayBpcyBhY2NlcHRlZC4gSXQgZG9lcyBub3QgY2hhbmdlIENFIGFyaXRobWV0aWMsCm5vcm1hbGl6YXRpb24gb3IgYmFja3dhcmQuIEZpbml0ZSBsb2dpdHMgYXJlIGFuIHVwc3RyZWFtIGdsb2JhbCBpbnZhcmlhbnQuCiIiIgppbXBvcnQgdG9yY2gKaW1wb3J0IHRyaXRvbgppbXBvcnQgdHJpdG9uLmxhbmd1YWdlIGFzIHRsCgoKQHRyaXRvbi5qaXQKZGVmIHNvdXBfY2VfbWV0cmljX3RpbGVzKFosIE1heGltdW0sIExvZ1N1bSwgUGFydGlhbEVudHJvcHksIFBhcnRpYWxBcmdtYXgsCiAgICAgICAgICAgICAgICAgICAgICAgICBWOiB0bC5jb25zdGV4cHIsIE5UOiB0bC5jb25zdGV4cHIsIEJMT0NLOiB0bC5jb25zdGV4cHIpOgogICAgcm93ID0gdGwucHJvZ3JhbV9pZCgwKQogICAgdGlsZSA9IHRsLnByb2dyYW1faWQoMSkKICAgIGNvbHVtbiA9IHRpbGUgKiBCTE9DSyArIHRsLmFyYW5nZSgwLCBCTE9DSykKICAgIHJlYWwgPSBjb2x1bW4gPCBWCiAgICB2YWx1ZSA9IHRsLmxvYWQoWiArIHJvdyAqIFYgKyBjb2x1bW4sIG1hc2s9cmVhbCwgb3RoZXI9MC4wKS50byh0bC5mbG9hdDMyKQogICAgbWF4aW11bSA9IHRsLmxvYWQoTWF4aW11bSArIHJvdykKICAgIGxvZ3N1bSA9IHRsLmxvYWQoTG9nU3VtICsgcm93KQogICAgIyBQYWRkaW5nIGlzIG5ldXRyYWwgQkVGT1JFIGV4cC9wcm9kdWN0LCBub3QgYW4gdW5zYWZlIDAgKiBpbmZpbml0eSBtYXNrLgogICAgbG9ncCA9IHRsLndoZXJlKHJlYWwsICh2YWx1ZSAtIG1heGltdW0pIC0gbG9nc3VtLCAwLjApCiAgICBjb250cmlidXRpb24gPSB0bC5leHAobG9ncCkgKiBsb2dwCiAgICBlbnRyb3B5ID0gLXRsLnN1bShjb250cmlidXRpb24sIGF4aXM9MCkKICAgIGZpcnN0ID0gdGwubWluKHRsLndoZXJlKHJlYWwgJiAodmFsdWUgPT0gbWF4aW11bSksIGNvbHVtbiwgMjE0NzQ4MzY0NyksIGF4aXM9MCkKICAgIHRsLnN0b3JlKFBhcnRpYWxFbnRyb3B5ICsgcm93ICogTlQgKyB0aWxlLCBlbnRyb3B5KQogICAgdGwuc3RvcmUoUGFydGlhbEFyZ21heCArIHJvdyAqIE5UICsgdGlsZSwgZmlyc3QpCgoKQHRyaXRvbi5qaXQKZGVmIHNvdXBfY2VfbWV0cmljX21lcmdlKFBhcnRpYWxFbnRyb3B5LCBQYXJ0aWFsQXJnbWF4LCBFbnRyb3B5LCBBcmdtYXgsCiAgICAgICAgICAgICAgICAgICAgICAgICBOVDogdGwuY29uc3RleHByLCBNRVJHRTogdGwuY29uc3RleHByKToKICAgIHJvdyA9IHRsLnByb2dyYW1faWQoMCkKICAgIGluZGV4ID0gdGwuYXJhbmdlKDAsIE1FUkdFKQogICAgZW50cm9weSA9IHRsLmxvYWQoUGFydGlhbEVudHJvcHkgKyByb3cgKiBOVCArIGluZGV4LCBtYXNrPWluZGV4IDwgTlQsIG90aGVyPTAuMCkKICAgIGZpcnN0ID0gdGwubG9hZChQYXJ0aWFsQXJnbWF4ICsgcm93ICogTlQgKyBpbmRleCwgbWFzaz1pbmRleCA8IE5ULCBvdGhlcj0yMTQ3NDgzNjQ3KQogICAgdGwuc3RvcmUoRW50cm9weSArIHJvdywgdGwuc3VtKGVudHJvcHksIGF4aXM9MCkpCiAgICB0bC5zdG9yZShBcmdtYXggKyByb3csIHRsLm1pbihmaXJzdCwgYXhpcz0wKS50byh0bC5pbnQ2NCkpCgoKZGVmIGRldGFjaGVkX21ldHJpY3MobG9naXRzLCBtYXhpbXVtLCBsb2dzdW0pOgogICAgIiIiUHJpdmF0ZSBmaW5pdGUtcm91dGUgaGVscGVyOyBubyBob3N0IHZhbHVlIHJlYWRzIG9yIGF1dG9ncmFkIHJldGVudGlvbi4iIiIKICAgIG4sIHYgPSBsb2dpdHMubnVtZWwoKSAvLyBsb2dpdHMuc2hhcGVbLTFdLCBsb2dpdHMuc2hhcGVbLTFdCiAgICBibG9jayA9IDIwNDgKICAgIG50ID0gdHJpdG9uLmNkaXYodiwgYmxvY2spCiAgICBzaGFwZSA9IHR1cGxlKGxvZ2l0cy5zaGFwZVs6LTFdKQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgcGFydGlhbF9lbnRyb3B5ID0gdG9yY2guZW1wdHkoKG4sIG50KSwgZGV2aWNlPWxvZ2l0cy5kZXZpY2UsIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICAgICAgcGFydGlhbF9hcmdtYXggPSB0b3JjaC5lbXB0eSgobiwgbnQpLCBkZXZpY2U9bG9naXRzLmRldmljZSwgZHR5cGU9dG9yY2guaW50MzIpCiAgICAgICAgZW50cm9weSA9IHRvcmNoLmVtcHR5KHNoYXBlLCBkZXZpY2U9bG9naXRzLmRldmljZSwgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgICAgICBhcmdtYXggPSB0b3JjaC5lbXB0eShzaGFwZSwgZGV2aWNlPWxvZ2l0cy5kZXZpY2UsIGR0eXBlPXRvcmNoLmludDY0KQogICAgICAgIHNvdXBfY2VfbWV0cmljX3RpbGVzWyhuLCBudCldKGxvZ2l0cywgbWF4aW11bSwgbG9nc3VtLCBwYXJ0aWFsX2VudHJvcHksIHBhcnRpYWxfYXJnbWF4LAogICAgICAgICAgICBWPXYsIE5UPW50LCBCTE9DSz1ibG9jaywgbnVtX3dhcnBzPTQsIGVuYWJsZV9mcF9mdXNpb249RmFsc2UpCiAgICAgICAgc291cF9jZV9tZXRyaWNfbWVyZ2VbKG4sKV0ocGFydGlhbF9lbnRyb3B5LCBwYXJ0aWFsX2FyZ21heCwgZW50cm9weSwgYXJnbWF4LAogICAgICAgICAgICBOVD1udCwgTUVSR0U9dHJpdG9uLm5leHRfcG93ZXJfb2ZfMihudCksIG51bV93YXJwcz00LCBlbmFibGVfZnBfZnVzaW9uPUZhbHNlKQogICAgcmV0dXJuIGVudHJvcHksIGFyZ21heAoKCmRlZiBwdWJsaXNoKHNpbmssIHNlbGVjdGVkLCB0YXJnZXQsIGVudHJvcHksIGFyZ21heCwgKiwgcm91dGUpOgogICAgIiIiRnJlc2ggc21hbGwgTkxMIGFsbG9jYXRpb24gcHJvdGVjdHMgdHJhaW5pbmctc2VsZWN0ZWQgc3RvcmFnZSBmcm9tIHNpbmsuIiIiCiAgICBpZiBzaW5rIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICB0b2tlbl9ubGwgPSB0b3JjaC53aGVyZSh0YXJnZXQudmlldygtMSkgIT0gLTEwMCwgLXNlbGVjdGVkLmRldGFjaCgpLCAwLjApCiAgICAgICAgdG9rZW5fbmxsID0gdG9rZW5fbmxsLnZpZXcodGFyZ2V0LnNoYXBlKS5jb250aWd1b3VzKCkKICAgICAgICBzaW5rKHRva2VuX25sbD10b2tlbl9ubGwsIHRva2VuX2VudHJvcHk9ZW50cm9weSwgZmlyc3RfYXJnbWF4PWFyZ21heCwgcm91dGU9cm91dGUpCg==', 'solution/original/mixed_rows.py': 'IiIiUFJPUE9TQUw6IGV4YWN0IG5hdGl2ZSBDRSByZXBsYXkgb24gZ3VhcmRlZCByb3dzLCBkaXNqb2ludCBjdXN0b20gcmVtYWluZGVyLgoKTm90IHRhcmdldCB2YWxpZGF0ZWQuIFRoaXMgaXMgYSBkaXN0aW5jdCBhbGdvcml0aG0sIG5vdCBhIHRlbGVtZXRyeS1vbmx5IHJlcnVuLgpGdWxsLXNoYXBlIG5hdGl2ZSBOTEwgc3VtbWF0aW9uIG9yZGVyIGlzIHJldGFpbmVkOyBuZXZlciBhZGQgdHdvIHN1YnNldCB0b3RhbHMuCiIiIgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCmltcG9ydCB0cml0b24KaW1wb3J0IHRyaXRvbi5sYW5ndWFnZSBhcyB0bAoKCkB0cml0b24uaml0CmRlZiBzb3VwX2NlX21peGVkX2JhY2t3YXJkKFosIFRhcmdldCwgTWF4aW11bSwgTG9nU3VtLCBRLCBOYXRpdmVSb3csIEdyYWQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIFY6IHRsLmNvbnN0ZXhwciwgQkxPQ0s6IHRsLmNvbnN0ZXhwcik6CiAgICByb3cgPSB0bC5wcm9ncmFtX2lkKDApCiAgICB0aWxlID0gdGwucHJvZ3JhbV9pZCgxKQogICAgIyBVbmlmb3JtIHBlci1wcm9ncmFtIGJyYW5jaC4gTmF0aXZlIHJvd3MgYXJlIG5vdCBjb21wdXRlZCBvciBzdG9yZWQgaGVyZS4KICAgIGlmIHRsLmxvYWQoTmF0aXZlUm93ICsgcm93KSA9PSAwOgogICAgICAgIGNvbHVtbiA9IHRpbGUgKiBCTE9DSyArIHRsLmFyYW5nZSgwLCBCTE9DSykKICAgICAgICByZWFsID0gY29sdW1uIDwgVgogICAgICAgIHZhbHVlID0gdGwubG9hZChaICsgcm93ICogViArIGNvbHVtbiwgbWFzaz1yZWFsLCBvdGhlcj0wLjApLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgbWF4aW11bSA9IHRsLmxvYWQoTWF4aW11bSArIHJvdykKICAgICAgICBsb2dzdW0gPSB0bC5sb2FkKExvZ1N1bSArIHJvdykKICAgICAgICBxID0gdGwubG9hZChRKS50byh0bC5mbG9hdDMyKQogICAgICAgIHRhcmdldCA9IHRsLmxvYWQoVGFyZ2V0ICsgcm93KQogICAgICAgIGFkam9pbnQgPSB0bC53aGVyZSh0YXJnZXQgIT0gLTEwMCwgLXEsIDAuMCkKICAgICAgICBsb2dwID0gKHZhbHVlIC0gbWF4aW11bSkgLSBsb2dzdW0KICAgICAgICBwcm9kdWN0ID0gdGwuZXhwKGxvZ3ApICogYWRqb2ludAogICAgICAgIHNwYXJzZSA9IHRsLndoZXJlKGNvbHVtbiA9PSB0YXJnZXQsIGFkam9pbnQsIDAuMCkKICAgICAgICBncmFkaWVudCA9IHNwYXJzZSAtIHByb2R1Y3QKICAgICAgICB0bC5zdG9yZShHcmFkICsgcm93ICogViArIGNvbHVtbiwgZ3JhZGllbnQsIG1hc2s9cmVhbCkKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBSb3dQYXJ0aXRpb246CiAgICAiIiJIb3N0IGNvdW50cyBmcm9tIG9uZSBjbGFzc2lmaWNhdGlvbiByZWFkOyBubyB0ZW5zb3IgdmFsdWVzIGNvcGllZCBoZXJlLiIiIgogICAgbmF0aXZlX3Jvd3M6IHRvcmNoLlRlbnNvciB8IE5vbmUKICAgIG5hdGl2ZV9zdXBlcnZpc2VkX3Jvd3M6IGludAogICAgc2FmZV9zdXBlcnZpc2VkX3Jvd3M6IGludAogICAgaWdub3JlZF9yb3dzOiBpbnQKCiAgICBkZWYgY291bnRzKHNlbGYpOgogICAgICAgIHJldHVybiB7Im5hdGl2ZV9zdXBlcnZpc2VkX3Jvd3MiOiBzZWxmLm5hdGl2ZV9zdXBlcnZpc2VkX3Jvd3MsCiAgICAgICAgICAgICAgICAic2FmZV9zdXBlcnZpc2VkX3Jvd3MiOiBzZWxmLnNhZmVfc3VwZXJ2aXNlZF9yb3dzLAogICAgICAgICAgICAgICAgImlnbm9yZWRfcm93cyI6IHNlbGYuaWdub3JlZF9yb3dzfQoKCmRlZiBwYXJ0aXRpb25fcm93cyhzZWxlY3RlZCwgdGFyZ2V0LCBuYXRpdmVfZ3VhcmQpOgogICAgIiIiS2VlcCBldmVyeSBndWFyZDsgY29uc29saWRhdGUgY2xhc3NpZmljYXRpb24gaW50byBvbmUgMy1pbnQgaG9zdCByZWFkLgoKICAgIE1hc2tlZCBsb2dpY2FsIG9wZXJhdGlvbnMgYXZvaWQgYm9vbGVhbi1pbmRleGluZyBzZWxlY3RlZCAod2hpY2ggY2FuIHJlcXVpcmUKICAgIGEgc2VwYXJhdGUgZHluYW1pYy1zaXplIHN5bmNocm9uaXphdGlvbikuIE5hdGl2ZSBub256ZXJvIGlzIGludGVudGlvbmFsbHkKICAgIHJldGFpbmVkIGFuZCBzdGlsbCBoYXMgYSBzeW5jaHJvbml6YXRpb24vYWxsb2NhdGlvbiBjb3N0LiBObyB0aW1pbmcgY2xhaW0uCiAgICAiIiIKICAgIHN1cGVydmlzZWQgPSB0YXJnZXQudmlldygtMSkgIT0gLTEwMAogICAgaW52YWxpZF9zZWxlY3RlZCA9IHRvcmNoLmFueShzdXBlcnZpc2VkICYgfnRvcmNoLmlzZmluaXRlKHNlbGVjdGVkKSkKICAgIG5hdGl2ZV9jb3VudCA9IG5hdGl2ZV9ndWFyZC5zdW0oZHR5cGU9dG9yY2guaW50NjQpCiAgICBpZ25vcmVkX2NvdW50ID0gKH5zdXBlcnZpc2VkKS5zdW0oZHR5cGU9dG9yY2guaW50NjQpCiAgICBpbnZhbGlkLCBuYXRpdmVfY291bnQsIGlnbm9yZWRfY291bnQgPSB0b3JjaC5zdGFjaygoCiAgICAgICAgaW52YWxpZF9zZWxlY3RlZC50byh0b3JjaC5pbnQ2NCksIG5hdGl2ZV9jb3VudCwgaWdub3JlZF9jb3VudCkpLnRvbGlzdCgpCiAgICBzYWZlX2NvdW50ID0gdGFyZ2V0Lm51bWVsKCkgLSBuYXRpdmVfY291bnQgLSBpZ25vcmVkX2NvdW50CiAgICBpZiBpbnZhbGlkOgogICAgICAgIHJldHVybiBOb25lLCAiaW52YWxpZF9zZWxlY3RlZF9sb2dwIgogICAgaWYgaWdub3JlZF9jb3VudCA9PSB0YXJnZXQubnVtZWwoKToKICAgICAgICByZXR1cm4gTm9uZSwgIm5vX3N1cGVydmlzZWRfcm93cyIKICAgIGlmIHNhZmVfY291bnQgPT0gMDoKICAgICAgICByZXR1cm4gTm9uZSwgImFsbF9zdXBlcnZpc2VkX3Jvd3NfbmVhcl9jZXJ0YWluIgogICAgIyBObyBhbGxvY2F0aW9uL25vbnplcm8gZm9yIHRoZSBvcmRpbmFyeSBhbGwtc2FmZSByb3V0ZS4KICAgIHJvd3MgPSB0b3JjaC5ub256ZXJvKG5hdGl2ZV9ndWFyZCwgYXNfdHVwbGU9RmFsc2UpLmZsYXR0ZW4oKSBpZiBuYXRpdmVfY291bnQgZWxzZSBOb25lCiAgICByZXR1cm4gUm93UGFydGl0aW9uKHJvd3MsIG5hdGl2ZV9jb3VudCwgc2FmZV9jb3VudCwgaWdub3JlZF9jb3VudCksIE5vbmUKCgpkZWYgbWl4ZWRfc2VsZWN0ZWRfdmFsdWVzKGxvZ2l0cywgdGFyZ2V0LCBzZWxlY3RlZCwgbmF0aXZlX3Jvd3MpOgogICAgIiIiTmF0aXZlIHBlci1yb3cgQ0Ugb25seSBmb3IgdGhlIHNlbGVjdGVkIHN1cGVydmlzZWQgcm93czsgcHJlc2VydmUgcm93IG9yZGVyLiIiIgogICAgZmxhdCA9IGxvZ2l0cy52aWV3KC0xLCBsb2dpdHMuc2hhcGVbLTFdKQogICAgbmF0aXZlX2xvZ2l0cyA9IGZsYXQuaW5kZXhfc2VsZWN0KDAsIG5hdGl2ZV9yb3dzKQogICAgbmF0aXZlX3RhcmdldCA9IHRhcmdldC52aWV3KC0xKS5pbmRleF9zZWxlY3QoMCwgbmF0aXZlX3Jvd3MpCiAgICBuYXRpdmVfbG9zcyA9IEYuY3Jvc3NfZW50cm9weShuYXRpdmVfbG9naXRzLmZsb2F0KCksIG5hdGl2ZV90YXJnZXQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlZHVjdGlvbj0ibm9uZSIsIGlnbm9yZV9pbmRleD0tMTAwKQogICAgdmFsdWVzID0gc2VsZWN0ZWQuY2xvbmUoKQogICAgdmFsdWVzLmluZGV4X2NvcHlfKDAsIG5hdGl2ZV9yb3dzLCAtbmF0aXZlX2xvc3MpCiAgICByZXR1cm4gdmFsdWVzCgoKZGVmIG9yZGVyZWRfbmxsX3N1bShzZWxlY3RlZCwgdGFyZ2V0KToKICAgIGZsYXRfdGFyZ2V0ID0gdGFyZ2V0LnZpZXcoLTEpCiAgICBpbnRlcm5hbF90YXJnZXQgPSB0b3JjaC53aGVyZShmbGF0X3RhcmdldCAhPSAtMTAwLCB0b3JjaC56ZXJvc19saWtlKGZsYXRfdGFyZ2V0KSwgZmxhdF90YXJnZXQpCiAgICByZXR1cm4gRi5ubGxfbG9zcyhzZWxlY3RlZFs6LCBOb25lXSwgaW50ZXJuYWxfdGFyZ2V0LCByZWR1Y3Rpb249InN1bSIsIGlnbm9yZV9pbmRleD0tMTAwKQoKCmRlZiBfcmVjb3JkKGRpYWdub3N0aWNzLCBldmVudCwgKip2YWx1ZXMpOgogICAgaWYgZGlhZ25vc3RpY3MgaXMgbm90IE5vbmU6CiAgICAgICAgZGlhZ25vc3RpY3MucmVjb3JkKGV2ZW50LCAqKnZhbHVlcykKCgpjbGFzcyBNaXhlZFJvd1N1bSh0b3JjaC5hdXRvZ3JhZC5GdW5jdGlvbik6CiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgZm9yd2FyZChjdHgsIGxvZ2l0cywgdGFyZ2V0LCBtYXhpbXVtLCBsb2dzdW0sIHNlbGVjdGVkLCBuYXRpdmVfZ3VhcmQsIHBhcnRpdGlvbiwgZGlhZ25vc3RpY3MsCiAgICAgICAgICAgICAgICBzdW1tYXJ5X3Npbms9Tm9uZSwgdG9rZW5fZW50cm9weT1Ob25lLCBmaXJzdF9hcmdtYXg9Tm9uZSk6CiAgICAgICAgY3R4LmlucHV0X2NvdW50ID0gbGVuKGN0eC5uZWVkc19pbnB1dF9ncmFkKQogICAgICAgIG5hdGl2ZV9yb3dzID0gcGFydGl0aW9uLm5hdGl2ZV9yb3dzCiAgICAgICAgdmFsdWVzID0gbWl4ZWRfc2VsZWN0ZWRfdmFsdWVzKGxvZ2l0cywgdGFyZ2V0LCBzZWxlY3RlZCwgbmF0aXZlX3Jvd3MpCiAgICAgICAgdG90YWwgPSBvcmRlcmVkX25sbF9zdW0odmFsdWVzLCB0YXJnZXQpCiAgICAgICAgaWYgc3VtbWFyeV9zaW5rIGlzIG5vdCBOb25lOgogICAgICAgICAgICBmcm9tIC5tZXRyaWNfc3VtbWFyeSBpbXBvcnQgcHVibGlzaAogICAgICAgICAgICBwdWJsaXNoKHN1bW1hcnlfc2luaywgdmFsdWVzLCB0YXJnZXQsIHRva2VuX2VudHJvcHksIGZpcnN0X2FyZ21heCwgcm91dGU9Im1peGVkX3Jvd3MiKQogICAgICAgIGN0eC5zYXZlX2Zvcl9iYWNrd2FyZChsb2dpdHMsIHRhcmdldCwgbWF4aW11bSwgbG9nc3VtLCBuYXRpdmVfZ3VhcmQsIG5hdGl2ZV9yb3dzKQogICAgICAgIGN0eC5kaWFnbm9zdGljcyA9IGRpYWdub3N0aWNzCiAgICAgICAgY3R4LnBhcnRpdGlvbl9jb3VudHMgPSBwYXJ0aXRpb24uY291bnRzKCkKICAgICAgICBjdHguc2V0X21hdGVyaWFsaXplX2dyYWRzKEZhbHNlKQogICAgICAgIF9yZWNvcmQoZGlhZ25vc3RpY3MsICJ0cml0b25fZm9yd2FyZCIsIHJvdXRlPSJtaXhlZF9yb3dzIiwgcm93cz10YXJnZXQubnVtZWwoKSwKICAgICAgICAgICAgICAgICoqY3R4LnBhcnRpdGlvbl9jb3VudHMsCiAgICAgICAgICAgICAgICBmb3J3YXJkX3N1bT0iZnVsbF9yb3dfb3JkZXJfbmF0aXZlX25sbF9ub3Rfc3Vic2V0X3RvdGFscyIpCiAgICAgICAgcmV0dXJuIHRvdGFsCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGJhY2t3YXJkKGN0eCwgbm9ybWFsaXplZF91cHN0cmVhbSk6CiAgICAgICAgaWYgbm9ybWFsaXplZF91cHN0cmVhbSBpcyBOb25lOgogICAgICAgICAgICBfcmVjb3JkKGN0eC5kaWFnbm9zdGljcywgInVudXNlZF9iYWNrd2FyZCIpCiAgICAgICAgICAgIHJldHVybiAoTm9uZSwpICogY3R4LmlucHV0X2NvdW50CiAgICAgICAgbG9naXRzLCB0YXJnZXQsIG1heGltdW0sIGxvZ3N1bSwgbmF0aXZlX2d1YXJkLCBuYXRpdmVfcm93cyA9IGN0eC5zYXZlZF90ZW5zb3JzCiAgICAgICAgaGlnaGVyX29yZGVyID0gdG9yY2guaXNfZ3JhZF9lbmFibGVkKCkKICAgICAgICBmaW5pdGUgPSBib29sKHRvcmNoLmlzZmluaXRlKG5vcm1hbGl6ZWRfdXBzdHJlYW0pLmFsbCgpLml0ZW0oKSkKICAgICAgICBpZiAoaGlnaGVyX29yZGVyIG9yIG5vdCBmaW5pdGUgb3Igbm9ybWFsaXplZF91cHN0cmVhbS5kdHlwZSAhPSB0b3JjaC5mbG9hdDMyCiAgICAgICAgICAgICAgICBvciBub3JtYWxpemVkX3Vwc3RyZWFtLmlzX25lZygpIG9yIG5vcm1hbGl6ZWRfdXBzdHJlYW0uaXNfY29uaigpKToKICAgICAgICAgICAgd2l0aCB0b3JjaC5lbmFibGVfZ3JhZCgpOgogICAgICAgICAgICAgICAgd29yayA9IGxvZ2l0cyBpZiBoaWdoZXJfb3JkZXIgZWxzZSBsb2dpdHMuZGV0YWNoKCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkKICAgICAgICAgICAgICAgIG5hdGl2ZV9zdW0gPSBGLmNyb3NzX2VudHJvcHkod29yay5mbG9hdCgpLnZpZXcoLTEsIHdvcmsuc2hhcGVbLTFdKSwgdGFyZ2V0LnZpZXcoLTEpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlZHVjdGlvbj0ic3VtIiwgaWdub3JlX2luZGV4PS0xMDApCiAgICAgICAgICAgICAgICBncmFkaWVudCwgPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKG5hdGl2ZV9zdW0sIHdvcmssIGdyYWRfb3V0cHV0cz1ub3JtYWxpemVkX3Vwc3RyZWFtLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjcmVhdGVfZ3JhcGg9aGlnaGVyX29yZGVyKQogICAgICAgICAgICBfcmVjb3JkKGN0eC5kaWFnbm9zdGljcywgIm5hdGl2ZV9iYWNrd2FyZF9mYWxsYmFjayIsCiAgICAgICAgICAgICAgICAgICAgcmVhc29uPSJoaWdoZXJfb3JkZXIiIGlmIGhpZ2hlcl9vcmRlciBlbHNlICJub25maW5pdGVfb3JfdW5zdXBwb3J0ZWRfdXBzdHJlYW0iKQogICAgICAgICAgICByZXR1cm4gKGdyYWRpZW50LCkgKyAoTm9uZSwpICogKGN0eC5pbnB1dF9jb3VudCAtIDEpCiAgICAgICAgZ3JhZGllbnQgPSB0b3JjaC5lbXB0eV9saWtlKGxvZ2l0cykKICAgICAgICBxID0gbm9ybWFsaXplZF91cHN0cmVhbS5jb250aWd1b3VzKCkKICAgICAgICBuLCB2ID0gdGFyZ2V0Lm51bWVsKCksIGxvZ2l0cy5zaGFwZVstMV0KICAgICAgICBzb3VwX2NlX21peGVkX2JhY2t3YXJkWyhuLCB0cml0b24uY2Rpdih2LCAyMDQ4KSldKAogICAgICAgICAgICBsb2dpdHMsIHRhcmdldCwgbWF4aW11bSwgbG9nc3VtLCBxLCBuYXRpdmVfZ3VhcmQsIGdyYWRpZW50LAogICAgICAgICAgICBWPXYsIEJMT0NLPTIwNDgsIG51bV93YXJwcz00LCBlbmFibGVfZnBfZnVzaW9uPUZhbHNlKQogICAgICAgICMgUmVjb21wdXRlIG5hdGl2ZSBDRSBvbmx5IGZvciBuZWFyLWNlcnRhaW4gcm93cy4gSXRzIEZQMzIgYWRqb2ludCBuYXJyb3dzCiAgICAgICAgIyB0aHJvdWdoIHRoZSBvcmRpbmFyeSAuZmxvYXQoKSBjYXN0IHRvIEZQMTYgQkVGT1JFIHJvdyBpbnNlcnRpb24uCiAgICAgICAgd2l0aCB0b3JjaC5lbmFibGVfZ3JhZCgpOgogICAgICAgICAgICBuYXRpdmVfbG9naXRzID0gbG9naXRzLmRldGFjaCgpLnZpZXcobiwgdikuaW5kZXhfc2VsZWN0KDAsIG5hdGl2ZV9yb3dzKS5yZXF1aXJlc19ncmFkXyhUcnVlKQogICAgICAgICAgICBuYXRpdmVfdGFyZ2V0ID0gdGFyZ2V0LnZpZXcoLTEpLmluZGV4X3NlbGVjdCgwLCBuYXRpdmVfcm93cykKICAgICAgICAgICAgbmF0aXZlX3N1bSA9IEYuY3Jvc3NfZW50cm9weShuYXRpdmVfbG9naXRzLmZsb2F0KCksIG5hdGl2ZV90YXJnZXQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByZWR1Y3Rpb249InN1bSIsIGlnbm9yZV9pbmRleD0tMTAwKQogICAgICAgICAgICBuYXRpdmVfZ3JhZGllbnQsID0gdG9yY2guYXV0b2dyYWQuZ3JhZChuYXRpdmVfc3VtLCBuYXRpdmVfbG9naXRzLCBncmFkX291dHB1dHM9cSkKICAgICAgICAjIERpc2pvaW50IHBhcnRpdGlvbjogbm8gaGFsZi1yb3VuZGVkIGdyYWRpZW50IGFkZGl0aW9uLCBubyBzZWNvbmQgZGl2aWRlLgogICAgICAgIGdyYWRpZW50LnZpZXcobiwgdikuaW5kZXhfY29weV8oMCwgbmF0aXZlX3Jvd3MsIG5hdGl2ZV9ncmFkaWVudCkKICAgICAgICBfcmVjb3JkKGN0eC5kaWFnbm9zdGljcywgInRyaXRvbl9iYWNrd2FyZCIsIHJvdXRlPSJtaXhlZF9yb3dzIiwKICAgICAgICAgICAgICAgICoqY3R4LnBhcnRpdGlvbl9jb3VudHMsIHBoYXNlPSJlbnF1ZXVlZF9ub3RfY29tcGxldGlvbiIpCiAgICAgICAgcmV0dXJuIChncmFkaWVudCwpICsgKE5vbmUsKSAqIChjdHguaW5wdXRfY291bnQgLSAxKQo=', 'solution/token_chunk_head.py': 'IiIiT3JpZ2luYWwgdG9rZW4tYXhpcyBoZWFkIHJlY29tcHV0YXRpb24gd2l0aCBzaGFyZWQgb3JpZ2luYWwgQ0UgdGlsZSBwcmltaXRpdmVzLgoKU21hbGwgZm9yd2FyZCBub3JtYWxpemVycy9yb3V0ZXMgYXJlIHJldGFpbmVkLCBuZXZlciBmdWxsIGxvZ2l0cyBvciBhIGdyYWRpZW50LgpUaGUgYWN0dWFsIGdsb2JhbCB1cHN0cmVhbSBhcnJpdmVzIGJlZm9yZSBDRSBiYWNrd2FyZCBhbmQgaXRzIEZQMTYgbmFycm93aW5nLgpObyBtb2RlbCBwYXRjaCwgZHR5cGUgcG9saWN5IGNoYW5nZSwgdGFzayBzd2l0Y2ggb3IgZHVwbGljYXRlIG1ldHJpYyBjb25zdW1lci4KIiIiCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCmZyb20gLmNvbnRyYWN0cyBpbXBvcnQgX2NoZWNrLHNoaWZ0X3RhcmdldHMsU3VtbWFyaWVzCgoKY2xhc3MgX1Rva2VuQ2h1bmtDRSh0b3JjaC5hdXRvZ3JhZC5GdW5jdGlvbik6CiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgZm9yd2FyZChjdHgsaCx3LGxhYmVscyxjaHVuayxkZW5vbWluYXRvcix0aWxlX2FwaSk6CiAgICAgICAgX2NoZWNrKGgsdyxsYWJlbHMsY2h1bmssZGVub21pbmF0b3IpCiAgICAgICAgYix0LGs9aC5zaGFwZQogICAgICAgIGgxNj1oLnRvKHRvcmNoLmZsb2F0MTYpLnJlc2hhcGUoLTEsayk7dzE2PXcudG8odG9yY2guZmxvYXQxNikKICAgICAgICB0YXJnZXRzPXNoaWZ0X3RhcmdldHMobGFiZWxzKS5yZXNoYXBlKC0xKQogICAgICAgIGVudHJvcHk9dG9yY2guZW1wdHkoYip0LGR0eXBlPXRvcmNoLmZsb2F0MzIsZGV2aWNlPWguZGV2aWNlKQogICAgICAgIG5sbD10b3JjaC5lbXB0eV9saWtlKGVudHJvcHkpO3ByZWRpY3Rpb25zPXRvcmNoLmVtcHR5KGIqdCxkdHlwZT10b3JjaC5pbnQ2NCxkZXZpY2U9aC5kZXZpY2UpCiAgICAgICAgZGl2aXNvcj0odGFyZ2V0cyE9LTEwMCkuc3VtKCkgaWYgZGVub21pbmF0b3IgaXMgTm9uZSBlbHNlIGRlbm9taW5hdG9yCiAgICAgICAgdGVuc29yX2Rpdmlzb3I9aXNpbnN0YW5jZShkaXZpc29yLHRvcmNoLlRlbnNvcikKICAgICAgICBzYXZlZD1baDE2LHcxNix0YXJnZXRzXSsoW2Rpdmlzb3JdIGlmIHRlbnNvcl9kaXZpc29yIGVsc2UgW10pCiAgICAgICAgc3BlY2lmaWNhdGlvbnM9W10KICAgICAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCxiKnQsY2h1bmspOgogICAgICAgICAgICBzdG9wPW1pbihzdGFydCtjaHVuayxiKnQpCiAgICAgICAgICAgIHJhdz1GLmxpbmVhcihoMTZbc3RhcnQ6c3RvcF0sdzE2KQogICAgICAgICAgICB0aWxlPXRpbGVfYXBpLmZvcndhcmRfdGlsZShyYXcsdGFyZ2V0c1tzdGFydDpzdG9wXSkKICAgICAgICAgICAgbmxsW3N0YXJ0OnN0b3BdPXRpbGUudG9rZW5fbmxsO2VudHJvcHlbc3RhcnQ6c3RvcF09dGlsZS50b2tlbl9lbnRyb3B5CiAgICAgICAgICAgIHByZWRpY3Rpb25zW3N0YXJ0OnN0b3BdPXRpbGUuZmlyc3RfYXJnbWF4CiAgICAgICAgICAgIG9mZnNldD1sZW4oc2F2ZWQpO3NhdmVkLmV4dGVuZCh0aWxlLnN0YXRlLnRlbnNvcnMpCiAgICAgICAgICAgIHNwZWNpZmljYXRpb25zLmFwcGVuZCgoc3RhcnQsc3RvcCx0aWxlLnN0YXRlLm1ldGFkYXRhLG9mZnNldCxsZW4odGlsZS5zdGF0ZS50ZW5zb3JzKSkpCiAgICAgICAgICAgIGRlbCByYXcsdGlsZQogICAgICAgICMgS2VlcCB0aGUgbmF0aXZlIGZ1bGwtcm93IE5MTCBTVU0gcmVkdWN0aW9uLCBub3Qgc3VtcyBvZiB0aWxlIG1lYW5zIG9yCiAgICAgICAgIyB0aWxlIHRvdGFscy4gVGhpcyBpcyBOTEwgb2Ygc2VsZWN0ZWQgZnVsbC12b2NhYnVsYXJ5IGxvZy1wcm9iYWJpbGl0aWVzLAogICAgICAgICMgbmV2ZXIgY3Jvc3MgZW50cm9weSBvdmVyIG9uZSBjbGFzcy4gRXhhY3RseSBvbmUgZ2xvYmFsIGRpdmlzaW9uLgogICAgICAgIGludGVybmFsX3RhcmdldD10b3JjaC53aGVyZSh0YXJnZXRzIT0tMTAwLHRvcmNoLnplcm9zX2xpa2UodGFyZ2V0cyksdGFyZ2V0cykKICAgICAgICB0b3RhbD1GLm5sbF9sb3NzKC1ubGxbOixOb25lXSxpbnRlcm5hbF90YXJnZXQscmVkdWN0aW9uPSdzdW0nLGlnbm9yZV9pbmRleD0tMTAwKQogICAgICAgIGxvc3M9dG90YWwvZGl2aXNvcgogICAgICAgIGN0eC5zYXZlX2Zvcl9iYWNrd2FyZCgqc2F2ZWQpCiAgICAgICAgY3R4LnRlbnNvcl9kaXZpc29yPXRlbnNvcl9kaXZpc29yO2N0eC5kaXZpc29yPU5vbmUgaWYgdGVuc29yX2Rpdmlzb3IgZWxzZSBkaXZpc29yCiAgICAgICAgY3R4LnNoYXBlPWguc2hhcGU7Y3R4LnRpbGVfYXBpPXRpbGVfYXBpO2N0eC5zcGVjaWZpY2F0aW9ucz10dXBsZShzcGVjaWZpY2F0aW9ucykKICAgICAgICBlbnRyb3B5PWVudHJvcHkucmVzaGFwZShiLHQpO25sbD1ubGwucmVzaGFwZShiLHQpO3ByZWRpY3Rpb25zPXByZWRpY3Rpb25zLnJlc2hhcGUoYix0KQogICAgICAgIGN0eC5tYXJrX25vbl9kaWZmZXJlbnRpYWJsZShlbnRyb3B5LG5sbCxwcmVkaWN0aW9ucykKICAgICAgICByZXR1cm4gbG9zcyxlbnRyb3B5LG5sbCxwcmVkaWN0aW9ucwoKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBiYWNrd2FyZChjdHgsZ3JhZF9sb3NzLF9lbnRyb3B5LF9ubGwsX3ByZWRpY3Rpb25zKToKICAgICAgICBpZiB0b3JjaC5pc19ncmFkX2VuYWJsZWQoKTpyYWlzZSBSdW50aW1lRXJyb3IoJ0hpZ2hlci1vcmRlciByZXF1ZXN0cyBub3Qgc3VwcG9ydGVkIGJ5IHN0YW5kYWxvbmUgY29tcG9uZW50JykKICAgICAgICBzYXZlZD1jdHguc2F2ZWRfdGVuc29ycztoMTYsdzE2LHRhcmdldHM9c2F2ZWRbOjNdCiAgICAgICAgZGl2aXNvcj1zYXZlZFszXSBpZiBjdHgudGVuc29yX2Rpdmlzb3IgZWxzZSBjdHguZGl2aXNvcgogICAgICAgIG5vcm1hbGl6ZWQ9Z3JhZF9sb3NzL2Rpdmlzb3IKICAgICAgICB1cHN0cmVhbT1jdHgudGlsZV9hcGkucHJlcGFyZV91cHN0cmVhbShub3JtYWxpemVkKQogICAgICAgIHJlc3VsdD10b3JjaC5lbXB0eV9saWtlKGgxNixkdHlwZT10b3JjaC5mbG9hdDMyKQogICAgICAgIGZvciBzdGFydCxzdG9wLHBsYW4sb2Zmc2V0LGNvdW50IGluIGN0eC5zcGVjaWZpY2F0aW9uczoKICAgICAgICAgICAgIyBTYW1lIG93bmVkIGhhbGYgc25hcHNob3RzLCBzaGFwZS9zdHJpZGVzIGFuZCBsaW5lYXIgcHJpbWl0aXZlIGFzCiAgICAgICAgICAgICMgZm9yd2FyZC4gVGFyZ2V0IHByZWZsaWdodCBjaGVja3MgcmVwZWF0LUdFTU0gYml0IGlkZW50aXR5IGJlZm9yZQogICAgICAgICAgICAjIGFjY2VwdGluZyBzYXZlZC1ub3JtYWxpemVyIHJldXNlLiBObyBSTkcgb3IgZHJvcG91dCBpbiB0aGlzIGhlYWQuCiAgICAgICAgICAgIHJhdz1GLmxpbmVhcihoMTZbc3RhcnQ6c3RvcF0sdzE2KQogICAgICAgICAgICBzdGF0ZT1jdHgudGlsZV9hcGkuVGlsZVN0YXRlKHBsYW4sdHVwbGUoc2F2ZWRbb2Zmc2V0Om9mZnNldCtjb3VudF0pKQogICAgICAgICAgICBkejE2PWN0eC50aWxlX2FwaS5iYWNrd2FyZF90aWxlKHJhdyx0YXJnZXRzW3N0YXJ0OnN0b3BdLHN0YXRlLHVwc3RyZWFtKQogICAgICAgICAgICBpZiBkejE2LmR0eXBlIT10b3JjaC5mbG9hdDE2IG9yIGR6MTYuc2hhcGUhPXJhdy5zaGFwZToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ0UgdGlsZSB2aW9sYXRlZCBuYXRpdmUgaGFsZi1hZGpvaW50IGJvdW5kYXJ5JykKICAgICAgICAgICAgcmVzdWx0W3N0YXJ0OnN0b3BdPShkejE2QHcxNikuZmxvYXQoKQogICAgICAgICAgICBkZWwgcmF3LGR6MTYKICAgICAgICByZXR1cm4gcmVzdWx0LnJlc2hhcGUoY3R4LnNoYXBlKSxOb25lLE5vbmUsTm9uZSxOb25lLE5vbmUKCgpkZWYgdG9rZW5fY2h1bmtfYm91bmRhcnkoaCx3LGxhYmVscyxjaHVuaz0xMjgsZGVub21pbmF0b3I9Tm9uZSxzdW1tYXJ5X3Npbms9Tm9uZSwqLHRpbGVfYXBpPU5vbmUpOgogICAgaWYgdGlsZV9hcGkgaXMgTm9uZToKICAgICAgICBmcm9tIC4gaW1wb3J0IGNlX3RpbGUgYXMgdGlsZV9hcGkKICAgIGxvc3MsZW50cm9weSxubGwscHJlZGljdGlvbnM9X1Rva2VuQ2h1bmtDRS5hcHBseShoLHcsbGFiZWxzLGNodW5rLGRlbm9taW5hdG9yLHRpbGVfYXBpKQogICAgaWYgc3VtbWFyeV9zaW5rIGlzIG5vdCBOb25lOgogICAgICAgIGlmIG5vdCBzdW1tYXJ5X3NpbmsodG9rZW5fbmxsPW5sbCx0b2tlbl9lbnRyb3B5PWVudHJvcHksZmlyc3RfYXJnbWF4PXByZWRpY3Rpb25zLHJvdXRlPSdzdHJlYW1pbmcnKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdTdHJlYW1pbmcgc3VtbWFyeSBwdWJsaWNhdGlvbiBmYWlsZWQnKQogICAgICAgIGlmIG5vdCBzdW1tYXJ5X3NpbmsuY29tcGxldGUobG9zcyk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignU3RyZWFtaW5nIG5vcm1hbGl6ZWQtbG9zcyBpZGVudGl0eSBwdWJsaWNhdGlvbiBmYWlsZWQnKQogICAgcmV0dXJuIGxvc3MsU3VtbWFyaWVzKGVudHJvcHksbmxsLHByZWRpY3Rpb25zKQo=', 'solution/verify_tile.py': 'IiIiQm91bmRlZCBhY3R1YWwtdm9jYWJ1bGFyeSB0YXJnZXQgZ2F0ZSBmb3IgdGhlIGlzb2xhdGVkIG9yaWdpbmFsIENFIHRpbGUgQVBJLgoKTm90IGEgcHJvamVjdGlvbiByZWNvbXB1dGF0aW9uIHByb29mOiB0aGUgc3RyZWFtaW5nIG93bmVyIHNlcGFyYXRlbHkgdmVyaWZpZXMKcmVwZWF0ZWQgZnVsbC1IL1YgR0VNTSBiaXRzIGFuZCBjb21wbGV0ZSBoaWRkZW4gZ3JhZGllbnRzL0dyYWRTY2FsZXIgYmVoYXZpb3IuCiIiIgppbXBvcnQgYXJncGFyc2UsanNvbixvcyx0cmFjZWJhY2sKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCmZyb20gLiBpbXBvcnQgY2VfdGlsZSBhcyB0aWxlCmZyb20gLm9yaWdpbmFsIGltcG9ydCBtYXRlcmlhbGl6ZWRfY2UgYXMgY2UKCgpkZWYgZXhhY3QoYSxiKToKICAgIGlmIGEuc2hhcGUhPWIuc2hhcGUgb3IgYS5kdHlwZSE9Yi5kdHlwZTpyYWlzZSBSdW50aW1lRXJyb3IoJ1NoYXBlIG9yIGR0eXBlIGNoYW5nZWQnKQogICAgaWYgbm90IHRvcmNoLmVxdWFsKGEucmVzaGFwZSgtMSkudmlldyh0b3JjaC51aW50OCksYi5yZXNoYXBlKC0xKS52aWV3KHRvcmNoLnVpbnQ4KSk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdGcm96ZW4gb3JpZ2luYWwgQ0UgZ3JhZGllbnQgYml0cyBjaGFuZ2VkJykKCgpkZWYgbWV0cmljX2Vycm9yKGFjdHVhbCxleHBlY3RlZCxydG9sKToKICAgIGZvciBmIGluICh0b3JjaC5pc25hbix0b3JjaC5pc3Bvc2luZix0b3JjaC5pc25lZ2luZik6CiAgICAgICAgaWYgbm90IHRvcmNoLmVxdWFsKGYoYWN0dWFsKSxmKGV4cGVjdGVkKSk6cmFpc2UgUnVudGltZUVycm9yKCdOYXRpdmUgbWV0cmljIG5vbmZpbml0ZSBjbGFzcyBjaGFuZ2VkJykKICAgIGZpbml0ZT1hY3R1YWwuaXNmaW5pdGUoKSZleHBlY3RlZC5pc2Zpbml0ZSgpCiAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShhY3R1YWxbZmluaXRlXSxleHBlY3RlZFtmaW5pdGVdLHJ0b2w9cnRvbCxhdG9sPTFlLTYpCiAgICByZXR1cm4geydtYXhfYWJzJzpmbG9hdCgoYWN0dWFsW2Zpbml0ZV0tZXhwZWN0ZWRbZmluaXRlXSkuYWJzKCkubWF4KCkpIGlmIGZpbml0ZS5hbnkoKSBlbHNlIDAuLAogICAgICAgICAgICAncnRvbCc6cnRvbCwnYXRvbCc6MWUtNiwnbmF0aXZlX25vbmZpbml0ZV9jbGFzc2VzX2V4YWN0JzpUcnVlfQoKCmRlZiBydW5fY2FzZXMoZGV2aWNlLHByb2dyZXNzKToKICAgIHY9dGlsZS5WT0NBQlVMQVJZO2dlbmVyYXRvcj10b3JjaC5HZW5lcmF0b3IoZGV2aWNlPWRldmljZSkubWFudWFsX3NlZWQoOTExKQogICAgYmFzZT10b3JjaC5yYW5kbig0LDMsdixkZXZpY2U9ZGV2aWNlLGdlbmVyYXRvcj1nZW5lcmF0b3IpLmhhbGYoKQogICAgbGFiZWxzPXRvcmNoLnRlbnNvcihbWzAsMSwyXSxbMCwzLDRdLFswLC0xMDAsNV0sWzAsNiw3XV0sZGV2aWNlPWRldmljZSkKICAgIHJvd3M9W107cHJvZ3Jlc3NbJ3Jvd3MnXT1yb3dzCiAgICBmb3IgbW9kZSBpbiAoJ2FsbF9zYWZlJywnbWl4ZWQnLCdhbGxfbmVhcicsJ2FsbF9pZ25vcmVkJywnaWdub3JlZF9uYW4nKToKICAgICAgICB6PWJhc2UuY2xvbmUoKTt5PWxhYmVscy5jbG9uZSgpO3RhcmdldD1jZS5fdGFyZ2V0cyh5KS5mbGF0dGVuKCkKICAgICAgICBpZiBtb2RlPT0nYWxsX3NhZmUnOnouemVyb18oKQogICAgICAgIGVsaWYgbW9kZSBpbiAoJ21peGVkJywnYWxsX25lYXInKToKICAgICAgICAgICAgY2hvc2VuPVswLDRdIGlmIG1vZGU9PSdtaXhlZCcgZWxzZSBbMCwxLDMsNCw3LDksMTBdCiAgICAgICAgICAgIGZvciBpbmRleCBpbiBjaG9zZW46CiAgICAgICAgICAgICAgICB6LnZpZXcoLTEsdilbaW5kZXhdLnplcm9fKCk7ei52aWV3KC0xLHYpW2luZGV4LHRhcmdldFtpbmRleF1dPTMwLgogICAgICAgIGVsaWYgbW9kZT09J2FsbF9pZ25vcmVkJzp5LmZpbGxfKC0xMDApO3RhcmdldD1jZS5fdGFyZ2V0cyh5KS5mbGF0dGVuKCkKICAgICAgICBlbHNlOnpbMCwyLDBdPWZsb2F0KCduYW4nKQogICAgICAgIGZvciBzY2FsZSBpbiAoMS4sNjU1MzYuKToKICAgICAgICAgICAgcHJvZ3Jlc3NbJ2N1cnJlbnRfY2FzZSddPXsnbW9kZSc6bW9kZSwnc2NhbGUnOnNjYWxlfQogICAgICAgICAgICBvcmlnaW5hbD16LmNsb25lKCkucmVxdWlyZXNfZ3JhZF8oKQogICAgICAgICAgICBsb3NzPWNlLm1hdGVyaWFsaXplZF9jYXVzYWxfbG9zcyhvcmlnaW5hbCx5LDEzKQogICAgICAgICAgICB3YW50ZWQsPXRvcmNoLmF1dG9ncmFkLmdyYWQobG9zcyxvcmlnaW5hbCx0b3JjaC50ZW5zb3Ioc2NhbGUsZGV2aWNlPWRldmljZSkpCiAgICAgICAgICAgIHJlc3VsdD10aWxlLmZvcndhcmRfdGlsZSh6LnJlc2hhcGUoLTEsdiksdGFyZ2V0KQogICAgICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgICAgIHVwc3RyZWFtPXRpbGUucHJlcGFyZV91cHN0cmVhbSh0b3JjaC50ZW5zb3Ioc2NhbGUsZGV2aWNlPWRldmljZSkvMTMpCiAgICAgICAgICAgICAgICBnb3Q9dGlsZS5iYWNrd2FyZF90aWxlKHoucmVzaGFwZSgtMSx2KS5jbG9uZSgpLHRhcmdldCxyZXN1bHQuc3RhdGUsdXBzdHJlYW0pCiAgICAgICAgICAgIGV4YWN0KGdvdC5yZXNoYXBlX2FzKHdhbnRlZCksd2FudGVkKQogICAgICAgICAgICBuYXRpdmVfbmxsPUYuY3Jvc3NfZW50cm9weSh6LmZsb2F0KCkucmVzaGFwZSgtMSx2KSx0YXJnZXQsaWdub3JlX2luZGV4PS0xMDAscmVkdWN0aW9uPSdub25lJykKICAgICAgICAgICAgbHA9Ri5sb2dfc29mdG1heCh6LmZsb2F0KCkucmVzaGFwZSgtMSx2KSwtMSk7ZW50cm9weT0tKGxwLmV4cCgpKmxwKS5zdW0oLTEpCiAgICAgICAgICAgIG5sbF9lcnJvcj1tZXRyaWNfZXJyb3IocmVzdWx0LnRva2VuX25sbCxuYXRpdmVfbmxsLDFlLTQpCiAgICAgICAgICAgIGVudHJvcHlfZXJyb3I9bWV0cmljX2Vycm9yKHJlc3VsdC50b2tlbl9lbnRyb3B5LGVudHJvcHksLjAxKQogICAgICAgICAgICBpZiBub3QgdG9yY2guZXF1YWwocmVzdWx0LmZpcnN0X2FyZ21heCx6LnJlc2hhcGUoLTEsdikuYXJnbWF4KC0xKSk6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0ZpcnN0LWluZGV4IGFyZ21heCBjaGFuZ2VkJykKICAgICAgICAgICAgZXhwZWN0ZWRfcm91dGU9eydhbGxfc2FmZSc6J2FsbF9zYWZlJywnbWl4ZWQnOidtaXhlZF9yb3dzJ30uZ2V0KG1vZGUsJ25hdGl2ZScpCiAgICAgICAgICAgIGlmIHJlc3VsdC5zdGF0ZS5tZXRhZGF0YS5yb3V0ZSE9ZXhwZWN0ZWRfcm91dGU6cmFpc2UgUnVudGltZUVycm9yKCdSZXF1aXJlZCBudW1lcmljYWwgcm91dGUgaW5hY3RpdmUnKQogICAgICAgICAgICBpZiBzdW0odC5udW1lbCgpIGZvciB0IGluIHJlc3VsdC5zdGF0ZS50ZW5zb3JzKT40KnRhcmdldC5udW1lbCgpOnJhaXNlIFJ1bnRpbWVFcnJvcignTm9uLXJvdy1zaXplZCBzYXZlZCBzdGF0ZScpCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsnbW9kZSc6bW9kZSwnc2NhbGUnOnNjYWxlLCdyb3V0ZSc6cmVzdWx0LnN0YXRlLm1ldGFkYXRhLnJvdXRlLAogICAgICAgICAgICAgICAgICAgICAgICAgJ2Zyb3plbl9jZV9ncmFkaWVudF9leGFjdCc6VHJ1ZSwnbmxsJzpubGxfZXJyb3IsJ2VudHJvcHknOmVudHJvcHlfZXJyb3IsCiAgICAgICAgICAgICAgICAgICAgICAgICAnYXJnbWF4X2V4YWN0JzpUcnVlLCdzYXZlZF9zdGF0ZV9lbGVtZW50cyc6c3VtKHQubnVtZWwoKSBmb3IgdCBpbiByZXN1bHQuc3RhdGUudGVuc29ycyl9KQogICAgIyBHZW51aW5lIHNjYWxlLWJlZm9yZS1oYWxmIGRpc2NyaW1pbmF0b3IgYXQgZnVsbCB2b2NhYnVsYXJ5LgogICAgcHJvZ3Jlc3NbJ2N1cnJlbnRfY2FzZSddPSdmdWxsX3ZvY2FidWxhcnlfc2NhbGVfYmVmb3JlX2hhbGZfbmVnYXRpdmUnCiAgICB6PXRvcmNoLnplcm9zKDEsdixkZXZpY2U9ZGV2aWNlLGR0eXBlPXRvcmNoLmZsb2F0MTYpO3RhcmdldD10b3JjaC56ZXJvcygxLGRldmljZT1kZXZpY2UsZHR5cGU9dG9yY2guaW50NjQpCiAgICByZXN1bHQ9dGlsZS5mb3J3YXJkX3RpbGUoeix0YXJnZXQpCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBvcmRpbmFyeT10aWxlLmJhY2t3YXJkX3RpbGUoeix0YXJnZXQscmVzdWx0LnN0YXRlLHRpbGUucHJlcGFyZV91cHN0cmVhbSh0b3JjaC50ZW5zb3IoMS4sZGV2aWNlPWRldmljZSkvMjA0OCkpCiAgICAgICAgc2NhbGVkPXRpbGUuYmFja3dhcmRfdGlsZSh6LHRhcmdldCxyZXN1bHQuc3RhdGUsdGlsZS5wcmVwYXJlX3Vwc3RyZWFtKHRvcmNoLnRlbnNvcig2NTUzNi4sZGV2aWNlPWRldmljZSkvMjA0OCkpCiAgICBpZiAodG9yY2guY291bnRfbm9uemVybyhvcmRpbmFyeVswLDE6XSkhPTAgb3IgdG9yY2guY291bnRfbm9uemVybyhzY2FsZWRbMCwxOl0pIT12LTEKICAgICAgICAgICAgb3IgdG9yY2guY291bnRfbm9uemVybygob3JkaW5hcnkuZmxvYXQoKSo2NTUzNilbMCwxOl0pIT0wKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1NjYWxlLWJlZm9yZS1oYWxmIG5lZ2F0aXZlIHdpdG5lc3Mgbm90IGVzdGFibGlzaGVkJykKICAgIHByb2dyZXNzWydzY2FsZV9iZWZvcmVfaGFsZl9uZWdhdGl2ZSddPVRydWUKICAgIHJldHVybiB7J3Jvd3MnOnJvd3MsJ3NjYWxlX2JlZm9yZV9oYWxmX25lZ2F0aXZlJzpUcnVlLCdzYXZlZF9zdGF0ZV9vbmx5X3Jvd19zaXplZCc6VHJ1ZSwKICAgICAgICAgICAgJ3Byb2plY3Rpb25fcmVjb21wdXRlX3ZlcmlmaWVkX2hlcmUnOkZhbHNlLCd0YXNrX3F1YWxpdHknOidVTktOT1dOJywnYWRtaXNzaW9uJzonVU5LTk9XTid9CgoKZGVmIHJ1bihwcm9ncmVzcz1Ob25lKToKICAgIGltcG9ydCB0cml0b24KICAgIGlmIG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyk6cmFpc2UgUnVudGltZUVycm9yKCdJbnRlcnByZXRlciBpcyBub3QgdGFyZ2V0IGV2aWRlbmNlJykKICAgIGlmIHRvcmNoLl9fdmVyc2lvbl9fIT0nMi4xMS4wK2N1MTI4JyBvciB0cml0b24uX192ZXJzaW9uX18hPSczLjYuMCc6cmFpc2UgUnVudGltZUVycm9yKCdQaW5uZWQgcnVudGltZSByZXF1aXJlZCcpCiAgICBpZiAodG9yY2guY3VkYS5kZXZpY2VfY291bnQoKSE9MSBvciB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSE9J1Rlc2xhIFQ0JwogICAgICAgICAgICBvciB0b3JjaC5jdWRhLmdldF9kZXZpY2VfY2FwYWJpbGl0eSgwKSE9KDcsNSkpOnJhaXNlIFJ1bnRpbWVFcnJvcignUGlubmVkIG9uZSBUNCByZXF1aXJlZCcpCiAgICBwcm9ncmVzcz17fSBpZiBwcm9ncmVzcyBpcyBOb25lIGVsc2UgcHJvZ3Jlc3MKICAgIGNwdV9ybmc9dG9yY2guZ2V0X3JuZ19zdGF0ZSgpLmNsb25lKCk7Y3VkYV9ybmc9dG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlKCkuY2xvbmUoKQogICAgcmVzdWx0PXJ1bl9jYXNlcyh0b3JjaC5kZXZpY2UoJ2N1ZGE6MCcpLHByb2dyZXNzKQogICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpCiAgICBpZiBub3QgdG9yY2guZXF1YWwoY3B1X3JuZyx0b3JjaC5nZXRfcm5nX3N0YXRlKCkpIG9yIG5vdCB0b3JjaC5lcXVhbChjdWRhX3JuZyx0b3JjaC5jdWRhLmdldF9ybmdfc3RhdGUoKSk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdHbG9iYWwgUk5HIGNoYW5nZWQnKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBtYWluKCk6CiAgICBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7cC5hZGRfYXJndW1lbnQoJy0tb3V0JyxyZXF1aXJlZD1UcnVlKTthPXAucGFyc2VfYXJncygpCiAgICByZXBvcnQ9eydzdGF0dXMnOidGQUlMRURfVElMRV9QUkVGTElHSFRfT05MWScsJ3Byb2dyZXNzJzp7fSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogICAgdHJ5OnJlcG9ydFsnY2hlY2tzJ109cnVuKHJlcG9ydFsncHJvZ3Jlc3MnXSk7cmVwb3J0WydzdGF0dXMnXT0nUEFTU0VEX1RJTEVfUFJFRkxJR0hUX09OTFknCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBlOnJlcG9ydC51cGRhdGUoZXJyb3I9cmVwcihlKSx0cmFjZWJhY2s9dHJhY2ViYWNrLmZvcm1hdF9leGMoKSkKICAgIFBhdGgoYS5vdXQpLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQsaW5kZW50PTIsYWxsb3dfbmFuPUZhbHNlKSsnXG4nKQogICAgcmFpc2UgU3lzdGVtRXhpdChyZXBvcnRbJ3N0YXR1cyddIT0nUEFTU0VEX1RJTEVfUFJFRkxJR0hUX09OTFknKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'shared_consumer/__init__.py': '', 'shared_consumer/bridge_state.py': 'IiIiUGVyLWNhbGwgZGV0YWNoZWQgbWV0cmljIG93bmVyc2hpcC4gTm8gbW9kZWwvVHJhaW5lciBwYXRjaCBvciB0ZW5zb3IgdmFsdWVzIHJlYWQuCgpDYWxsZXIgbXVzdCBmaXJzdCB2ZXJpZnkgdGhlIG5hdGl2ZSBRd2VuLT5QRUZULT5BY2NlbGVyYXRlIHNvdXJjZS9jYWxsYWJsZSBjaGFpbi4KVGhhdCBwcm9vZiwgb25lIHByb2R1Y2VyLCBhbmQgb25lIGRpcmVjdCBIRi1wYXJlbnQgcmV0dXJuIGJpbmQgaGFsZiBzb3VyY2UgbG9naXRzCnRvIHRoZSBwdWJsaWMgRlAzMiB0ZW5zb3IuIFBFRlQgYnlwYXNzZXMgaW5uZXIgUXdlbiBtb2R1bGUgaG9va3MsIHNvIG5vbmUgYXJlIHVzZWQuCiIiIgpmcm9tIGNvbnRleHRsaWIgaW1wb3J0IGNvbnRleHRtYW5hZ2VyCmZyb20gY29udGV4dHZhcnMgaW1wb3J0IENvbnRleHRWYXIKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmltcG9ydCB3ZWFrcmVmCmltcG9ydCB0b3JjaAoKX0FDVElWRT1Db250ZXh0VmFyKCdzb3VwX2NlX21ldHJpY190cmFuc2FjdGlvbicsZGVmYXVsdD1Ob25lKQpfRklFTERTPSgndG9rZW5fZW50cm9weScsJ2ZpcnN0X2FyZ21heCcsJ3Rva2VuX25sbCcpCgoKZGVmIG1ldGFkYXRhKHQpOgogICAgcmV0dXJuICh0dXBsZSh0LnNoYXBlKSx0dXBsZSh0LnN0cmlkZSgpKSx0LmR0eXBlLHQuZGV2aWNlLHQuc3RvcmFnZV9vZmZzZXQoKSx0Ll92ZXJzaW9uKQoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIERldGFjaGVkU3VtbWFyeToKICAgIHRva2VuX25sbDogdG9yY2guVGVuc29yCiAgICB0b2tlbl9lbnRyb3B5OiB0b3JjaC5UZW5zb3IKICAgIGZpcnN0X2FyZ21heDogdG9yY2guVGVuc29yCiAgICByb3V0ZTogc3RyCgoKY2xhc3MgVHJhbnNhY3Rpb246CiAgICBkZWYgX19pbml0X18oc2VsZixvd25lcixsYWJlbHMsdm9jYWJ1bGFyeT1Ob25lKToKICAgICAgICBzZWxmLm93bmVyX3JlZj13ZWFrcmVmLnJlZihvd25lcikKICAgICAgICBzZWxmLmxhYmVsc19yZWY9d2Vha3JlZi5yZWYobGFiZWxzKQogICAgICAgIHNlbGYubGFiZWxzX21ldGFkYXRhPW1ldGFkYXRhKGxhYmVscykKICAgICAgICBzZWxmLnJhd19yZWY9Tm9uZTtzZWxmLnJhd19tZXRhZGF0YT1Ob25lO3NlbGYucHVibGljX3JlZj1Ob25lO3NlbGYucHVibGljX21ldGFkYXRhPU5vbmUKICAgICAgICBzZWxmLnNvdXJjZV9raW5kPU5vbmU7c2VsZi5zb3VyY2Vfc2hhcGU9Tm9uZTtzZWxmLnNvdXJjZV9kZXZpY2U9Tm9uZTtzZWxmLnZvY2FidWxhcnk9dm9jYWJ1bGFyeQogICAgICAgIHNlbGYubG9zc19yZWY9Tm9uZTtzZWxmLmxvc3NfbWV0YWRhdGE9Tm9uZTtzZWxmLmNvbXBsZXRlZD1GYWxzZQogICAgICAgIHNlbGYuc3VtbWFyeT1Ob25lO3NlbGYuc3VtbWFyeV9tZXRhZGF0YT17fTtzZWxmLnJlcXVlc3RlZD1GYWxzZTtzZWxmLm9mZmVyZWQ9RmFsc2UKICAgICAgICBzZWxmLmJvdW5kPUZhbHNlO3NlbGYuY2xvc2VkPUZhbHNlO3NlbGYuaW52YWxpZF9yZWFzb249Tm9uZTtzZWxmLmNvbnN1bWVkPXNldCgpCgogICAgZGVmIGludmFsaWRhdGUoc2VsZixyZWFzb24pOgogICAgICAgIHNlbGYuaW52YWxpZF9yZWFzb249cmVhc29uCiAgICAgICAgc2VsZi5zdW1tYXJ5PU5vbmU7c2VsZi5zdW1tYXJ5X21ldGFkYXRhLmNsZWFyKCkKICAgICAgICByZXR1cm4gTm9uZQoKICAgIGRlZiBvZmZlcihzZWxmLCosdG9rZW5fbmxsLHRva2VuX2VudHJvcHksZmlyc3RfYXJnbWF4LHJvdXRlKToKICAgICAgICBpZiBzZWxmLmNsb3NlZCBvciBfQUNUSVZFLmdldCgpIGlzIG5vdCBzZWxmIG9yIHNlbGYuaW52YWxpZF9yZWFzb24gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIGlmIHNlbGYub2ZmZXJlZDoKICAgICAgICAgICAgc2VsZi5pbnZhbGlkYXRlKCdkdXBsaWNhdGVfcHJvZHVjZXJfb2ZmZXInKTtyZXR1cm4gRmFsc2UKICAgICAgICBzZWxmLm9mZmVyZWQ9VHJ1ZQogICAgICAgIGxhYmVscz1zZWxmLmxhYmVsc19yZWYoKQogICAgICAgIGlmIHNlbGYuc291cmNlX3NoYXBlIGlzIE5vbmUgb3IgbGFiZWxzIGlzIE5vbmUgb3IgbWV0YWRhdGEobGFiZWxzKSE9c2VsZi5sYWJlbHNfbWV0YWRhdGE6CiAgICAgICAgICAgIHNlbGYuaW52YWxpZGF0ZSgncHJvZHVjZXJfbGFiZWxfb3Jfc291cmNlX2NoYW5nZWQnKTtyZXR1cm4gRmFsc2UKICAgICAgICBleHBlY3RlZF9zaGFwZT1zZWxmLnNvdXJjZV9zaGFwZVs6LTFdO2V4cGVjdGVkX2RldmljZT1zZWxmLnNvdXJjZV9kZXZpY2UKICAgICAgICB2YWx1ZXM9eyd0b2tlbl9ubGwnOnRva2VuX25sbCwndG9rZW5fZW50cm9weSc6dG9rZW5fZW50cm9weSwnZmlyc3RfYXJnbWF4JzpmaXJzdF9hcmdtYXh9CiAgICAgICAgZm9yIG5hbWUsdmFsdWUgaW4gdmFsdWVzLml0ZW1zKCk6CiAgICAgICAgICAgIGR0eXBlPXRvcmNoLmludDY0IGlmIG5hbWU9PSdmaXJzdF9hcmdtYXgnIGVsc2UgdG9yY2guZmxvYXQzMgogICAgICAgICAgICBpZiAodHlwZSh2YWx1ZSkgaXMgbm90IHRvcmNoLlRlbnNvciBvciB2YWx1ZS5yZXF1aXJlc19ncmFkIG9yIHZhbHVlLmdyYWRfZm4gaXMgbm90IE5vbmUKICAgICAgICAgICAgICAgICAgICBvciB0dXBsZSh2YWx1ZS5zaGFwZSkhPWV4cGVjdGVkX3NoYXBlIG9yIHZhbHVlLmRldmljZSE9ZXhwZWN0ZWRfZGV2aWNlCiAgICAgICAgICAgICAgICAgICAgb3IgdmFsdWUuZHR5cGUhPWR0eXBlIG9yIG5vdCB2YWx1ZS5pc19jb250aWd1b3VzKCkgb3IgdmFsdWUuaXNfbmVnKCkgb3IgdmFsdWUuaXNfY29uaigpKToKICAgICAgICAgICAgICAgIHNlbGYuaW52YWxpZGF0ZSgnc3VtbWFyeV9tZXRhZGF0YTonK25hbWUpO3JldHVybiBGYWxzZQogICAgICAgIGlmIHJvdXRlIG5vdCBpbiAoJ2FsbF9zYWZlJywnbWl4ZWRfcm93cycsJ3N0cmVhbWluZycpOgogICAgICAgICAgICBzZWxmLmludmFsaWRhdGUoJ3N1bW1hcnlfcm91dGUnKTtyZXR1cm4gRmFsc2UKICAgICAgICBzZWxmLnN1bW1hcnk9RGV0YWNoZWRTdW1tYXJ5KCoqdmFsdWVzLHJvdXRlPXJvdXRlKQogICAgICAgIHNlbGYuc3VtbWFyeV9tZXRhZGF0YT17bjptZXRhZGF0YSh2KSBmb3Igbix2IGluIHZhbHVlcy5pdGVtcygpfQogICAgICAgIHJldHVybiBUcnVlCgogICAgZGVmIGNvbXBsZXRlKHNlbGYsbG9zcyk6CiAgICAgICAgaWYgc2VsZi5jbG9zZWQgb3IgX0FDVElWRS5nZXQoKSBpcyBub3Qgc2VsZiBvciBzZWxmLmludmFsaWRfcmVhc29uIGlzIG5vdCBOb25lOnJldHVybiBGYWxzZQogICAgICAgIGlmIHNlbGYuY29tcGxldGVkOgogICAgICAgICAgICBzZWxmLmludmFsaWRhdGUoJ2R1cGxpY2F0ZV9sb3NzX2NvbXBsZXRpb24nKTtyZXR1cm4gRmFsc2UKICAgICAgICBzZWxmLmNvbXBsZXRlZD1UcnVlCiAgICAgICAgaWYgKHNlbGYuc3VtbWFyeSBpcyBOb25lIG9yIHR5cGUobG9zcykgaXMgbm90IHRvcmNoLlRlbnNvciBvciBsb3NzLmR0eXBlIT10b3JjaC5mbG9hdDMyCiAgICAgICAgICAgICAgICBvciBsb3NzLm5kaW0hPTAgb3Igbm90IGxvc3MucmVxdWlyZXNfZ3JhZCBvciBsb3NzLmRldmljZSE9c2VsZi5zb3VyY2VfZGV2aWNlKToKICAgICAgICAgICAgc2VsZi5pbnZhbGlkYXRlKCdwcm9kdWNlcl9sb3NzX2NvbnRyYWN0Jyk7cmV0dXJuIEZhbHNlCiAgICAgICAgc2VsZi5sb3NzX3JlZj13ZWFrcmVmLnJlZihsb3NzKQogICAgICAgIHJldHVybiBUcnVlCgogICAgZGVmIGJpbmRfb3V0cHV0cyhzZWxmLGxvc3Msb3V0cHV0cyk6CiAgICAgICAgaWYgc2VsZi5jbG9zZWQgb3IgX0FDVElWRS5nZXQoKSBpcyBub3Qgc2VsZiBvciBzZWxmLmludmFsaWRfcmVhc29uIGlzIG5vdCBOb25lOnJldHVybiBGYWxzZQogICAgICAgIGlmIHNlbGYuYm91bmQ6CiAgICAgICAgICAgIHNlbGYuaW52YWxpZGF0ZSgnZHVwbGljYXRlX3B1YmxpY19iaW5kJyk7cmV0dXJuIEZhbHNlCiAgICAgICAgc2VsZi5ib3VuZD1UcnVlCiAgICAgICAgaWYgKHNlbGYuc3VtbWFyeSBpcyBOb25lIG9yIHNlbGYubG9zc19yZWYgaXMgTm9uZSBvciBzZWxmLmxvc3NfcmVmKCkgaXMgbm90IGxvc3MKICAgICAgICAgICAgICAgIG9yIGdldGF0dHIob3V0cHV0cywnbG9zcycsTm9uZSkgaXMgbm90IGxvc3MpOgogICAgICAgICAgICBzZWxmLmludmFsaWRhdGUoJ25vX2NvbXBsZXRlZF9wcm9kdWNlcl9vcl9sb3NzX2lkZW50aXR5Jyk7cmV0dXJuIEZhbHNlCiAgICAgICAgbGFiZWxzPXNlbGYubGFiZWxzX3JlZigpO3B1YmxpY19sb2dpdHM9Z2V0YXR0cihvdXRwdXRzLCdsb2dpdHMnLE5vbmUpCiAgICAgICAgaWYgbGFiZWxzIGlzIE5vbmUgb3IgbWV0YWRhdGEobGFiZWxzKSE9c2VsZi5sYWJlbHNfbWV0YWRhdGE6CiAgICAgICAgICAgIHNlbGYuaW52YWxpZGF0ZSgncHVibGljX29yX2xhYmVsX21ldGFkYXRhJyk7cmV0dXJuIEZhbHNlCiAgICAgICAgaWYgc2VsZi5zb3VyY2Vfa2luZD09J21hdGVyaWFsaXplZCc6CiAgICAgICAgICAgIGlmICh0eXBlKHB1YmxpY19sb2dpdHMpIGlzIG5vdCB0b3JjaC5UZW5zb3Igb3IgcHVibGljX2xvZ2l0cy5kdHlwZSE9dG9yY2guZmxvYXQzMgogICAgICAgICAgICAgICAgICAgIG9yIHR1cGxlKHB1YmxpY19sb2dpdHMuc2hhcGUpIT1zZWxmLnNvdXJjZV9zaGFwZSBvciBwdWJsaWNfbG9naXRzLmRldmljZSE9c2VsZi5zb3VyY2VfZGV2aWNlCiAgICAgICAgICAgICAgICAgICAgb3Igbm90IHB1YmxpY19sb2dpdHMuaXNfY29udGlndW91cygpIG9yIHB1YmxpY19sb2dpdHMuaXNfbmVnKCkgb3IgcHVibGljX2xvZ2l0cy5pc19jb25qKCkpOgogICAgICAgICAgICAgICAgc2VsZi5pbnZhbGlkYXRlKCdwdWJsaWNfb3JfbGFiZWxfbWV0YWRhdGEnKTtyZXR1cm4gRmFsc2UKICAgICAgICAgICAgc2VsZi5wdWJsaWNfcmVmPXdlYWtyZWYucmVmKHB1YmxpY19sb2dpdHMpO3NlbGYucHVibGljX21ldGFkYXRhPW1ldGFkYXRhKHB1YmxpY19sb2dpdHMpCiAgICAgICAgZWxpZiBzZWxmLnNvdXJjZV9raW5kPT0nc3RyZWFtaW5nJzoKICAgICAgICAgICAgaWYgcHVibGljX2xvZ2l0cyBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHNlbGYuaW52YWxpZGF0ZSgnc3RyZWFtaW5nX3VuZXhwZWN0ZWRfZnVsbF9sb2dpdHMnKTtyZXR1cm4gRmFsc2UKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLmludmFsaWRhdGUoJ3Vua25vd25fc291cmNlX2tpbmQnKTtyZXR1cm4gRmFsc2UKICAgICAgICAjIE5hdGl2ZSBBY2NlbGVyYXRlIHByZXNlcnZlcyBGUDMyIHNjYWxhciBUZW5zb3IgaWRlbnRpdHkuIEhGJ3Mgb3B0aW9uYWwKICAgICAgICAjIG5hdGl2ZSBpbi1wbGFjZSBsb3NzIHNjYWxlIHJldGFpbnMgdGhhdCBpZGVudGl0eTsgc3RhbXAgaXRzIGN1cnJlbnQKICAgICAgICAjIG1ldGFkYXRhIG5vdywgbm90IGJlZm9yZSB0aGF0IGtub3duIG5hdGl2ZSBvcGVyYXRpb24uCiAgICAgICAgc2VsZi5sb3NzX21ldGFkYXRhPW1ldGFkYXRhKGxvc3MpCiAgICAgICAgcmV0dXJuIFRydWUKCiAgICBkZWYgdGFrZShzZWxmLG5hbWUscHVibGljX2xvZ2l0cyxsYWJlbHMpOgogICAgICAgIGlmIChuYW1lIG5vdCBpbiBfRklFTERTIG9yIHNlbGYuY2xvc2VkIG9yIF9BQ1RJVkUuZ2V0KCkgaXMgbm90IHNlbGYKICAgICAgICAgICAgICAgIG9yIHNlbGYuaW52YWxpZF9yZWFzb24gaXMgbm90IE5vbmUgb3Igc2VsZi5zdW1tYXJ5IGlzIE5vbmUpOnJldHVybiBOb25lCiAgICAgICAgaWYgbmFtZSBpbiBzZWxmLmNvbnN1bWVkOgogICAgICAgICAgICByZXR1cm4gc2VsZi5pbnZhbGlkYXRlKCdkdXBsaWNhdGVfY29uc3VtZXI6JytuYW1lKQogICAgICAgIG9yaWdpbmFsX2xhYmVscz1zZWxmLmxhYmVsc19yZWYoKQogICAgICAgIHB1YmxpY19vaz0oc2VsZi5zb3VyY2Vfa2luZD09InN0cmVhbWluZyIgYW5kIHB1YmxpY19sb2dpdHMgaXMgTm9uZSkgb3IgKAogICAgICAgICAgICBzZWxmLnNvdXJjZV9raW5kPT0ibWF0ZXJpYWxpemVkIiBhbmQgc2VsZi5wdWJsaWNfcmVmIGlzIG5vdCBOb25lCiAgICAgICAgICAgIGFuZCBzZWxmLnB1YmxpY19yZWYoKSBpcyBwdWJsaWNfbG9naXRzIGFuZCBtZXRhZGF0YShwdWJsaWNfbG9naXRzKT09c2VsZi5wdWJsaWNfbWV0YWRhdGEpCiAgICAgICAgbG9zcz1zZWxmLmxvc3NfcmVmKCkgaWYgc2VsZi5sb3NzX3JlZiBpcyBub3QgTm9uZSBlbHNlIE5vbmUKICAgICAgICBpZiAobm90IHB1YmxpY19vayBvciBsb3NzIGlzIE5vbmUgb3IgbWV0YWRhdGEobG9zcykhPXNlbGYubG9zc19tZXRhZGF0YQogICAgICAgICAgICAgICAgb3Igb3JpZ2luYWxfbGFiZWxzIGlzIG5vdCBsYWJlbHMgb3IgbWV0YWRhdGEobGFiZWxzKSE9c2VsZi5sYWJlbHNfbWV0YWRhdGEpOgogICAgICAgICAgICByZXR1cm4gc2VsZi5pbnZhbGlkYXRlKCdjb25zdW1lcl9pZGVudGl0eV9vcl92ZXJzaW9uJykKICAgICAgICBmb3IgZmllbGQgaW4gX0ZJRUxEUzoKICAgICAgICAgICAgaWYgbWV0YWRhdGEoZ2V0YXR0cihzZWxmLnN1bW1hcnksZmllbGQpKSE9c2VsZi5zdW1tYXJ5X21ldGFkYXRhW2ZpZWxkXToKICAgICAgICAgICAgICAgIHJldHVybiBzZWxmLmludmFsaWRhdGUoJ3N1bW1hcnlfbW9kaWZpZWQ6JytmaWVsZCkKICAgICAgICBzZWxmLmNvbnN1bWVkLmFkZChuYW1lKQogICAgICAgIHJldHVybiBnZXRhdHRyKHNlbGYuc3VtbWFyeSxuYW1lKQoKICAgIGRlZiBjbG9zZShzZWxmKToKICAgICAgICBzZWxmLmNsb3NlZD1UcnVlO3NlbGYuc3VtbWFyeT1Ob25lO3NlbGYuc3VtbWFyeV9tZXRhZGF0YS5jbGVhcigpCiAgICAgICAgc2VsZi5yYXdfcmVmPU5vbmU7c2VsZi5wdWJsaWNfcmVmPU5vbmU7c2VsZi5sb3NzX3JlZj1Ob25lCiAgICAgICAgc2VsZi5wcm9kdWNlcl9vd25lcl9yZWY9Tm9uZQogICAgICAgIGlmIGhhc2F0dHIoc2VsZiwnY2h1bmtlZF9wYWNrZXQnKTpkZWwgc2VsZi5jaHVua2VkX3BhY2tldAoKCkBjb250ZXh0bWFuYWdlcgpkZWYgc2NvcGUob3duZXIsbGFiZWxzLCosdm9jYWJ1bGFyeT1Ob25lKToKICAgIGlmIF9BQ1RJVkUuZ2V0KCkgaXMgbm90IE5vbmU6cmFpc2UgUnVudGltZUVycm9yKCdOZXN0ZWQgbWV0cmljIG93bmVyc2hpcCByZXF1aXJlcyBuYXRpdmUgc3VzcGVuZGVkIGZhbGxiYWNrJykKICAgIHRyYW5zYWN0aW9uPVRyYW5zYWN0aW9uKG93bmVyLGxhYmVscyx2b2NhYnVsYXJ5KTt0b2tlbj1fQUNUSVZFLnNldCh0cmFuc2FjdGlvbikKICAgIHRyeTp5aWVsZCB0cmFuc2FjdGlvbgogICAgZmluYWxseTp0cmFuc2FjdGlvbi5jbG9zZSgpO19BQ1RJVkUucmVzZXQodG9rZW4pCgoKQGNvbnRleHRtYW5hZ2VyCmRlZiBzdXNwZW5kZWQoKToKICAgIHRva2VuPV9BQ1RJVkUuc2V0KE5vbmUpCiAgICB0cnk6eWllbGQKICAgIGZpbmFsbHk6X0FDVElWRS5yZXNldCh0b2tlbikKCgpkZWYgcmVxdWVzdF9zaW5rKG93bmVyLHJhd19sb2dpdHMsbGFiZWxzKToKICAgIHRyYW5zYWN0aW9uPV9BQ1RJVkUuZ2V0KCkKICAgIGlmIHRyYW5zYWN0aW9uIGlzIE5vbmUgb3IgdHJhbnNhY3Rpb24uY2xvc2VkIG9yIHRyYW5zYWN0aW9uLmludmFsaWRfcmVhc29uIGlzIG5vdCBOb25lOnJldHVybiBOb25lCiAgICBpZiB0cmFuc2FjdGlvbi5yZXF1ZXN0ZWQ6CiAgICAgICAgdHJhbnNhY3Rpb24uaW52YWxpZGF0ZSgnbXVsdGlwbGVfbG9zc19wcm9kdWNlcnMnKTtyZXR1cm4gTm9uZQogICAgaWYgKHRyYW5zYWN0aW9uLm93bmVyX3JlZigpIGlzIG5vdCBvd25lciBvciB0cmFuc2FjdGlvbi5sYWJlbHNfcmVmKCkgaXMgbm90IGxhYmVscwogICAgICAgICAgICBvciB0eXBlKHJhd19sb2dpdHMpIGlzIG5vdCB0b3JjaC5UZW5zb3Igb3IgcmF3X2xvZ2l0cy5kdHlwZSE9dG9yY2guZmxvYXQxNgogICAgICAgICAgICBvciByYXdfbG9naXRzLm5kaW0hPTMgb3IgcmF3X2xvZ2l0cy5zaGFwZVs6LTFdIT1sYWJlbHMuc2hhcGUKICAgICAgICAgICAgb3IgcmF3X2xvZ2l0cy5kZXZpY2UhPWxhYmVscy5kZXZpY2Ugb3Igbm90IHJhd19sb2dpdHMuaXNfY29udGlndW91cygpCiAgICAgICAgICAgIG9yIHJhd19sb2dpdHMuaXNfbmVnKCkgb3IgcmF3X2xvZ2l0cy5pc19jb25qKCkgb3IgbWV0YWRhdGEobGFiZWxzKSE9dHJhbnNhY3Rpb24ubGFiZWxzX21ldGFkYXRhKToKICAgICAgICB0cmFuc2FjdGlvbi5pbnZhbGlkYXRlKCdwcm9kdWNlcl9pZGVudGl0eV9vcl9tZXRhZGF0YScpO3JldHVybiBOb25lCiAgICBpZiB0cmFuc2FjdGlvbi52b2NhYnVsYXJ5IGlzIG5vdCBOb25lIGFuZCByYXdfbG9naXRzLnNoYXBlWy0xXSE9dHJhbnNhY3Rpb24udm9jYWJ1bGFyeToKICAgICAgICB0cmFuc2FjdGlvbi5pbnZhbGlkYXRlKCJwcm9kdWNlcl92b2NhYnVsYXJ5Iik7cmV0dXJuIE5vbmUKICAgIHRyYW5zYWN0aW9uLnJlcXVlc3RlZD1UcnVlO3RyYW5zYWN0aW9uLnJhd19yZWY9d2Vha3JlZi5yZWYocmF3X2xvZ2l0cykKICAgIHRyYW5zYWN0aW9uLnJhd19tZXRhZGF0YT1tZXRhZGF0YShyYXdfbG9naXRzKTt0cmFuc2FjdGlvbi5zb3VyY2Vfa2luZD0ibWF0ZXJpYWxpemVkIgogICAgdHJhbnNhY3Rpb24uc291cmNlX3NoYXBlPXR1cGxlKHJhd19sb2dpdHMuc2hhcGUpO3RyYW5zYWN0aW9uLnNvdXJjZV9kZXZpY2U9cmF3X2xvZ2l0cy5kZXZpY2UKICAgIHJldHVybiBQcm9kdWNlclNpbmsodHJhbnNhY3Rpb24pCgoKZGVmIGN1cnJlbnQoKTpyZXR1cm4gX0FDVElWRS5nZXQoKQoKCmNsYXNzIFByb2R1Y2VyU2luazoKICAgIGRlZiBfX2luaXRfXyhzZWxmLHRyYW5zYWN0aW9uKTpzZWxmLnRyYW5zYWN0aW9uPXRyYW5zYWN0aW9uCiAgICBkZWYgX19jYWxsX18oc2VsZiwqKmZpZWxkcyk6cmV0dXJuIHNlbGYudHJhbnNhY3Rpb24ub2ZmZXIoKipmaWVsZHMpCiAgICBkZWYgY29tcGxldGUoc2VsZixsb3NzKTpyZXR1cm4gc2VsZi50cmFuc2FjdGlvbi5jb21wbGV0ZShsb3NzKQoKCmRlZiByZXF1ZXN0X3N0cmVhbWluZ19zaW5rKG93bmVyLGxhYmVscywqLHZvY2FidWxhcnkpOgogICAgdHJhbnNhY3Rpb249X0FDVElWRS5nZXQoKQogICAgaWYgdHJhbnNhY3Rpb24gaXMgTm9uZSBvciB0cmFuc2FjdGlvbi5jbG9zZWQgb3IgdHJhbnNhY3Rpb24uaW52YWxpZF9yZWFzb24gaXMgbm90IE5vbmU6cmV0dXJuIE5vbmUKICAgIGlmIHRyYW5zYWN0aW9uLnJlcXVlc3RlZDoKICAgICAgICB0cmFuc2FjdGlvbi5pbnZhbGlkYXRlKCdtdWx0aXBsZV9sb3NzX3Byb2R1Y2VycycpO3JldHVybiBOb25lCiAgICBpZiAodHJhbnNhY3Rpb24ub3duZXJfcmVmKCkgaXMgbm90IG93bmVyIG9yIHRyYW5zYWN0aW9uLmxhYmVsc19yZWYoKSBpcyBub3QgbGFiZWxzCiAgICAgICAgICAgIG9yIG1ldGFkYXRhKGxhYmVscykhPXRyYW5zYWN0aW9uLmxhYmVsc19tZXRhZGF0YSBvciB0eXBlKHZvY2FidWxhcnkpIGlzIG5vdCBpbnQKICAgICAgICAgICAgb3Igdm9jYWJ1bGFyeTw9MSBvciB0cmFuc2FjdGlvbi52b2NhYnVsYXJ5IT12b2NhYnVsYXJ5KToKICAgICAgICB0cmFuc2FjdGlvbi5pbnZhbGlkYXRlKCdzdHJlYW1pbmdfcHJvZHVjZXJfaWRlbnRpdHlfb3JfbWV0YWRhdGEnKTtyZXR1cm4gTm9uZQogICAgdHJhbnNhY3Rpb24ucmVxdWVzdGVkPVRydWU7dHJhbnNhY3Rpb24uc291cmNlX2tpbmQ9J3N0cmVhbWluZycKICAgIHRyYW5zYWN0aW9uLnNvdXJjZV9zaGFwZT10dXBsZShsYWJlbHMuc2hhcGUpKyh2b2NhYnVsYXJ5LCk7dHJhbnNhY3Rpb24uc291cmNlX2RldmljZT1sYWJlbHMuZGV2aWNlCiAgICByZXR1cm4gUHJvZHVjZXJTaW5rKHRyYW5zYWN0aW9uKQo=', 'shared_consumer/consumer.py': 'IiIiQm91bmRlZCBjb25zdW1lciBjb21wb25lbnQ7IG5vIHByb2R1Y3Rpb24gaW5zdGFsbGVyIG9yIGF1dG9tYXRpYyBwYXRjaGluZy4KClRoZSBjYWxsZXIgbXVzdCBlc3RhYmxpc2ggbmF0aXZlIHNvdXJjZSBvd25lcnNoaXAgYW5kIHN1cHBseSBhIG1ldGFkYXRhIGd1YXJkLgpPbmx5IGFuIG93bmVkIG91dGVyIFJld2luZCBsb3NzLW9ubHkgdHJhbnNhY3Rpb24gbWF5IHVzZSBkZXRhY2hlZCBzdW1tYXJpZXMuCk5hdGl2ZSBIRiBzdGlsbCBleGVjdXRlcyB0aGUgbW9kZWwgZXhhY3RseSBvbmNlLiBVbnN1cHBvcnRlZCBwcmUtZm9yd2FyZCBjYWxscwpyZW1haW4gZnVsbHkgbmF0aXZlOyBwb3N0LWZvcndhcmQgbWF0ZXJpYWxpemVkIG1pc3NlcyB1c2UgdGhlIFNBTUUgbG9naXRzLgoiIiIKZnJvbSBmdW5jdG9vbHMgaW1wb3J0IHdyYXBzCmZyb20gdHlwZXMgaW1wb3J0IFNpbXBsZU5hbWVzcGFjZQppbXBvcnQgd2Vha3JlZgppbXBvcnQgdG9yY2gKZnJvbSAuIGltcG9ydCBicmlkZ2Vfc3RhdGUgYXMgc3RhdGUKCgpkZWYgbWF0ZXJpYWxpemVkX3NpbmsoaW5uZXJfb3duZXIscmF3X2xvZ2l0cyxsYWJlbHMpOgogICAgdHJhbnNhY3Rpb249c3RhdGUuY3VycmVudCgpCiAgICBpZiB0cmFuc2FjdGlvbiBpcyBOb25lIG9yIGdldGF0dHIodHJhbnNhY3Rpb24sJ3Byb2R1Y2VyX293bmVyX3JlZicsbGFtYmRhOk5vbmUpKCkgaXMgbm90IGlubmVyX293bmVyOgogICAgICAgIHJldHVybiBOb25lCiAgICBvdXRlcj10cmFuc2FjdGlvbi5vd25lcl9yZWYoKQogICAgcmV0dXJuIHN0YXRlLnJlcXVlc3Rfc2luayhvdXRlcixyYXdfbG9naXRzLGxhYmVscykgaWYgb3V0ZXIgaXMgbm90IE5vbmUgZWxzZSBOb25lCgoKZGVmIHN0cmVhbWluZ19zaW5rKGlubmVyX293bmVyLGxhYmVscywqLHZvY2FidWxhcnkpOgogICAgdHJhbnNhY3Rpb249c3RhdGUuY3VycmVudCgpCiAgICBpZiB0cmFuc2FjdGlvbiBpcyBOb25lIG9yIGdldGF0dHIodHJhbnNhY3Rpb24sJ3Byb2R1Y2VyX293bmVyX3JlZicsbGFtYmRhOk5vbmUpKCkgaXMgbm90IGlubmVyX293bmVyOgogICAgICAgIHJldHVybiBOb25lCiAgICBvdXRlcj10cmFuc2FjdGlvbi5vd25lcl9yZWYoKQogICAgcmV0dXJuIHN0YXRlLnJlcXVlc3Rfc3RyZWFtaW5nX3Npbmsob3V0ZXIsbGFiZWxzLHZvY2FidWxhcnk9dm9jYWJ1bGFyeSkgaWYgb3V0ZXIgaXMgbm90IE5vbmUgZWxzZSBOb25lCgoKZGVmIG5hdGl2ZV9vcmRlcl9yb3dfbG9zc2VzKHRva2VuX25sbCxsYWJlbHMpOgogICAgIiIiTWF0Y2ggbmF0aXZlIHJvd19sb3NzZXMnIHBlci1leGFtcGxlIFQtMSBGUDMyIG1hc2svc3VtL2NsYW1wIG9yZGVyLiIiIgogICAgbG9zc2VzPVtdO2NvdW50cz1bXQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgdGFyZ2V0cz1sYWJlbHNbOiwxOl0KICAgICAgICBmb3IgaSBpbiByYW5nZSh0YXJnZXRzLnNpemUoMCkpOgogICAgICAgICAgICB0b2tlbl9jZT10b2tlbl9ubGxbaSw6LTFdLmZsb2F0KCkKICAgICAgICAgICAgbWFzaz0odGFyZ2V0c1tpXSE9LTEwMCkudG8odG9rZW5fY2UuZHR5cGUpCiAgICAgICAgICAgIHRva2Vucz1tYXNrLnN1bSgpO3RvdGFsPSh0b2tlbl9jZSptYXNrKS5zdW0oKQogICAgICAgICAgICBsb3NzZXMuYXBwZW5kKGZsb2F0KHRvdGFsL3Rva2Vucy5jbGFtcChtaW49MS4wKSkpCiAgICAgICAgICAgIGNvdW50cy5hcHBlbmQoaW50KHRva2Vucy5pdGVtKCkpKQogICAgcmV0dXJuIGxvc3Nlcyxjb3VudHMKCgpkZWYgcHVibGlzaF9tZXRyaWNzKHRyYWluZXIsb3V0cHV0cyxsYWJlbHMsYXR0ZW50aW9uX21hc2ssZW50cm9weV9mdW5jdGlvbix0cmFuc2FjdGlvbik6CiAgICBsb2dpdHM9b3V0cHV0cy5sb2dpdHMKICAgIGVudHJvcHk9dHJhbnNhY3Rpb24udGFrZSgndG9rZW5fZW50cm9weScsbG9naXRzLGxhYmVscykKICAgIHByZWRpY3Rpb25zPXRyYW5zYWN0aW9uLnRha2UoJ2ZpcnN0X2FyZ21heCcsbG9naXRzLGxhYmVscykKICAgIGlmIGVudHJvcHkgaXMgTm9uZSBvciBwcmVkaWN0aW9ucyBpcyBOb25lOgogICAgICAgIGlmIGxvZ2l0cyBpcyBOb25lOnJhaXNlIFJ1bnRpbWVFcnJvcignU3RyZWFtaW5nIHN1bW1hcnkgYWJzZW50IG9yIGludmFsaWQgYmVmb3JlIG1ldHJpY3MnKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBlbnRyb3B5PWVudHJvcHlfZnVuY3Rpb24obG9naXRzKQogICAgICAgICAgICBwcmVkaWN0aW9ucz1sb2dpdHNbLi4uLCA6LTEsOl0uY29udGlndW91cygpLmFyZ21heChkaW09LTEpCiAgICBlbHNlOnByZWRpY3Rpb25zPXByZWRpY3Rpb25zWy4uLiw6LTFdCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICB2YWx1ZT0oZW50cm9weSphdHRlbnRpb25fbWFzaykuc3VtKCkvYXR0ZW50aW9uX21hc2suc3VtKCkKICAgICAgICB2YWx1ZT10cmFpbmVyLmFjY2VsZXJhdG9yLmdhdGhlcl9mb3JfbWV0cmljcyh2YWx1ZSkubWVhbigpLml0ZW0oKQogICAgdHJhaW5lci5fbWV0cmljc1sndHJhaW4nXVsnZW50cm9weSddLmFwcGVuZCh2YWx1ZSkKICAgIHRva2Vucz10cmFpbmVyLmFjY2VsZXJhdG9yLmdhdGhlcl9mb3JfbWV0cmljcyhhdHRlbnRpb25fbWFzay5zdW0oKSkuc3VtKCkuaXRlbSgpCiAgICB0cmFpbmVyLl90b3RhbF90cmFpbl90b2tlbnMrPXRva2VucwogICAgdHJhaW5lci5fbWV0cmljc1sndHJhaW4nXVsnbnVtX3Rva2VucyddPVt0cmFpbmVyLl90b3RhbF90cmFpbl90b2tlbnNdCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICB0YXJnZXRzPWxhYmVsc1suLi4sMTpdLmNvbnRpZ3VvdXMoKTttYXNrPXRhcmdldHMhPS0xMDAKICAgICAgICBjb3JyZWN0PSgocHJlZGljdGlvbnM9PXRhcmdldHMpJm1hc2spLnN1bSgpO3RvdGFsPW1hc2suc3VtKCkKICAgICAgICBjb3JyZWN0PXRyYWluZXIuYWNjZWxlcmF0b3IuZ2F0aGVyX2Zvcl9tZXRyaWNzKGNvcnJlY3QpCiAgICAgICAgdG90YWw9dHJhaW5lci5hY2NlbGVyYXRvci5nYXRoZXJfZm9yX21ldHJpY3ModG90YWwpCiAgICAgICAgdG90YWxfc3VtPXRvdGFsLnN1bSgpCiAgICAgICAgYWNjdXJhY3k9KGNvcnJlY3Quc3VtKCkvdG90YWxfc3VtKS5pdGVtKCkgaWYgdG90YWxfc3VtPjAgZWxzZSAwLjAKICAgIHRyYWluZXIuX21ldHJpY3NbJ3RyYWluJ11bJ21lYW5fdG9rZW5fYWNjdXJhY3knXS5hcHBlbmQoYWNjdXJhY3kpCgoKY2xhc3MgQ29uc3VtZXJJbnN0YWxsYXRpb246CiAgICAiIiJFeHBsaWNpdCBmaXh0dXJlL2ludGVncmF0aW9uIEFQSS4gTk9UIGEgc291cmNlLXZlcmlmeWluZyBpbnN0YWxsZXIuCgogICAgZ3VhcmQodHJhaW5lcixtb2RlbCxpbnB1dHMscmV0dXJuX291dHB1dHMpIHJldHVybnMgTm9uZSBvbmx5IHdoZW4gZXZlcnkKICAgIHByb2R1Y3Rpb24gb3duZXJzaGlwL2NvbmZpZ3VyYXRpb24gcmVxdWlyZW1lbnQgaGFzIGJlZW4gZXN0YWJsaXNoZWQuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCoscmV3aW5kX2NsYXNzLHNmdF9jbGFzcyxoZl9jbGFzcyxlbnRyb3B5X2Z1bmN0aW9uLAogICAgICAgICAgICAgICAgIHJld2luZF9yb3dzLHJld2luZF90b2tlbl9yb3dzLGd1YXJkLHZvY2FidWxhcnksY2h1bmtlZF9hdHRyLHByb2R1Y2VyX293bmVyPWxhbWJkYSBtb2RlbDptb2RlbCk6CiAgICAgICAgc2VsZi5yZXdpbmRfY2xhc3M9cmV3aW5kX2NsYXNzO3NlbGYuc2Z0X2NsYXNzPXNmdF9jbGFzcztzZWxmLmhmX2NsYXNzPWhmX2NsYXNzCiAgICAgICAgc2VsZi5vcmlnaW5hbF9yZXdpbmQ9cmV3aW5kX2NsYXNzLmNvbXB1dGVfbG9zcwogICAgICAgIHNlbGYub3JpZ2luYWxfc2Z0PXNmdF9jbGFzcy5jb21wdXRlX2xvc3M7c2VsZi5vcmlnaW5hbF9oZj1oZl9jbGFzcy5jb21wdXRlX2xvc3MKICAgICAgICBzZWxmLm9yaWdpbmFsX3Jvd3M9cmV3aW5kX3Jvd3M7c2VsZi5vcmlnaW5hbF90b2tlbl9yb3dzPXJld2luZF90b2tlbl9yb3dzCiAgICAgICAgc2VsZi5nbG9iYWxzPXNlbGYub3JpZ2luYWxfcmV3aW5kLl9fZ2xvYmFsc19fO3NlbGYuY2h1bmtlZF9hdHRyPWNodW5rZWRfYXR0cgogICAgICAgIGlmIChzZWxmLmdsb2JhbHMuZ2V0KCdyb3dfbG9zc2VzJykgaXMgbm90IHJld2luZF9yb3dzCiAgICAgICAgICAgICAgICBvciBzZWxmLmdsb2JhbHMuZ2V0KCdyb3dfbG9zc2VzX2Zyb21fdG9rZW5fbG9zc2VzJykgaXMgbm90IHJld2luZF90b2tlbl9yb3dzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignTmF0aXZlIFJld2luZCBoZWxwZXIgb3duZXJzIHJlcXVpcmVkJykKICAgICAgICBvcmlnaW5hbF9yZXdpbmQ9c2VsZi5vcmlnaW5hbF9yZXdpbmQ7b3JpZ2luYWxfc2Z0PXNlbGYub3JpZ2luYWxfc2Z0CiAgICAgICAgb3JpZ2luYWxfaGY9c2VsZi5vcmlnaW5hbF9oZjtzZW50aW5lbD1vYmplY3QoKQoKICAgICAgICBAd3JhcHMob3JpZ2luYWxfcmV3aW5kKQogICAgICAgIGRlZiBvdXRlcih0cmFpbmVyLG1vZGVsLGlucHV0cyxyZXR1cm5fb3V0cHV0cz1GYWxzZSxudW1faXRlbXNfaW5fYmF0Y2g9Tm9uZSk6CiAgICAgICAgICAgIGludGFjdD0ocmV3aW5kX2NsYXNzLmNvbXB1dGVfbG9zcyBpcyBvdXRlciBhbmQgc2Z0X2NsYXNzLmNvbXB1dGVfbG9zcyBpcyBzZnQKICAgICAgICAgICAgICAgIGFuZCBoZl9jbGFzcy5jb21wdXRlX2xvc3MgaXMgb3JpZ2luYWxfaGYgYW5kIHNlbGYuZ2xvYmFscy5nZXQoJ3Jvd19sb3NzZXMnKSBpcyByb3dzCiAgICAgICAgICAgICAgICBhbmQgc2VsZi5nbG9iYWxzLmdldCgncm93X2xvc3Nlc19mcm9tX3Rva2VuX2xvc3NlcycpIGlzIHRva2VuX3Jvd3MpCiAgICAgICAgICAgIHJlYXNvbj1ndWFyZCh0cmFpbmVyLG1vZGVsLGlucHV0cyxyZXR1cm5fb3V0cHV0cykgaWYgaW50YWN0IGVsc2UgJ2NvbnN1bWVyX293bmVyX2NoYW5nZWQnCiAgICAgICAgICAgIGlmIHJlYXNvbiBpcyBub3QgTm9uZSBvciByZXR1cm5fb3V0cHV0cyBvciBzdGF0ZS5jdXJyZW50KCkgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB3aXRoIHN0YXRlLnN1c3BlbmRlZCgpOgogICAgICAgICAgICAgICAgICAgIHJldHVybiBvcmlnaW5hbF9yZXdpbmQodHJhaW5lcixtb2RlbCxpbnB1dHMscmV0dXJuX291dHB1dHM9cmV0dXJuX291dHB1dHMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBudW1faXRlbXNfaW5fYmF0Y2g9bnVtX2l0ZW1zX2luX2JhdGNoKQogICAgICAgICAgICBwcmV2aW91cz1nZXRhdHRyKHRyYWluZXIsY2h1bmtlZF9hdHRyLHNlbnRpbmVsKTtvd25lZF9wYWNrZXQ9Tm9uZQogICAgICAgICAgICB3aXRoIHN0YXRlLnNjb3BlKG1vZGVsLGlucHV0c1snbGFiZWxzJ10sdm9jYWJ1bGFyeT12b2NhYnVsYXJ5KSBhcyB0cmFuc2FjdGlvbjoKICAgICAgICAgICAgICAgIHRyYW5zYWN0aW9uLnByb2R1Y2VyX293bmVyX3JlZj13ZWFrcmVmLnJlZihwcm9kdWNlcl9vd25lcihtb2RlbCkpCiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0PW9yaWdpbmFsX3Jld2luZCh0cmFpbmVyLG1vZGVsLGlucHV0cyxyZXR1cm5fb3V0cHV0cz1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51bV9pdGVtc19pbl9iYXRjaD1udW1faXRlbXNfaW5fYmF0Y2gpCiAgICAgICAgICAgICAgICAgICAgaWYgdHJhbnNhY3Rpb24uc291cmNlX2tpbmQ9PSdzdHJlYW1pbmcnIGFuZCAoCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB0cmFuc2FjdGlvbi5pbnZhbGlkX3JlYXNvbiBpcyBub3QgTm9uZQogICAgICAgICAgICAgICAgICAgICAgICAgICAgb3IgdHJhbnNhY3Rpb24uY29uc3VtZWQhPXNldChzdGF0ZS5fRklFTERTKSk6CiAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignU3RyZWFtaW5nIHN1bW1hcmllcyB3ZXJlIG5vdCBjb21wbGV0ZWx5IGNvbnN1bWVkJykKICAgICAgICAgICAgICAgICAgICByZXR1cm4gcmVzdWx0CiAgICAgICAgICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICAgICAgICAgIG93bmVkX3BhY2tldD1nZXRhdHRyKHRyYW5zYWN0aW9uLCdjaHVua2VkX3BhY2tldCcsTm9uZSkKICAgICAgICAgICAgICAgICAgICBpZiBvd25lZF9wYWNrZXQgaXMgbm90IE5vbmUgYW5kIGdldGF0dHIodHJhaW5lcixjaHVua2VkX2F0dHIsc2VudGluZWwpIGlzIG93bmVkX3BhY2tldDoKICAgICAgICAgICAgICAgICAgICAgICAgaWYgcHJldmlvdXMgaXMgc2VudGluZWw6ZGVsYXR0cih0cmFpbmVyLGNodW5rZWRfYXR0cikKICAgICAgICAgICAgICAgICAgICAgICAgZWxzZTpzZXRhdHRyKHRyYWluZXIsY2h1bmtlZF9hdHRyLHByZXZpb3VzKQoKICAgICAgICBAd3JhcHMob3JpZ2luYWxfc2Z0KQogICAgICAgIGRlZiBzZnQodHJhaW5lcixtb2RlbCxpbnB1dHMscmV0dXJuX291dHB1dHM9RmFsc2UsbnVtX2l0ZW1zX2luX2JhdGNoPU5vbmUpOgogICAgICAgICAgICB0cmFuc2FjdGlvbj1zdGF0ZS5jdXJyZW50KCkKICAgICAgICAgICAgaWYgKHRyYW5zYWN0aW9uIGlzIE5vbmUgb3IgdHJhbnNhY3Rpb24ub3duZXJfcmVmKCkgaXMgbm90IG1vZGVsCiAgICAgICAgICAgICAgICAgICAgb3IgdHJhbnNhY3Rpb24ubGFiZWxzX3JlZigpIGlzIG5vdCBpbnB1dHMuZ2V0KCdsYWJlbHMnKSk6CiAgICAgICAgICAgICAgICByZXR1cm4gb3JpZ2luYWxfc2Z0KHRyYWluZXIsbW9kZWwsaW5wdXRzLHJldHVybl9vdXRwdXRzPXJldHVybl9vdXRwdXRzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBudW1faXRlbXNfaW5fYmF0Y2g9bnVtX2l0ZW1zX2luX2JhdGNoKQogICAgICAgICAgICBwYXJlbnQ9Z2V0YXR0cihzdXBlcihzZnRfY2xhc3MsdHJhaW5lciksJ2NvbXB1dGVfbG9zcycsTm9uZSkKICAgICAgICAgICAgaWYgZ2V0YXR0cihwYXJlbnQsJ19fZnVuY19fJyxOb25lKSBpcyBub3Qgb3JpZ2luYWxfaGY6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ05hdGl2ZSBIRiBwYXJlbnQgY2hhbmdlZCBpbnNpZGUgb3duZWQgdHJhbnNhY3Rpb24nKQogICAgICAgICAgICBsYWJlbHM9aW5wdXRzWydsYWJlbHMnXTttYXNrPWlucHV0c1snYXR0ZW50aW9uX21hc2snXQogICAgICAgICAgICBpbnB1dHMucG9wKCdfcHJlZGljdGlvbl9sb3NzX29ubHknLE5vbmUpO2lucHV0c1sndXNlX2NhY2hlJ109RmFsc2UKICAgICAgICAgICAgbG9zcyxvdXRwdXRzPXBhcmVudChtb2RlbCxpbnB1dHMscmV0dXJuX291dHB1dHM9VHJ1ZSxudW1faXRlbXNfaW5fYmF0Y2g9bnVtX2l0ZW1zX2luX2JhdGNoKQogICAgICAgICAgICBib3VuZD10cmFuc2FjdGlvbi5iaW5kX291dHB1dHMobG9zcyxvdXRwdXRzKQogICAgICAgICAgICBpZiBvdXRwdXRzLmxvZ2l0cyBpcyBOb25lOgogICAgICAgICAgICAgICAgaWYgbm90IGJvdW5kIG9yIHRyYW5zYWN0aW9uLnNvdXJjZV9raW5kIT0nc3RyZWFtaW5nJzoKICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ01pc3NpbmcgZnVsbCBsb2dpdHMgd2l0aG91dCB2YWxpZGF0ZWQgc3RyZWFtaW5nIHN1bW1hcmllcycpCiAgICAgICAgICAgICAgICB0cmFuc2FjdGlvbi5jaHVua2VkX3BhY2tldD1TaW1wbGVOYW1lc3BhY2UodG9rZW5fbG9zcz10cmFuc2FjdGlvbi5zdW1tYXJ5LnRva2VuX25sbCkKICAgICAgICAgICAgICAgIHNldGF0dHIodHJhaW5lcixjaHVua2VkX2F0dHIsdHJhbnNhY3Rpb24uY2h1bmtlZF9wYWNrZXQpCiAgICAgICAgICAgIHB1Ymxpc2hfbWV0cmljcyh0cmFpbmVyLG91dHB1dHMsbGFiZWxzLG1hc2ssZW50cm9weV9mdW5jdGlvbix0cmFuc2FjdGlvbikKICAgICAgICAgICAgcmV0dXJuIChsb3NzLG91dHB1dHMpIGlmIHJldHVybl9vdXRwdXRzIGVsc2UgbG9zcwoKICAgICAgICBkZWYgcm93cyhsb2dpdHMsbGFiZWxzKToKICAgICAgICAgICAgdHJhbnNhY3Rpb249c3RhdGUuY3VycmVudCgpCiAgICAgICAgICAgIHRva2VuX25sbD10cmFuc2FjdGlvbi50YWtlKCd0b2tlbl9ubGwnLGxvZ2l0cyxsYWJlbHMpIGlmIHRyYW5zYWN0aW9uIGVsc2UgTm9uZQogICAgICAgICAgICByZXR1cm4gbmF0aXZlX29yZGVyX3Jvd19sb3NzZXModG9rZW5fbmxsLGxhYmVscykgaWYgdG9rZW5fbmxsIGlzIG5vdCBOb25lIGVsc2UgcmV3aW5kX3Jvd3MobG9naXRzLGxhYmVscykKCiAgICAgICAgZGVmIHRva2VuX3Jvd3ModG9rZW5fbG9zcyxsYWJlbHMpOgogICAgICAgICAgICB0cmFuc2FjdGlvbj1zdGF0ZS5jdXJyZW50KCkKICAgICAgICAgICAgaWYgdHJhbnNhY3Rpb24gaXMgTm9uZTpyZXR1cm4gcmV3aW5kX3Rva2VuX3Jvd3ModG9rZW5fbG9zcyxsYWJlbHMpCiAgICAgICAgICAgIGlmICh0cmFuc2FjdGlvbi5zb3VyY2Vfa2luZCE9J3N0cmVhbWluZycgb3IgdHJhbnNhY3Rpb24uc3VtbWFyeSBpcyBOb25lCiAgICAgICAgICAgICAgICAgICAgb3IgdG9rZW5fbG9zcyBpcyBub3QgdHJhbnNhY3Rpb24uc3VtbWFyeS50b2tlbl9ubGwpOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdGb3JlaWduIHN0cmVhbWluZyB0b2tlbi1sb3NzIHBhY2tldCcpCiAgICAgICAgICAgIHRva2VuX25sbD10cmFuc2FjdGlvbi50YWtlKCd0b2tlbl9ubGwnLE5vbmUsbGFiZWxzKQogICAgICAgICAgICBpZiB0b2tlbl9ubGwgaXMgTm9uZTpyYWlzZSBSdW50aW1lRXJyb3IoJ0ludmFsaWQgc3RyZWFtaW5nIFJld2luZCBzdW1tYXJ5JykKICAgICAgICAgICAgcmV0dXJuIG5hdGl2ZV9vcmRlcl9yb3dfbG9zc2VzKHRva2VuX25sbCxsYWJlbHMpCgogICAgICAgIHNlbGYub3V0ZXI9b3V0ZXI7c2VsZi5zZnQ9c2Z0O3NlbGYucm93cz1yb3dzO3NlbGYudG9rZW5fcm93cz10b2tlbl9yb3dzCiAgICAgICAgcmV3aW5kX2NsYXNzLmNvbXB1dGVfbG9zcz1vdXRlcjtzZnRfY2xhc3MuY29tcHV0ZV9sb3NzPXNmdAogICAgICAgIHNlbGYuZ2xvYmFsc1sncm93X2xvc3NlcyddPXJvd3M7c2VsZi5nbG9iYWxzWydyb3dfbG9zc2VzX2Zyb21fdG9rZW5fbG9zc2VzJ109dG9rZW5fcm93cwoKICAgIGRlZiByZXN0b3JlKHNlbGYpOgogICAgICAgIHJlc3VsdD1UcnVlCiAgICAgICAgZm9yIG93bmVyLGtleSxvdXJzLG9sZCBpbiBbKHNlbGYucmV3aW5kX2NsYXNzLCdjb21wdXRlX2xvc3MnLHNlbGYub3V0ZXIsc2VsZi5vcmlnaW5hbF9yZXdpbmQpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIChzZWxmLnNmdF9jbGFzcywnY29tcHV0ZV9sb3NzJyxzZWxmLnNmdCxzZWxmLm9yaWdpbmFsX3NmdCldOgogICAgICAgICAgICBpZiBnZXRhdHRyKG93bmVyLGtleSkgaXMgb3VyczpzZXRhdHRyKG93bmVyLGtleSxvbGQpCiAgICAgICAgICAgIGVsc2U6cmVzdWx0PUZhbHNlCiAgICAgICAgZm9yIGtleSxvdXJzLG9sZCBpbiBbKCdyb3dfbG9zc2VzJyxzZWxmLnJvd3Msc2VsZi5vcmlnaW5hbF9yb3dzKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAoJ3Jvd19sb3NzZXNfZnJvbV90b2tlbl9sb3NzZXMnLHNlbGYudG9rZW5fcm93cyxzZWxmLm9yaWdpbmFsX3Rva2VuX3Jvd3MpXToKICAgICAgICAgICAgaWYgc2VsZi5nbG9iYWxzLmdldChrZXkpIGlzIG91cnM6c2VsZi5nbG9iYWxzW2tleV09b2xkCiAgICAgICAgICAgIGVsc2U6cmVzdWx0PUZhbHNlCiAgICAgICAgcmV0dXJuIHJlc3VsdAo=', 'old_solution/__init__.py': 'IiIiU3RhbmRhbG9uZSBoZWFkIGNvbXBvbmVudCBvbmx5LiBObyBtb2RlbC9UcmFpbmVyIGluc3RhbGxhdGlvbiBvciBrZXJuZWwgY3JlZGl0LiIiIgo=', 'old_solution/token_chunk_head.py': 'IiIiT3JpZ2luYWwgbmF0aXZlLUdFTU0gdG9rZW4tY2h1bmsgY29tcG9uZW50OyBubyBtb2RlbCBwYXRjaCBvciBUcml0b24gY3JlZGl0LgoKVG9rZW4tYXhpcyBjaHVua3Mga2VlcCBlYWNoIGhpZGRlbi1ncmFkaWVudCByb3cncyBlbnRpcmUgdm9jYWJ1bGFyeSByZWR1Y3Rpb24uCkJhY2t3YXJkIHJlY29tcHV0ZXMgdGhlIGhlYWQgQUZURVIgdGhlIHJlYWwgdXBzdHJlYW0vR3JhZFNjYWxlciBzY2FsYXIgYXJyaXZlcy4KTm8gU291cCB2ZW5kb3IgY2h1bmsgaW1wbGVtZW50YXRpb24gaXMgaW1wb3J0ZWQgb3IgY29waWVkLiBOYXRpdmUgUHlUb3JjaCBvcHMKYXJlIHJlZmVyZW5jZSBwcmltaXRpdmVzIGFuZCBlYXJuIG5vIG9yaWdpbmFsLVRyaXRvbiBjcmVkaXQuIFRhcmdldCBzbGljZWQvZnVsbApHRU1NIGVxdWl2YWxlbmNlIGlzIHRlc3RlZCwgbm90IGFzc3VtZWQuIE5vIG1vZGVsL29wdGltaXplci90cmFpbmluZyBpcyBpbnN0YWxsZWQuCiIiIgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCkBkYXRhY2xhc3MKY2xhc3MgU3VtbWFyaWVzOgogICAgZW50cm9weTogdG9yY2guVGVuc29yCiAgICB0b2tlbl9ubGw6IHRvcmNoLlRlbnNvcgogICAgcHJlZGljdGlvbnM6IHRvcmNoLlRlbnNvcgoKCmRlZiBfY2hlY2soaCx3LGxhYmVscyxjaHVuayxkZW5vbWluYXRvcik6CiAgICBhc3NlcnQgdHlwZShoKSBpcyB0b3JjaC5UZW5zb3IgYW5kIHR5cGUodykgaW4gKHRvcmNoLlRlbnNvcix0b3JjaC5ubi5QYXJhbWV0ZXIpIGFuZCB0eXBlKGxhYmVscykgaXMgdG9yY2guVGVuc29yCiAgICBhc3NlcnQgaC5kZXZpY2U9PXcuZGV2aWNlPT1sYWJlbHMuZGV2aWNlIGFuZCBoLmRldmljZS50eXBlIGluICgnY3B1JywnY3VkYScpCiAgICBpZiBoLmRldmljZS50eXBlPT0nY3VkYSc6CiAgICAgICAgYXNzZXJ0IGguc2hhcGVbLTFdPT0xNTM2IGFuZCB3LnNoYXBlPT0oMTUxOTM2LDE1MzYpLCAnVGFzay1zcGVjaWZpYyBDVURBIGdlb21ldHJ5JwogICAgICAgIGFzc2VydCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfY2FwYWJpbGl0eShoLmRldmljZSk9PSg3LDUpLCAnVDQgY29tcG9uZW50IG9ubHknCiAgICAgICAgYXNzZXJ0IHRvcmNoLmlzX2F1dG9jYXN0X2VuYWJsZWQoJ2N1ZGEnKSBhbmQgdG9yY2guZ2V0X2F1dG9jYXN0X2R0eXBlKCdjdWRhJyk9PXRvcmNoLmZsb2F0MTYsICdOYXRpdmUgQU1QIEZQMTYgYm91bmRhcnkgb25seScKICAgIGFzc2VydCBoLmR0eXBlPT13LmR0eXBlPT10b3JjaC5mbG9hdDMyIGFuZCBsYWJlbHMuZHR5cGU9PXRvcmNoLmludDY0CiAgICBhc3NlcnQgaC5uZGltPT0zIGFuZCB3Lm5kaW09PTIgYW5kIGxhYmVscy5zaGFwZT09aC5zaGFwZVs6Ml0gYW5kIGguc2hhcGVbLTFdPT13LnNoYXBlWzFdCiAgICBhc3NlcnQgaC5pc19jb250aWd1b3VzKCkgYW5kIHcuaXNfY29udGlndW91cygpIGFuZCBsYWJlbHMuaXNfY29udGlndW91cygpCiAgICBhc3NlcnQgbm90IGFueSh4LmlzX25lZygpIG9yIHguaXNfY29uaigpIGZvciB4IGluIChoLHcsbGFiZWxzKSkKICAgIGFzc2VydCBub3Qgdy5yZXF1aXJlc19ncmFkLCdGcm96ZW4gaGVhZCBvbmx5OiBubyB2b2NhYnVsYXJ5L3dlaWdodC1ncmFkaWVudCBhY2N1bXVsYXRpb24gcHJvb2YnCiAgICBhc3NlcnQgaC51bnR5cGVkX3N0b3JhZ2UoKS5kYXRhX3B0cigpIT13LnVudHlwZWRfc3RvcmFnZSgpLmRhdGFfcHRyKCkKICAgIGFzc2VydCB0eXBlKGNodW5rKSBpcyBpbnQgYW5kIGNodW5rPjAgYW5kIGgubnVtZWwoKT4wIGFuZCB3LnNoYXBlWzBdPjAKICAgIGFzc2VydCBkZW5vbWluYXRvciBpcyBOb25lIG9yICh0eXBlKGRlbm9taW5hdG9yKSBpcyBpbnQgYW5kIGRlbm9taW5hdG9yPj0wKSBvciAodHlwZShkZW5vbWluYXRvcikgaXMgdG9yY2guVGVuc29yIGFuZCBkZW5vbWluYXRvci5kZXZpY2U9PWguZGV2aWNlIGFuZCBkZW5vbWluYXRvci5kdHlwZT09dG9yY2guaW50NjQgYW5kIGRlbm9taW5hdG9yLm5kaW09PTAgYW5kIG5vdCBkZW5vbWluYXRvci5yZXF1aXJlc19ncmFkKQogICAgIyBOYXRpdmUgcHJpbWl0aXZlcyByZXRhaW4gdGhlaXIgZXhjZXB0aW9uYWwtdmFsdWUgYmVoYXZpb3I7IG5vIGRhdGEtZGVwZW5kZW50CiAgICAjIHN1Y2Nlc3Mgcm91dGUsIHBhZGRpbmcgcmVtb3ZhbCwgZ2xvYmFsIEFNUC9wcmVjaXNpb24tc2V0dGluZyBjaGFuZ2UsIG9yIGNhY2hlLgoKCmRlZiBzaGlmdF90YXJnZXRzKGxhYmVscyk6CiAgICByZXR1cm4gRi5wYWQobGFiZWxzLCgwLDEpLHZhbHVlPS0xMDApWy4uLiwxOl0uY29udGlndW91cygpCgoKZGVmIG5hdGl2ZV9ib3VuZGFyeShoLHcsbGFiZWxzLGRlbm9taW5hdG9yPU5vbmUpOgogICAgIyBFeHBsaWNpdCBDUFUgZW11bGF0aW9uIG9mIENVREEgQU1QJ3MgRlAzMiAtPiBGUDE2IGxpbmVhciBib3VuZGFyeS4KICAgIHJhdz1GLmxpbmVhcihoLnRvKHRvcmNoLmZsb2F0MTYpLHcudG8odG9yY2guZmxvYXQxNikpCiAgICB0YXJnZXRzPXNoaWZ0X3RhcmdldHMobGFiZWxzKQogICAgbG9zcz1GLmNyb3NzX2VudHJvcHkocmF3LmZsb2F0KCkudmlldygtMSx3LnNoYXBlWzBdKSx0YXJnZXRzLnZpZXcoLTEpLGlnbm9yZV9pbmRleD0tMTAwLAogICAgICAgICAgICAgICAgICAgICAgICAgcmVkdWN0aW9uPSdtZWFuJyBpZiBkZW5vbWluYXRvciBpcyBOb25lIGVsc2UgJ3N1bScpCiAgICBpZiBkZW5vbWluYXRvciBpcyBub3QgTm9uZTpsb3NzPWxvc3MvZGVub21pbmF0b3IKICAgIHB1YmxpYz1yYXcuZmxvYXQoKQogICAgbG9ncD1GLmxvZ19zb2Z0bWF4KHB1YmxpYyxkaW09LTEpCiAgICBubGw9Ri5jcm9zc19lbnRyb3B5KHB1YmxpYy5yZXNoYXBlKC0xLHcuc2hhcGVbMF0pLHRhcmdldHMucmVzaGFwZSgtMSksaWdub3JlX2luZGV4PS0xMDAscmVkdWN0aW9uPSdub25lJykucmVzaGFwZV9hcyhsYWJlbHMpCiAgICByZXR1cm4gbG9zcyxTdW1tYXJpZXMoLShsb2dwLmV4cCgpKmxvZ3ApLnN1bSgtMSksbmxsLHB1YmxpYy5hcmdtYXgoLTEpKSxyYXcKCgpjbGFzcyBfVG9rZW5DaHVuayh0b3JjaC5hdXRvZ3JhZC5GdW5jdGlvbik6CiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgZm9yd2FyZChjdHgsaCx3LGxhYmVscyxjaHVuayxkZW5vbWluYXRvcik6CiAgICAgICAgX2NoZWNrKGgsdyxsYWJlbHMsY2h1bmssZGVub21pbmF0b3IpCiAgICAgICAgYix0LGs9aC5zaGFwZTt2PXcuc2hhcGVbMF0KICAgICAgICAjIEZQMzIgaW5wdXRzIGltcGx5IHRoZXNlIGNhc3RzIGNyZWF0ZSBvd25lZCBzbmFwc2hvdHMsIG5vdCBhbGlhc2VzLgogICAgICAgIGgxNj1oLnRvKHRvcmNoLmZsb2F0MTYpLnJlc2hhcGUoLTEsaykKICAgICAgICB3MTY9dy50byh0b3JjaC5mbG9hdDE2KQogICAgICAgIHRhcmdldHM9c2hpZnRfdGFyZ2V0cyhsYWJlbHMpLnJlc2hhcGUoLTEpCiAgICAgICAgZW50cm9weT10b3JjaC5lbXB0eShiKnQsZHR5cGU9dG9yY2guZmxvYXQzMixkZXZpY2U9aC5kZXZpY2UpCiAgICAgICAgbmxsPXRvcmNoLmVtcHR5KGIqdCxkdHlwZT10b3JjaC5mbG9hdDMyLGRldmljZT1oLmRldmljZSkKICAgICAgICBwcmVkaWN0aW9ucz10b3JjaC5lbXB0eShiKnQsZHR5cGU9dG9yY2guaW50NjQsZGV2aWNlPWguZGV2aWNlKQogICAgICAgIGZvciBzdGFydCBpbiByYW5nZSgwLGIqdCxjaHVuayk6CiAgICAgICAgICAgIHN0b3A9bWluKHN0YXJ0K2NodW5rLGIqdCkKICAgICAgICAgICAgbG9naXRzPUYubGluZWFyKGgxNltzdGFydDpzdG9wXSx3MTYpLmZsb2F0KCkKICAgICAgICAgICAgbG9ncD1GLmxvZ19zb2Z0bWF4KGxvZ2l0cyxkaW09LTEpCiAgICAgICAgICAgIGVudHJvcHlbc3RhcnQ6c3RvcF09LShsb2dwLmV4cCgpKmxvZ3ApLnN1bSgtMSkKICAgICAgICAgICAgbmxsW3N0YXJ0OnN0b3BdPUYuY3Jvc3NfZW50cm9weShsb2dpdHMsdGFyZ2V0c1tzdGFydDpzdG9wXSxpZ25vcmVfaW5kZXg9LTEwMCxyZWR1Y3Rpb249J25vbmUnKQogICAgICAgICAgICBwcmVkaWN0aW9uc1tzdGFydDpzdG9wXT1sb2dpdHMuYXJnbWF4KC0xKQogICAgICAgICAgICBkZWwgbG9naXRzLGxvZ3AKICAgICAgICBkaXZpc29yPSh0YXJnZXRzIT0tMTAwKS5zdW0oKSBpZiBkZW5vbWluYXRvciBpcyBOb25lIGVsc2UgZGVub21pbmF0b3IKICAgICAgICAjIFRoaXMgc21hbGwgZ2xvYmFsIHJlZHVjdGlvbiBjYW4gZGlmZmVyIGJ5IHJvdW5kaW5nIGZyb20gbmF0aXZlIE5MTC1zdW0uCiAgICAgICAgIyBJdCBpcyBOT1QgY2xhaW1lZCBiaXQtZXhhY3Q7IHRhc2sgdG9sZXJhbmNlcyBhbmQgdGFyZ2V0IGdhdGVzIHN0aWxsIGFwcGx5LgogICAgICAgIGxvc3M9bmxsLnN1bSgpL2Rpdmlzb3IKICAgICAgICAjIEEgY2FsbGVyLW93bmVkIGRlbm9taW5hdG9yIHRlbnNvciBoYXMgbmF0aXZlIHNhdmVkLXZlcnNpb24gc2VtYW50aWNzLgogICAgICAgICMgU2F2aW5nIGl0IChyYXRoZXIgdGhhbiBvbmx5IGFzc2lnbmluZyBjdHguZGl2aXNvcikgY2F0Y2hlcyBtdXRhdGlvbi4KICAgICAgICB0ZW5zb3JfZGl2aXNvcj1pc2luc3RhbmNlKGRpdmlzb3IsdG9yY2guVGVuc29yKQogICAgICAgIGN0eC5zYXZlX2Zvcl9iYWNrd2FyZChoMTYsdzE2LHRhcmdldHMsKigoZGl2aXNvciwpIGlmIHRlbnNvcl9kaXZpc29yIGVsc2UgKCkpKQogICAgICAgIGN0eC5zaGFwZT1oLnNoYXBlO2N0eC5jaHVuaz1jaHVuaztjdHguZGl2aXNvcj1Ob25lIGlmIHRlbnNvcl9kaXZpc29yIGVsc2UgZGl2aXNvcgogICAgICAgIGVudHJvcHk9ZW50cm9weS5yZXNoYXBlKGIsdCk7bmxsPW5sbC5yZXNoYXBlKGIsdCk7cHJlZGljdGlvbnM9cHJlZGljdGlvbnMucmVzaGFwZShiLHQpCiAgICAgICAgY3R4Lm1hcmtfbm9uX2RpZmZlcmVudGlhYmxlKGVudHJvcHksbmxsLHByZWRpY3Rpb25zKQogICAgICAgIHJldHVybiBsb3NzLGVudHJvcHksbmxsLHByZWRpY3Rpb25zCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGJhY2t3YXJkKGN0eCxncmFkX2xvc3MsX2VudHJvcHksX25sbCxfcHJlZGljdGlvbnMpOgogICAgICAgIGlmIHRvcmNoLmlzX2dyYWRfZW5hYmxlZCgpOnJhaXNlIFJ1bnRpbWVFcnJvcignSGlnaGVyLW9yZGVyIHJlcXVlc3RzIG5vdCBzdXBwb3J0ZWQgYnkgc3RhbmRhbG9uZSBjb21wb25lbnQnKQogICAgICAgIHNhdmVkPWN0eC5zYXZlZF90ZW5zb3JzCiAgICAgICAgaDE2LHcxNix0YXJnZXRzPXNhdmVkWzozXQogICAgICAgIGRpdmlzb3I9c2F2ZWRbM10gaWYgbGVuKHNhdmVkKT09NCBlbHNlIGN0eC5kaXZpc29yCiAgICAgICAgIyBNYWluIGxvc3MgZGl2aXNpb24ncyB1cHN0cmVhbSBncmFkaWVudCBoYXBwZW5zIEJFRk9SRSBuYXRpdmUgQ0UgLT4gaGFsZi4KICAgICAgICBub3JtYWxpemVkPWdyYWRfbG9zcy9kaXZpc29yCiAgICAgICAgcmVzdWx0PXRvcmNoLmVtcHR5X2xpa2UoaDE2LGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsaDE2LnNoYXBlWzBdLGN0eC5jaHVuayk6CiAgICAgICAgICAgIHN0b3A9bWluKHN0YXJ0K2N0eC5jaHVuayxoMTYuc2hhcGVbMF0pCiAgICAgICAgICAgIHdpdGggdG9yY2guZW5hYmxlX2dyYWQoKToKICAgICAgICAgICAgICAgIHJhdz1GLmxpbmVhcihoMTZbc3RhcnQ6c3RvcF0sdzE2KS5kZXRhY2goKS5yZXF1aXJlc19ncmFkXygpCiAgICAgICAgICAgICAgICBsb3NzX3N1bT1GLmNyb3NzX2VudHJvcHkocmF3LmZsb2F0KCksdGFyZ2V0c1tzdGFydDpzdG9wXSxpZ25vcmVfaW5kZXg9LTEwMCxyZWR1Y3Rpb249J3N1bScpCiAgICAgICAgICAgICAgICAoZHoxNiwpPXRvcmNoLmF1dG9ncmFkLmdyYWQobG9zc19zdW0scmF3LGdyYWRfb3V0cHV0cz1ub3JtYWxpemVkKQogICAgICAgICAgICAjIE9uZSBmdWxsLVYgZG90IHBlciBoaWRkZW4gcm93OyBvdXRwdXQgcm91bmRlZCB0byBGUDE2IEJFRk9SRSBmcDMyIGNhc3QuCiAgICAgICAgICAgIHJlc3VsdFtzdGFydDpzdG9wXT0oZHoxNiBAIHcxNikuZmxvYXQoKQogICAgICAgICAgICBkZWwgcmF3LGxvc3Nfc3VtLGR6MTYKICAgICAgICByZXR1cm4gcmVzdWx0LnJlc2hhcGUoY3R4LnNoYXBlKSxOb25lLE5vbmUsTm9uZSxOb25lCgoKZGVmIHRva2VuX2NodW5rX2JvdW5kYXJ5KGgsdyxsYWJlbHMsY2h1bms9MTI4LGRlbm9taW5hdG9yPU5vbmUsc3VtbWFyeV9zaW5rPU5vbmUpOgogICAgbG9zcyxlbnRyb3B5LG5sbCxwcmVkaWN0aW9ucz1fVG9rZW5DaHVuay5hcHBseShoLHcsbGFiZWxzLGNodW5rLGRlbm9taW5hdG9yKQogICAgaWYgc3VtbWFyeV9zaW5rIGlzIG5vdCBOb25lOgogICAgICAgIGlmIG5vdCBzdW1tYXJ5X3NpbmsodG9rZW5fbmxsPW5sbCx0b2tlbl9lbnRyb3B5PWVudHJvcHksZmlyc3RfYXJnbWF4PXByZWRpY3Rpb25zLHJvdXRlPSdzdHJlYW1pbmcnKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdTdHJlYW1pbmcgc3VtbWFyeSBwdWJsaWNhdGlvbiBmYWlsZWQnKQogICAgICAgIGlmIG5vdCBzdW1tYXJ5X3NpbmsuY29tcGxldGUobG9zcyk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignU3RyZWFtaW5nIG5vcm1hbGl6ZWQtbG9zcyBpZGVudGl0eSBwdWJsaWNhdGlvbiBmYWlsZWQnKQogICAgcmV0dXJuIGxvc3MsU3VtbWFyaWVzKGVudHJvcHksbmxsLHByZWRpY3Rpb25zKQoK', 'process_guard.py': 'IiIiSXNvbGF0ZWQgTGludXggc3VicmVhcGVyIGd1YXJkaWFuLiBPd25zIGNoaWxkcmVuIGFuZCBob2xkcyB0aGUgR1BVIGxvY2suCgpUaGUgZ3VhcmRpYW4gd2F0Y2hlcyBhIHBhcmVudC1saXZlbmVzcyBwaXBlOiBkcml2ZXIgU0lHS0lMTCBvciBvdGhlciBhYnJ1cHQKZGVhdGggY2xvc2VzIHRoZSBwaXBlIGFuZCB0cmlnZ2VycyBxdWFyYW50aW5lZCBjbGVhbnVwLiBQSURGRCBzaWduYWxpbmcgaW4gdGhlCnNoYXJlZCB0YXNrIGhlbHBlciBwcmV2ZW50cyBQSUQgcmV1c2UgcmFjZXM7IGRlc2NlbmRhbnRzIGFyZSBvbmx5IHRoaXMgZGVkaWNhdGVkCmd1YXJkaWFuJ3MgY2hpbGRyZW4sIGluY2x1ZGluZyBuZXdseSBhZG9wdGVkIG9ycGhhbnMgZnJvbSBuZXN0ZWQgc2Vzc2lvbnMuClRoaXMgaXMgdGFzayBvcmNoZXN0cmF0aW9uLCBuZXZlciBzb2x1dGlvbi8gd29yayBvciBrZXJuZWwgcGFydGljaXBhdGlvbi4KIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNlbGVjdAppbXBvcnQgc2lnbmFsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQoKCmRlZiBxdWFyYW50aW5lKHBhdGgsIHJlYXNvbik6CiAgICAiIiJGaXJzdCBmYWlsdXJlIHdpbnM7IG5ldmVyIG92ZXJ3cml0ZSBvciBhdXRvbWF0aWNhbGx5IGNsZWFyIGEgbWFya2VyLiIiIgogICAgdHJ5OgogICAgICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fV1JPTkxZIHwgb3MuT19DUkVBVCB8IG9zLk9fRVhDTCB8IG9zLk9fTk9GT0xMT1csIDBvNjAwKQogICAgZXhjZXB0IEZpbGVFeGlzdHNFcnJvcjoKICAgICAgICByZXR1cm4KICAgIHdpdGggb3MuZmRvcGVuKGRlc2NyaXB0b3IsICJ3IikgYXMgc3RyZWFtOgogICAgICAgIGpzb24uZHVtcCh7InN0YXR1cyI6ICJPUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCIsICJyZWFzb24iOiByZWFzb24sCiAgICAgICAgICAgICAgICAgICAiZ3VhcmRpYW5fcGlkIjogb3MuZ2V0cGlkKCksICJjcmVhdGVkX3VuaXgiOiB0aW1lLnRpbWUoKX0sIHN0cmVhbSkKICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKCgpkZWYgX3dyaXRlX3JlY2VpcHQocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBlbmRpbmcgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnBlbmRpbmciKQogICAgcGVuZGluZy53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yKSArICJcbiIpCiAgICBwZW5kaW5nLnJlcGxhY2UocGF0aCkKCgpkZWYgZ3VhcmRpYW5fbWFpbigpOgogICAgY29uZmlnID0ganNvbi5sb2FkcyhzeXMuc3RkaW4ucmVhZGxpbmUoKSkKICAgIHJlYWRfZmQsIGxvY2tfZmQgPSBjb25maWdbInBhcmVudF9waXBlX2ZkIl0sIGNvbmZpZ1sibG9ja19mZCJdCiAgICBzdGFydGVkLCByZXF1ZXN0ZWRfc2lnbmFsLCBwcm9jZXNzID0gdGltZS5tb25vdG9uaWMoKSwgW05vbmVdLCBOb25lCiAgICByZXN1bHQgPSB7ImV4aXRfY29kZSI6IDEyNSwgInJvb3RfZXhpdF9jb2RlIjogTm9uZSwgInRpbWVkX291dCI6IEZhbHNlLAogICAgICAgICAgICAgICJpbnRlcnJ1cHRlZCI6IEZhbHNlLCAiY2xlYW51cF9jb25maXJtZWQiOiBGYWxzZSwKICAgICAgICAgICAgICAicmVhc29uIjogTm9uZSwgImd1YXJkaWFuX3BpZCI6IG9zLmdldHBpZCgpfQogICAgZGVmIHJlcXVlc3Qoc2lnbnVtLCBmcmFtZSk6CiAgICAgICAgcmVxdWVzdGVkX3NpZ25hbFswXSA9IHNpZ251bQogICAgZm9yIHNpZyBpbiAoc2lnbmFsLlNJR0lOVCwgc2lnbmFsLlNJR1RFUk0pOgogICAgICAgIHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KQogICAgdHJ5OgogICAgICAgIHRyZWUucHJlcGFyZSgpICAjIHJlcXVpcmVkIGJlZm9yZSBhbnkgY2hpbGQ7IG5ldmVyIHNpbGVudGx5IGRlZ3JhZGUKICAgICAgICB3aXRoIFBhdGgoY29uZmlnWyJsb2ciXSkub3BlbigieCIpIGFzIHN0cmVhbToKICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29uZmlnWyJjb21tYW5kIl0sIGN3ZD1jb25maWdbImN3ZCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3RyZWFtLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICByZWFkeSwgXywgXyA9IHNlbGVjdC5zZWxlY3QoW3JlYWRfZmRdLCBbXSwgW10sIDAuMDUpCiAgICAgICAgICAgICAgICBpZiByZWFkeSBhbmQgb3MucmVhZChyZWFkX2ZkLCAxKSA9PSBiIiI6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249InBhcmVudF9saXZlbmVzc19waXBlX2Nsb3NlZCIsIGludGVycnVwdGVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHJlcXVlc3RlZF9zaWduYWxbMF0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX3NpZ25hbF8iICsgc3RyKHJlcXVlc3RlZF9zaWduYWxbMF0pLCBpbnRlcnJ1cHRlZD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCA+PSBjb25maWdbInRpbWVvdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHQudXBkYXRlKHJlYXNvbj0ib3V0ZXJfdGltZW91dCIsIHRpbWVkX291dD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICByZXR1cm5jb2RlID0gcHJvY2Vzcy5wb2xsKCkKICAgICAgICAgICAgICAgIGlmIHJldHVybmNvZGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyb290X2V4aXRfY29kZSJdID0gcmV0dXJuY29kZQogICAgICAgICAgICAgICAgICAgIGlmIHRyZWUuZGVzY2VuZGFudHMoKToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyZWFzb24iXSA9ICJyb290X2V4aXRlZF93aXRoX293bmVkX2Rlc2NlbmRhbnRzIgogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoZXhpdF9jb2RlPXJldHVybmNvZGUsIGNsZWFudXBfY29uZmlybWVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgcmVzdWx0WyJyZWFzb24iXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHF1YXJhbnRpbmUoY29uZmlnWyJtYXJrZXIiXSwgcmVzdWx0WyJyZWFzb24iXSkKICAgICAgICAgICAgICAgICMgY2xlYW51cCBpZ25vcmVzIHJlcGVhdGVkIElOVC9URVJNLCBzdG9wcy9yZXNjYW5zL2tpbGxzIGJ5CiAgICAgICAgICAgICAgICAjIHBpZGZkIGFuZCB3YWl0cy9yZWFwcyB3aXRob3V0IGFueSB0aW1lb3V0LXVubG9jayBlc2NhcGUuCiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUocm9vdF9leGl0X2NvZGU9cHJvY2Vzcy5yZXR1cm5jb2RlLCBjbGVhbnVwX2NvbmZpcm1lZD1UcnVlKQogICAgICAgICAgICAgICAgaWYgcmVzdWx0WyJ0aW1lZF9vdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTI0CiAgICAgICAgICAgICAgICBlbGlmIHJlc3VsdFsiaW50ZXJydXB0ZWQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTMwCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX2V4Y2VwdGlvbjoiICsgcmVwcihleGMpLCBleGl0X2NvZGU9MTI1KQogICAgICAgIHRyeToKICAgICAgICAgICAgcXVhcmFudGluZShjb25maWdbIm1hcmtlciJdLCByZXN1bHRbInJlYXNvbiJdKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHByb2Nlc3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgcmVzdWx0WyJjbGVhbnVwX2NvbmZpcm1lZCJdID0gbm90IHRyZWUuZGVzY2VuZGFudHMoKQogICAgZmluYWxseToKICAgICAgICByZXN1bHRbIm91dGVyX3dhbGxfcyJdID0gdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQKICAgICAgICBfd3JpdGVfcmVjZWlwdChjb25maWdbInJlY2VpcHQiXSwgcmVzdWx0KQogICAgICAgIG9zLmNsb3NlKHJlYWRfZmQpCiAgICAgICAgIyBUaGUgaW5oZXJpdGVkIG9wZW4tZmlsZSBkZXNjcmlwdGlvbiByZXRhaW5zIHRoZSBjYW5vbmljYWwgZmxvY2sgZXZlbgogICAgICAgICMgaWYgdGhlIGRyaXZlciBoYXMgZGllZC4gQ2xvc2UgT05MWSBhZnRlciBvd25lZC10cmVlIGNsZWFudXAgY29tcGxldGVzLgogICAgICAgIG9zLmNsb3NlKGxvY2tfZmQpCiAgICByZXR1cm4gcmVzdWx0WyJleGl0X2NvZGUiXQoKCmRlZiBydW4oY29tbWFuZCwgKiwgY3dkLCBlbnYsIGxvZywgdGltZW91dCwgbG9ja19mZCwKICAgICAgICBtYXJrZXI9Ii90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCIpOgogICAgIiIiRGVkaWNhdGVkIGRyaXZlciBBUEkuIEFueSBCYXNlRXhjZXB0aW9uIGFza3MgZ3VhcmRpYW4gY2xlYW51cCBhbmQgd2FpdHMuIiIiCiAgICByZWNlaXB0ID0gUGF0aChzdHIobG9nKSArICIucHJvY2Vzcy5qc29uIikucmVzb2x2ZSgpCiAgICBpZiByZWNlaXB0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUHJvY2VzcyByZWNlaXB0IGFscmVhZHkgZXhpc3RzOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgIHJlYWRfZmQsIHdyaXRlX2ZkID0gb3MucGlwZSgpCiAgICBjb25maWcgPSB7ImNvbW1hbmQiOiBbc3RyKHgpIGZvciB4IGluIGNvbW1hbmRdLCAiY3dkIjogc3RyKFBhdGgoY3dkKS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICJsb2ciOiBzdHIoUGF0aChsb2cpLnJlc29sdmUoKSksICJ0aW1lb3V0IjogdGltZW91dCwKICAgICAgICAgICAgICAicmVjZWlwdCI6IHN0cihyZWNlaXB0KSwgInBhcmVudF9waXBlX2ZkIjogcmVhZF9mZCwKICAgICAgICAgICAgICAibG9ja19mZCI6IGxvY2tfZmQsICJtYXJrZXIiOiBzdHIobWFya2VyKX0KICAgIGd1YXJkaWFuID0gTm9uZQogICAgaW50ZXJydXB0ZWQgPSBOb25lCiAgICBkZWYgcmVxdWVzdChzaWdudW0sIGZyYW1lKToKICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCJEcml2ZXIgcmVjZWl2ZWQgc2lnbmFsICIgKyBzdHIoc2lnbnVtKSkKICAgIG9sZF9oYW5kbGVycyA9IHtzaWc6IHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KSBmb3Igc2lnIGluIChzaWduYWwuU0lHSU5ULCBzaWduYWwuU0lHVEVSTSl9CiAgICB0cnk6CiAgICAgICAgZ3VhcmRpYW4gPSBzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKSksICItLWd1YXJkaWFuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuUElQRSwgZW52PWVudiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFzc19mZHM9KHJlYWRfZmQsIGxvY2tfZmQpLCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUpCiAgICAgICAgb3MuY2xvc2UocmVhZF9mZCkKICAgICAgICByZWFkX2ZkID0gTm9uZQogICAgICAgIGd1YXJkaWFuLnN0ZGluLndyaXRlKGpzb24uZHVtcHMoY29uZmlnKSArICJcbiIpCiAgICAgICAgZ3VhcmRpYW4uc3RkaW4uY2xvc2UoKQogICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGludGVycnVwdGVkID0gcmVwcihleGMpCiAgICAgICAgIyBJZ25vcmUgcmVwZWF0ZWQgc3RvcCBzaWduYWxzIHVudGlsIHRoZSBndWFyZGlhbiBjb25maXJtcyBjbGVhbnVwLgogICAgICAgIGZvciBzaWcgaW4gb2xkX2hhbmRsZXJzOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgc2lnbmFsLlNJR19JR04pCiAgICAgICAgaWYgd3JpdGVfZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG9zLmNsb3NlKHdyaXRlX2ZkKQogICAgICAgICAgICB3cml0ZV9mZCA9IE5vbmUKICAgICAgICBpZiBndWFyZGlhbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgaWYgZ3VhcmRpYW4uc3RkaW4gaXMgbm90IE5vbmUgYW5kIG5vdCBndWFyZGlhbi5zdGRpbi5jbG9zZWQ6CiAgICAgICAgICAgICAgICBndWFyZGlhbi5zdGRpbi5jbG9zZSgpCiAgICAgICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZmluYWxseToKICAgICAgICBpZiByZWFkX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZShyZWFkX2ZkKQogICAgICAgIGlmIHdyaXRlX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZSh3cml0ZV9mZCkKICAgICAgICBmb3Igc2lnLCBoYW5kbGVyIGluIG9sZF9oYW5kbGVycy5pdGVtcygpOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgaGFuZGxlcikKICAgIGlmIG5vdCByZWNlaXB0LmlzX2ZpbGUoKToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX21pc3NpbmdfcmVjZWlwdCIpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJHdWFyZGlhbiByZWNlaXB0IG1pc3Npbmc7IEdQVSBxdWFyYW50aW5lIHJlcXVpcmVzIG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uIikKICAgIHJlc3VsdCA9IGpzb24ubG9hZHMocmVjZWlwdC5yZWFkX3RleHQoKSkKICAgIGlmIGludGVycnVwdGVkIGlzIG5vdCBOb25lOgogICAgICAgIHJlc3VsdC51cGRhdGUocGFyZW50X2V4Y2VwdGlvbj1pbnRlcnJ1cHRlZCwgaW50ZXJydXB0ZWQ9VHJ1ZSkKICAgICAgICBpZiByZXN1bHRbImV4aXRfY29kZSJdID09IDA6CiAgICAgICAgICAgIHJlc3VsdFsiZXhpdF9jb2RlIl0gPSAxMzAKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImRyaXZlcl9pbnRlcnJ1cHRpb246IiArIGludGVycnVwdGVkKQogICAgaWYgbm90IHJlc3VsdFsiY2xlYW51cF9jb25maXJtZWQiXToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX3VuY2VydGFpbl9jbGVhbnVwIikKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkd1YXJkaWFuIGNsZWFudXAgdW5jZXJ0YWluOyBxdWFyYW50aW5lIHJldGFpbmVkIikKICAgIHJldHVybiByZXN1bHQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaWYgc3lzLmFyZ3ZbMTpdICE9IFsiLS1ndWFyZGlhbiJdOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkludGVybmFsIGd1YXJkaWFuIGVudHJ5cG9pbnQgb25seSIpCiAgICByYWlzZSBTeXN0ZW1FeGl0KGd1YXJkaWFuX21haW4oKSkK', 'owned_process_tree.py': 'IyBTaGFyZWQgdGFzayBvcmNoZXN0cmF0aW9uIGZyb20gdGhlIExvUkEgbGFuZTsgbm8ga2VybmVsIG93bmVyc2hpcCBjcmVkaXQuCiIiIkxpbnV4LW9ubHkgZGVkaWNhdGVkLXJ1bm5lciBjbGVhbnVwOyBubyBHUFUgaW1wb3J0cyBvciBiYWNrZ3JvdW5kIGFjdGl2aXR5LgoKT3JpZ2luYWwgdGFzay1vcmNoZXN0cmF0aW9uIGhlbHBlci4gTm90IHBhcnQgb2Ygc29sdXRpb24vIG9yIG93bi1rZXJuZWwgY3JlZGl0LgoiIiIKaW1wb3J0IGN0eXBlcwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKCmRlZiBwcmVwYXJlKCk6CiAgICAiIiJGYWlsIGJlZm9yZSBHUFUgd29yayBpZiByYWNlLXNhZmUgcHJvY2VzcyBvd25lcnNoaXAgaXMgdW5hdmFpbGFibGUuIiIiCiAgICBpZiBub3QgaGFzYXR0cihvcywncGlkZmRfb3BlbicpIG9yIG5vdCBoYXNhdHRyKHNpZ25hbCwncGlkZmRfc2VuZF9zaWduYWwnKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0xpbnV4IHBpZGZkIHByb2Nlc3Mgb3duZXJzaGlwIHJlcXVpcmVkJykKICAgIGZkPW9zLnBpZGZkX29wZW4ob3MuZ2V0cGlkKCkpO29zLmNsb3NlKGZkKQogICAgbGliYz1jdHlwZXMuQ0RMTChOb25lLHVzZV9lcnJubz1UcnVlKQogICAgaWYgbGliYy5wcmN0bCgzNiwxLDAsMCwwKSE9MDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0Nhbm5vdCBlbmFibGUgY2hpbGQgc3VicmVhcGVyOyBubyBHUFUgam9iIHN0YXJ0ZWQnKQogICAgZW5hYmxlZD1jdHlwZXMuY19pbnQoMCkKICAgIGlmIGxpYmMucHJjdGwoMzcsY3R5cGVzLmJ5cmVmKGVuYWJsZWQpLDAsMCwwKSE9MCBvciBlbmFibGVkLnZhbHVlIT0xOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ2hpbGQgc3VicmVhcGVyIHZlcmlmaWNhdGlvbiBmYWlsZWQnKQoKCmRlZiBpZGVudGl0eShwaWQpOgogICAgdHJ5OgogICAgICAgIGZpZWxkcz1QYXRoKGYnL3Byb2Mve3BpZH0vc3RhdCcpLnJlYWRfdGV4dCgpLnJzcGxpdCgnKScsMSlbMV0uc3BsaXQoKQogICAgICAgIHJldHVybiBpbnQoZmllbGRzWzFdKSxpbnQoZmllbGRzWzE5XSksZmllbGRzWzBdCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcixJbmRleEVycm9yKTpyZXR1cm4gTm9uZQoKCmRlZiBkZXNjZW5kYW50cygpOgogICAgcmVjb3Jkcz17fQogICAgZm9yIHAgaW4gUGF0aCgnL3Byb2MnKS5pdGVyZGlyKCk6CiAgICAgICAgaWYgcC5uYW1lLmlzZGlnaXQoKToKICAgICAgICAgICAgcmVjPWlkZW50aXR5KGludChwLm5hbWUpKQogICAgICAgICAgICBpZiByZWMgaXMgbm90IE5vbmU6cmVjb3Jkc1tpbnQocC5uYW1lKV09cmVjCiAgICBvd25lZD17b3MuZ2V0cGlkKCl9CiAgICB3aGlsZSBUcnVlOgogICAgICAgIGFkZD17cGlkIGZvciBwaWQsKHBhcmVudCxfLF8pIGluIHJlY29yZHMuaXRlbXMoKSBpZiBwYXJlbnQgaW4gb3duZWR9LW93bmVkCiAgICAgICAgaWYgbm90IGFkZDpicmVhawogICAgICAgIG93bmVkLnVwZGF0ZShhZGQpCiAgICByZXR1cm4ge3BpZDpyZWNvcmRzW3BpZF0gZm9yIHBpZCBpbiBvd25lZCBpZiBwaWQhPW9zLmdldHBpZCgpIGFuZCBwaWQgaW4gcmVjb3Jkc30KCgpkZWYgc2lnbmFsX2V4YWN0KHBpZCxiaXJ0aCxzaWcpOgogICAgIiIiTmV2ZXIgc2lnbmFsIGEgcmVjeWNsZWQgUElELiBUaGUgRkQgcGlucyBpZGVudGl0eSB0aHJvdWdoIHRoZSBzZW5kLiIiIgogICAgdHJ5OmZkPW9zLnBpZGZkX29wZW4ocGlkKQogICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjpyZXR1cm4KICAgIHRyeToKICAgICAgICBub3c9aWRlbnRpdHkocGlkKQogICAgICAgIGlmIG5vdyBpcyBub3QgTm9uZSBhbmQgbm93WzFdPT1iaXJ0aDoKICAgICAgICAgICAgdHJ5OnNpZ25hbC5waWRmZF9zZW5kX3NpZ25hbChmZCxzaWcsTm9uZSwwKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOnBhc3MKICAgIGZpbmFsbHk6b3MuY2xvc2UoZmQpCgoKZGVmIF90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWcpOgogICAgdHJ5OnNpZ25hbF9leGFjdChwaWQsYmlydGgsc2lnKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgICMgRG8gbm90IHVud2luZC9yZWxlYXNlIEdQVSBvd25lcnNoaXAgd2hlbiBjbGVhbnVwIGlzIHVuY2VydGFpbi4KICAgICAgICBxdWFyYW50aW5lKCdDbGVhbnVwIHNpZ25hbCBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgICAgIHByaW50KCdTT1VQX0NMRUFOVVBfUEVORElORyAnK3JlcHIoZXhjKSxmaWxlPXN5cy5zdGRlcnIsZmx1c2g9VHJ1ZSkKCgpkZWYgY2xlYW51cChwcm9jKToKICAgIHNhdmVkPXtzOnNpZ25hbC5zaWduYWwocyxzaWduYWwuU0lHX0lHTikgZm9yIHMgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pfQogICAgdHJ5OgogICAgICAgIGZyb3plbj1zZXQoKQogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIG9ic2VydmVkPWRlc2NlbmRhbnRzKCkKICAgICAgICAgICAgbmV3PXsocGlkLHZbMV0pIGZvciBwaWQsdiBpbiBvYnNlcnZlZC5pdGVtcygpfS1mcm96ZW4KICAgICAgICAgICAgaWYgbm90IG5ldzpicmVhawogICAgICAgICAgICBmb3IgcGlkLGJpcnRoIGluIG5ldzpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR1NUT1ApCiAgICAgICAgICAgIGZyb3plbi51cGRhdGUobmV3KQogICAgICAgIGZvciBwaWQsYmlydGggaW4gZnJvemVuOl90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWduYWwuU0lHS0lMTCkKICAgICAgICBwcm9jLndhaXQoKQogICAgICAgICMgTm8gdGltZW91dC1iYXNlZCByZWxlYXNlOiBpZiBhIEdQVSBjaGlsZCBpcyBzdHVjayBpbiBrZXJuZWwgZXhpdCwKICAgICAgICAjIHJldGFpbiB0aGlzIHN1cGVydmlzb3IgYW5kIGl0cyBsb2NrIHVudGlsIGV4aXQgaXMgYWN0dWFsbHkgb2JzZXJ2ZWQuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwaWQsXz1vcy53YWl0cGlkKC0xLG9zLldOT0hBTkcpCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHBpZDpicmVhawogICAgICAgICAgICAgICAgZXhjZXB0IENoaWxkUHJvY2Vzc0Vycm9yOmJyZWFrCiAgICAgICAgICAgIHJlbWFpbmluZz1kZXNjZW5kYW50cygpCiAgICAgICAgICAgIGlmIG5vdCByZW1haW5pbmc6YnJlYWsKICAgICAgICAgICAgZm9yIHBpZCwoXyxiaXJ0aCxfKSBpbiByZW1haW5pbmcuaXRlbXMoKTpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoLjA1KQogICAgZmluYWxseToKICAgICAgICBmb3Igc2lnLGhhbmRsZXIgaW4gc2F2ZWQuaXRlbXMoKTpzaWduYWwuc2lnbmFsKHNpZyxoYW5kbGVyKQoKCmRlZiBydW4oYXJndiwqLGN3ZCxlbnYsc3Rkb3V0LHRpbWVvdXQ9Tm9uZSxzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpOgogICAgIiIiUnVuIGluIGRlZGljYXRlZCBzdXBlcnZpc29yIG9ubHk7IGNsZWFudXAgbmVzdGVkIHNlc3Npb25zIG9uIGFueSBmYWlsdXJlLiIiIgogICAgcHJvYz1zdWJwcm9jZXNzLlBvcGVuKGFyZ3YsY3dkPWN3ZCxlbnY9ZW52LHN0ZG91dD1zdGRvdXQsc3RkZXJyPXN0ZGVycixzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgdHJ5OgogICAgICAgIGNvZGU9cHJvYy53YWl0KHRpbWVvdXQ9dGltZW91dCkKICAgICAgICBpZiBkZXNjZW5kYW50cygpOgogICAgICAgICAgICBxdWFyYW50aW5lKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50cycpCiAgICAgICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50czsgY2xlYW5lZCBhbmQgcmVqZWN0ZWQnKQogICAgICAgIHJldHVybiBjb2RlCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcXVhcmFudGluZSgnT3V0ZXIgdGltZW91dC9pbnRlcnJ1cHRpb24vZXJyb3I6ICcrcmVwcihleGMpKQogICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICByYWlzZQoKCmRlZiBpbnN0YWxsX3NpZ25hbF9oYW5kbGVycygpOgogICAgZGVmIGludGVycnVwdGVkKHNpZ251bSxmcmFtZSk6CiAgICAgICAgcXVhcmFudGluZSgnUnVubmVyIGludGVycnVwdGVkIGJ5IHNpZ25hbCAnK3N0cihzaWdudW0pKQogICAgICAgIHJhaXNlIEludGVycnVwdGVkRXJyb3IoJ1J1bm5lciBpbnRlcnJ1cHRlZCBieSBzaWduYWwgJytzdHIoc2lnbnVtKSkKICAgIGZvciBzaWcgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pOnNpZ25hbC5zaWduYWwoc2lnLGludGVycnVwdGVkKQoKIyBBIGNyYXNoLXN1cnZpdmluZyBtYXJrZXIgaXMgcGxhY2VkIGJlZm9yZSBhbnkgR1BVIHN1YnByb2Nlc3MuIE9ubHkgaXRzIGV4YWN0CiMgb3duZXIgbWF5IGNsZWFyIGl0IG9uIGEgY2xlYW4sIG5vbi1pbnRlcnJ1cHRlZCBleGl0IHdpdGggbm8gbGl2ZSBkZXNjZW5kYW50cy4KIyBJbnRlcnJ1cHRlZC90aW1lZC1vdXQgd29yayBsZWF2ZXMgaXQgZm9yIGV4cGxpY2l0IG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uLgpfTUFSS0VSID0gTm9uZQpfVE9LRU4gPSBOb25lCl9LRUVQX01BUktFUiA9IEZhbHNlCgoKZGVmIGNoZWNrX3F1YXJhbnRpbmUocGF0aCk6CiAgICBpZiBQYXRoKHBhdGgpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVMCBxdWFyYW50aW5lZDogcmVjb25jaWxlICcrc3RyKHBhdGgpKycgYmVmb3JlIGFub3RoZXIgcnVuJykKCgpkZWYgYmVnaW5fcXVhcmFudGluZShwYXRoKToKICAgIGdsb2JhbCBfTUFSS0VSLF9UT0tFTixfS0VFUF9NQVJLRVIKICAgIGltcG9ydCBqc29uLHV1aWQKICAgIF9NQVJLRVI9UGF0aChwYXRoKTtfVE9LRU49dXVpZC51dWlkNCgpLmhleDtfS0VFUF9NQVJLRVI9RmFsc2UKICAgIGZkPW9zLm9wZW4oc3RyKF9NQVJLRVIpLG9zLk9fV1JPTkxZfG9zLk9fQ1JFQVR8b3MuT19FWENMfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICB3aXRoIG9zLmZkb3BlbihmZCwndycpIGFzIG91dDoKICAgICAgICBqc29uLmR1bXAoeydvd25lcl90b2tlbic6X1RPS0VOLCdvd25lcl9waWQnOm9zLmdldHBpZCgpLCdzdGF0dXMnOidPV05FRF9HUFVfV09SS19JTl9QUk9HUkVTUyd9LG91dCkKICAgICAgICBvdXQuZmx1c2goKTtvcy5mc3luYyhvdXQuZmlsZW5vKCkpCgoKZGVmIHF1YXJhbnRpbmUocmVhc29uKToKICAgIGdsb2JhbCBfS0VFUF9NQVJLRVIKICAgIF9LRUVQX01BUktFUj1UcnVlCiAgICBpZiBfTUFSS0VSIGlzIE5vbmU6cmV0dXJuCiAgICBpbXBvcnQganNvbgogICAgdHJ5OnZhbHVlPWpzb24ubG9hZHMoX01BUktFUi5yZWFkX3RleHQoKSkKICAgIGV4Y2VwdCAoT1NFcnJvcixWYWx1ZUVycm9yKTpyZXR1cm4KICAgIGlmIHZhbHVlLmdldCgnb3duZXJfdG9rZW4nKSE9X1RPS0VOOnJldHVybgogICAgdmFsdWUudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9PUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCcscmVhc29uPXN0cihyZWFzb24pKQogICAgX01BUktFUi53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUpKydcbicpCgoKZGVmIGZpbmlzaF9xdWFyYW50aW5lKCk6CiAgICBpZiBfTUFSS0VSIGlzIE5vbmUgb3IgX0tFRVBfTUFSS0VSOnJldHVybgogICAgaWYgZGVzY2VuZGFudHMoKToKICAgICAgICBxdWFyYW50aW5lKCdMaXZlIG93bmVkIGRlc2NlbmRhbnRzIHJlbWFpbiBhdCBzdXBlcnZpc29yIGV4aXQnKQogICAgICAgIHJldHVybgogICAgaW1wb3J0IGpzb24KICAgIHRyeTp2YWx1ZT1qc29uLmxvYWRzKF9NQVJLRVIucmVhZF90ZXh0KCkpCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcik6cmV0dXJuCiAgICBpZiB2YWx1ZS5nZXQoJ293bmVyX3Rva2VuJyk9PV9UT0tFTjpfTUFSS0VSLnVubGluaygpCg==', 'runtime_probe.py': 'IiIiUmVhZC1vbmx5LCBzaG9ydC1saXZlZCBkZXBsb3llZC10ZW1wbGF0ZSBnYXRlLiBOZXZlciBwcmludHMgc2VjcmV0IHZhbHVlcy4iIiIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtZXRhZGF0YSxqc29uLG9zLHBsYXRmb3JtLHN1YnByb2Nlc3Msc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApFWFBFQ1RFRD17J3RvcmNoJzonMi4xMS4wK2N1MTI4JywndHJpdG9uJzonMy42LjAnLCdudW1weSc6JzIuMy41Jywnc291cC1jbGknOicwLjc3LjAnLCd0cmFuc2Zvcm1lcnMnOic1LjE2LjEnLCdwZWZ0JzonMC4yMC4wJywndHJsJzonMC4yOS4xJywnYml0c2FuZGJ5dGVzJzonMC41MC4yJywnYWNjZWxlcmF0ZSc6JzEuMTQuMCcsJ2RhdGFzZXRzJzonNC44LjUnfQoKZGVmIG1haW4oKToKIG91dD1QYXRoKHN5cy5hcmd2WzFdKTtyPXsncGFzc2VkJzpGYWxzZSwnZ3B1X3dvcmtsb2FkX2V4ZWN1dGVkJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogdHJ5OgogIHZlcnNpb25zPXtrOm1ldGFkYXRhLnZlcnNpb24oaykgZm9yIGsgaW4gRVhQRUNURUR9CiAgci51cGRhdGUocHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCkscGFja2FnZXM9dmVyc2lvbnMsdWlkPW9zLmdldHVpZCgpLGNwdV9hZmZpbml0eT1zb3J0ZWQob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpKQogIGFzc2VydCBwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpPT0nMy4xMy4xNScsJ1B5dGhvbiBwaW4gbWlzbWF0Y2gnCiAgYXNzZXJ0IHZlcnNpb25zPT1FWFBFQ1RFRCwnRXhhY3QgcGFja2FnZSBwaW5zIG1pc21hdGNoJwogIGFzc2VydCBvcy5nZXR1aWQoKT09MTAwMDEsJ0RlcGxveWVkIHJ1bnRpbWUgdWlkIG1pc21hdGNoJwogIGFzc2VydCBsZW4ob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpPT00LCdGb3VyIGFzc2lnbmVkIENQVXMgcmVxdWlyZWQnCiAgZm9yIHAsd2FudGVkIGluIFsoJ21lbW9yeS5tYXgnLHN0cigzMCoxMDI0KiozKSksKCdtZW1vcnkuc3dhcC5tYXgnLCcwJyksKCdjcHVzZXQuY3B1cy5lZmZlY3RpdmUnLCcwLTMnKV06CiAgIGFjdHVhbD0oUGF0aCgnL3N5cy9mcy9jZ3JvdXAnKS9wKS5yZWFkX3RleHQoKS5zdHJpcCgpO3JbcF09YWN0dWFsCiAgIGFzc2VydCBhY3R1YWw9PXdhbnRlZCwnRGVwbG95ZWQgY2dyb3VwIG1pc21hdGNoOiAnK3AKICBhc3NlcnQgbm90IFBhdGgoJy92YXIvcnVuL2RvY2tlci5zb2NrJykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgRG9ja2VyIHNvY2tldCcKICBhc3NlcnQgbm90IFBhdGgoJy9ldGMvc291cC1xdWV1ZS93b3JrZXIuZW52JykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgd29ya2VyIGVudmlyb25tZW50IGZpbGUnCiAgYXNzZXJ0IG5vdCBhbnkoJ1RPS0VOJyBpbiBrIGFuZCBrIT0nVE9LRU5JWkVSU19QQVJBTExFTElTTScgZm9yIGsgaW4gb3MuZW52aXJvbiksJ1Rva2VuLWJlYXJpbmcgZW52aXJvbm1lbnQgcmVqZWN0ZWQgd2l0aG91dCByZWFkaW5nIGl0cyB2YWx1ZScKICBhc3NlcnQgbm90IG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyksJ0ludGVycHJldGVyIG1vZGUgZm9yYmlkZGVuJwogIGltcG9ydCB0b3JjaCx0cml0b24sbnVtcHkKICBhc3NlcnQgdG9yY2guX192ZXJzaW9uX189PScyLjExLjArY3UxMjgnIGFuZCB0b3JjaC52ZXJzaW9uLmN1ZGE9PScxMi44JwogIGFzc2VydCB0cml0b24uX192ZXJzaW9uX189PSczLjYuMCcgYW5kIG51bXB5Ll9fdmVyc2lvbl9fPT0nMi4zLjUnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk9PTEgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApPT0nVGVzbGEgVDQnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApPT0oNyw1KQogIHIudXBkYXRlKGdwdV9uYW1lPXRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLGN1ZGFfYnVpbGQ9dG9yY2gudmVyc2lvbi5jdWRhLHRvcmNoX2dpdD10b3JjaC52ZXJzaW9uLmdpdF92ZXJzaW9uKQogIGZyb20gaHVnZ2luZ2ZhY2VfaHViLmNvbnN0YW50cyBpbXBvcnQgSEZfSFVCX0NBQ0hFCiAgclsncmVzb2x2ZWRfcHVibGljX2h1Yl9jYWNoZSddPXN0cihQYXRoKEhGX0hVQl9DQUNIRSkucmVzb2x2ZSgpKQogIHJbJ3Bhc3NlZCddPVRydWUKIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6clsnZXJyb3InXT1yZXByKGUpCiBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHIsaW5kZW50PTIpKydcbicpO3ByaW50KGpzb24uZHVtcHMocixpbmRlbnQ9MikpCiByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgclsncGFzc2VkJ10gZWxzZSAxKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'official_manifest.json': 'ewogICJoYXJuZXNzL190cmFpbi5weSI6ICI0NWE2MjY0MmRjZWFiMzJjZjNlZmNkYTUwNGRlODdhOWMyNWVhMjhiMTE5YjE4MDZmNTZkMTRmZjExNDZhODlkIiwKICAiaGFybmVzcy9jaGVja19ydWxlcy5weSI6ICI4NzlhYjE4NjE2NzRmNmU3MTRlYWRkMDBhYWYwNDQyMzVkOTg1MjZkZDc1ZmFiMTAzNWRlZWMzNDAxMTIxZWY1IiwKICAiaGFybmVzcy9jb21tb24ucHkiOiAiYzgzMDY2ZjI0ZjI2NmRhYjYyZWJjYzljMmQ0NGY1YTY1OTAyNzYwYjUzZGE5MTJlNjVlYmZlNTExNTc5YzJkYiIsCiAgImhhcm5lc3MvZXZhbF9hZGFwdGVyLnB5IjogIjQ3Y2RkYmI4ZTQ1NzljNjlmOTUzNWIwOTlmZjIzNzFiMWMzYWYwOGJmYmQ5M2QxNGJmNzlhYWRkNzRkMzg0ODIiLAogICJoYXJuZXNzL2V4YW1wbGVfcmVmZXJlbmNlLnB5IjogIjZhMTEwNTVmNGExODIyZTUzOTcxYzIyOGM2ZTNlM2FhNmI4OGVkZDllYjZkYjgxYTM4OGUyNmE3MGEwMzBjZWIiLAogICJoYXJuZXNzL3J1bl90YXNrLnB5IjogImQ5MzVhODk1OWYzNDg5NzczMTNmMmVlZjM2ODcwZmYyNjk5YjliYTAxM2E3OTc4OTE4ZmYzNWUxNDg5NjhhNzkiLAogICJoYXJuZXNzL3Njb3JlLnB5IjogIjczMWM5YjY0NGRjYzQwNjliMGIwNjk2MTZmNDM4MDY0ZDFmN2YzNzZhMjAyZmViMTUyMDFmZDkxMGVjNjg4YzUiLAogICJ0YXNrL3ByZXBhcmVfZGF0YS5weSI6ICJhODU0M2YxMWE1NWFmZjgwZGJkYzcxNGEwZTYyMzkxMjQ0YTUyOTY2MTZlZWEyZTM0ZGMyZWIzZmYzZWMxOGY3IiwKICAidGFzay90YXNrLnlhbWwiOiAiZGQ3MGM3MzlmY2QxNDNlMDZhZWYzYmIxZWZjYzkzODZmNjgyYmYxZGUyNmYzMGUwODQwOWQ4NmE5N2ZmNDdjZSIsCiAgInRlc3RzL2NvbmZ0ZXN0LnB5IjogImE4MmVhYmFiNDY4ZWEwZWYxZTAwM2NjY2Q3YzM4MjdjZTJmNGQwMGQwOWMzMjk0OGEzNGZiNDgyMjFmNzMzN2EiLAogICJ0ZXN0cy90ZXN0X2NhbGxiYWNrLnB5IjogIjE5MTkzMDNhMzY3MGMwYTMzYWJiZmNlOWIzM2VhYjA5MTNiYjY0YTNhMjFjMzMxMmQ4NjUzMWMzMmI1YmY0OTAiLAogICJ0ZXN0cy90ZXN0X2NvbmZpZy5weSI6ICJkYjgwOGM0NDQ3MGQxNDRiN2NmMThiNzMyOTQ5MmY0MWY3NmUzN2NmYzI0MzA2ZTdjYmEwNDg4YmNhOWQ3ODdjIiwKICAidGVzdHMvdGVzdF9mb3JtYXRzLnB5IjogIjVkNjU2ZWQ1MGM1OWY3OTRlOTM0NDczZDQ1OWVhODU2MzVmODkyYmEwMjhlMTkyNDE0NDc0ZTliNWQ2NTkxOWUiLAogICJ0ZXN0cy90ZXN0X2xvYWRlci5weSI6ICIzZGFkMWJlYTJkMzNhYWI5N2U1MmI3Y2Q0NDg4OGQ3NjhiYzk1MTlhZjQ0ZmJlNmNmM2U5NmMyNjAwMjY2MzJmIiwKICAidGVzdHMvdGVzdF90cmFpbmVyX2luaXQucHkiOiAiYjc2MDhkNjJhNDA5ODNiZjdjOWU1YjI3MGUwZTUyN2I2NDFhODM3ZTQ2MmQ2ZGYyOTA5ZmVjODhmNjA3ZTgyMSIKfQo=', 'reviewed_support.py': 'aW1wb3J0IGFyZ3BhcnNlLGZjbnRsLGhhc2hsaWIsanNvbixtYXRoLG9zLHNodXRpbCxzeXMsdGhyZWFkaW5nLHRpbWUsdXVpZCx6aXBmaWxlCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwcm9jZXNzX2d1YXJkCgppbXBvcnQgb3duZWRfcHJvY2Vzc190cmVlIGFzIHRyZWUKCmRlZiBzaGEocCk6cmV0dXJuIGhhc2hsaWIuc2hhMjU2KFBhdGgocCkucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKQoKZGVmIHJlYWQocCk6cmV0dXJuIGpzb24ubG9hZHMoUGF0aChwKS5yZWFkX3RleHQoKSkKCmRlZiB3cml0ZShwLHZhbHVlKToKIHA9UGF0aChwKTtwZW5kaW5nPXAud2l0aF9uYW1lKHAubmFtZSsnLnBlbmRpbmcnKTtwZW5kaW5nLndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSxpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpKydcbicpO3BlbmRpbmcucmVwbGFjZShwKQoKZGVmIHZlcmlmeV9vZmZpY2lhbChyZXBvLG1hbmlmZXN0KToKIGlmIG5vdCByZXBvLmlzX2RpcigpIG9yIHJlcG8uaXNfc3ltbGluaygpOnJhaXNlIFJ1bnRpbWVFcnJvcignU09VUF9SRVBPIG11c3QgYmUgYSByZWFsIHByZXBhcmVkIGpvYiBkaXJlY3RvcnknKQogZm9yIG5hbWUsd2FudGVkIGluIG1hbmlmZXN0Lml0ZW1zKCk6CiAgcD1yZXBvL25hbWUKICBpZiBub3QgcC5pc19maWxlKCkgb3IgcC5pc19zeW1saW5rKCkgb3Igc2hhKHApIT13YW50ZWQ6cmFpc2UgUnVudGltZUVycm9yKCdQcm90ZWN0ZWQgb2ZmaWNpYWwgc291cmNlIG1pc21hdGNoOiAnK25hbWUpCiBhY3R1YWw9e3N0cihwLnJlbGF0aXZlX3RvKHJlcG8pKSBmb3IgZm9sZGVyIGluICgnaGFybmVzcycsJ3Rhc2snLCd0ZXN0cycpIGZvciBwIGluIChyZXBvL2ZvbGRlcikucmdsb2IoJyonKQogICAgICAgICBpZiBwLmlzX2ZpbGUoKSBhbmQgJ19fcHljYWNoZV9fJyBub3QgaW4gcC5wYXJ0cyBhbmQgcC5zdWZmaXggbm90IGluICgnLnB5YycsJy5weW8nKX0KIGlmIGFjdHVhbCE9c2V0KG1hbmlmZXN0KTpyYWlzZSBSdW50aW1lRXJyb3IoJ1VuZXhwZWN0ZWQgcHJvdGVjdGVkIG9mZmljaWFsIGZpbGVzJykKCkVYUE9SVF9CVURHRVQ9NTYqMTAyNCoqMgoKRVhQT1JUX09NSVRURUQ9e30KCmRlZiBleHBvcnRfc21hbGwocnVuLGV4cG9ydCk6CiAiIiJSZWFkLW9ubHkgaW5jcmVtZW50YWwgZXhwb3J0OyBubyBhZGFwdGVycywgd2VpZ2h0cyBvciBjb21waWxlciBjYWNoZXMuIiIiCiB1c2VkPXN1bShwLnN0YXQoKS5zdF9zaXplIGZvciBwIGluIGV4cG9ydC5yZ2xvYignKicpIGlmIHAuaXNfZmlsZSgpKQogZXhjbHVkZWQ9eydhZGFwdGVyJywndHJpdG9uX2NhY2hlJywnaW5kdWN0b3JfY2FjaGUnLCdwcmVmbGlnaHRfdHJpdG9uJywncHJlZmxpZ2h0X2luZHVjdG9yJywnb3JpZ2luYWxfc29sdXRpb24nfQogZm9yIGJhc2UsZGlycyxmaWxlcyBpbiBvcy53YWxrKHJ1bik6CiAgZGlyc1s6XT1bZCBmb3IgZCBpbiBkaXJzIGlmIGQgbm90IGluIGV4Y2x1ZGVkIGFuZCBub3QgZC5zdGFydHN3aXRoKCdjaGVja3BvaW50LScpIGFuZCBub3QgKFBhdGgoYmFzZSkvZCkuaXNfc3ltbGluaygpXQogIGZvciBuYW1lIGluIGZpbGVzOgogICBwYXRoPVBhdGgoYmFzZSkvbmFtZTtyZWw9cGF0aC5yZWxhdGl2ZV90byhydW4pCiAgIHRyeToKICAgIGlmIG5vdCBwYXRoLmlzX3N5bWxpbmsoKSBhbmQgcGF0aC5zdWZmaXggaW4gKCcuanNvbicsJy5qc29ubCcsJy5sb2cnLCcueWFtbCcpOgogICAgIGlmIHBhdGguc3RhdCgpLnN0X3NpemU+PTQqMTAyNCoqMjoKICAgICAgRVhQT1JUX09NSVRURURbc3RyKHJlbCldPSdwZXJfZmlsZV9saW1pdF80TWlCJztjb250aW51ZQogICAgIGRlc3Q9ZXhwb3J0LydyYXcnL3JlbDtkZXN0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgICBwcmlvcj1kZXN0LnN0YXQoKS5zdF9zaXplIGlmIGRlc3QuZXhpc3RzKCkgZWxzZSAwCiAgICAgc2l6ZT1wYXRoLnN0YXQoKS5zdF9zaXplCiAgICAgaWYgdXNlZC1wcmlvcitzaXplPkVYUE9SVF9CVURHRVQ6CiAgICAgIEVYUE9SVF9PTUlUVEVEW3N0cihyZWwpXT0nYWdncmVnYXRlX2J1ZGdldF81Nk1pQic7Y29udGludWUKICAgICB0ZW1wPWRlc3Qud2l0aF9uYW1lKGRlc3QubmFtZSsnLmNvcHlpbmcnKTtzaHV0aWwuY29weWZpbGUocGF0aCx0ZW1wKTt0ZW1wLnJlcGxhY2UoZGVzdCk7dXNlZD11c2VkLXByaW9yK3NpemUKICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvcixPU0Vycm9yKTpwYXNzCgpjbGFzcyBFeHBvcnRlcjoKIGRlZiBfX2luaXRfXyhzZWxmLHJ1bixleHBvcnQpOnNlbGYucnVuPXJ1bjtzZWxmLmV4cG9ydD1leHBvcnQ7c2VsZi5zdG9wPXRocmVhZGluZy5FdmVudCgpO3NlbGYudGhyZWFkPU5vbmUKIGRlZiBfX2VudGVyX18oc2VsZik6CiAgZGVmIGxvb3AoKToKICAgd2hpbGUgbm90IHNlbGYuc3RvcC53YWl0KDEpOmV4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKICBzZWxmLnRocmVhZD10aHJlYWRpbmcuVGhyZWFkKHRhcmdldD1sb29wLGRhZW1vbj1UcnVlKTtzZWxmLnRocmVhZC5zdGFydCgpO3JldHVybiBzZWxmCiBkZWYgX19leGl0X18oc2VsZiwqZXhjKTpzZWxmLnN0b3Auc2V0KCk7c2VsZi50aHJlYWQuam9pbigpO2V4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKCmRlZiBpZGxlX2NoZWNrKGVudixyZXBvLHJ1bixsb2NrZmQpOgogY29kZT0nJydpbXBvcnQganNvbixzdWJwcm9jZXNzLHN5cwpwPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktY29tcHV0ZS1hcHBzPXBpZCx1c2VkX2dwdV9tZW1vcnknLCctLWZvcm1hdD1jc3Ysbm9oZWFkZXIsbm91bml0cyddLGNhcHR1cmVfb3V0cHV0PVRydWUsdGV4dD1UcnVlLGNoZWNrPVRydWUpCmFzc2VydCBub3QgcC5zdGRvdXQuc3RyaXAoKSwnR1BVIHByb2Nlc3NlcyBzdGlsbCBhY3RpdmUgYWZ0ZXIgcHJvYmUvcHJlZmxpZ2h0JwpyPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktZ3B1PWluZGV4LG1lbW9yeS51c2VkJywnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyLG5vdW5pdHMnXSxjYXB0dXJlX291dHB1dD1UcnVlLHRleHQ9VHJ1ZSxjaGVjaz1UcnVlKQpyb3dzPVtsaW5lLnNwbGl0KCcsJykgZm9yIGxpbmUgaW4gci5zdGRvdXQuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KYXNzZXJ0IHJvd3MgYW5kIHN1bShpbnQoeFsxXSkgZm9yIHggaW4gcm93cyk8MTAwMCwnR1BVIHBoeXNpY2FsIGludmVudG9yeSBub3QgaWRsZScKcHJpbnQoanNvbi5kdW1wcyh7J2lkbGUnOlRydWUsJ3BoeXNpY2FsX21lbW9yeV9yb3dzJzpyb3dzfSkpJycnCiByZXR1cm4gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctYycsY29kZV0sY3dkPXJlcG8sZW52PWVudixsb2c9cnVuLydpZGxlLmxvZycsdGltZW91dD0yMCxsb2NrX2ZkPWxvY2tmZCkK', 'worker.py': '...', 'native_source_manifest.json': 'ewogICJsb3NzX3V0aWxzLnB5IjogIjgzZGIxNmIyNGNlNWMzYTA2NDIwOTc2MjRhYTlhMGFlN2ViNzI0NDVjNDFkOGI0ZDUwNTlhMTI5ODYyZmRiYmUiLAogICJ1dGlscy5weSI6ICJlMWE2YWNkNDJjMTY5ZjczMTA1NmJiMDJiMTRmMDA5NjAwOGM2ZGVmOTc3ODIyZjMyNWMxYTdiZjFmNjQ1ZDFmIiwKICAicmV3aW5kX2hmLnB5IjogIjFmZTk1NDJlMWFiNjM0ZDFlNmYzY2U5ZGM1MGYyODg3MmQwMDk2Y2QwZjdmZGQyYmUzNDRjY2UxMGFmZDU2MDQiLAogICJvcGVyYXRpb25zLnB5IjogIjk3YTJlNThkNDkyODQ5MTI2OWM1NzM5Y2M3MjNlODRjNjQ4N2Q3OTQ2MmQ4OWQ4NmE5MmZlYjRmZjczYjdiNzMiCn0K', 'shared_consumer_manifest.json': 'ewogICJicmlkZ2Vfc3RhdGUucHkiOiAiZTNiYTY0NDQ2YThiYjNlYjhjZDZlYzMxYjRlNWQwMDE4YTQ3YzhmM2I5MGQwMGRhOTU1ZGMyZTQ0NGViMThlOSIsCiAgImNvbnN1bWVyLnB5IjogImYwMThhOWI2ODA4ZDc2MzgxYTIwNTU4MGU4YTJmZjlkMmMwZWI3MDZmYTM0Yzg0MzRmZTQxYjBhZDk5ZGEyYmMiLAogICJfX2luaXRfXy5weSI6ICJlM2IwYzQ0Mjk4ZmMxYzE0OWFmYmY0Yzg5OTZmYjkyNDI3YWU0MWU0NjQ5YjkzNGNhNDk1OTkxYjc4NTJiODU1Igp9Cg==', 'tile_source_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICJlM2IwYzQ0Mjk4ZmMxYzE0OWFmYmY0Yzg5OTZmYjkyNDI3YWU0MWU0NjQ5YjkzNGNhNDk1OTkxYjc4NTJiODU1IiwKICAic29sdXRpb24vY2VfdGlsZS5weSI6ICIyYmFlYTFmNGI3YzU2NzM3OWUzNjc0Yjc2MjA0ZDM5ZTlmYjEyOGViNmNmY2RhOWE4ZDhlZGViYmZkMjMwZTdiIiwKICAic29sdXRpb24vb3JpZ2luYWwvX19pbml0X18ucHkiOiAiZTNiMGM0NDI5OGZjMWMxNDlhZmJmNGM4OTk2ZmI5MjQyN2FlNDFlNDY0OWI5MzRjYTQ5NTk5MWI3ODUyYjg1NSIsCiAgInNvbHV0aW9uL29yaWdpbmFsL21hdGVyaWFsaXplZF9jZS5weSI6ICI2NTE5YzhmMWRkY2U3ZGIyNzBmYmU0MjVhOGU4NzcxYWVjZGEzZjI5NGVjYzgzODM5ZDQ3NTk1MzFmZDFkY2UxIiwKICAic29sdXRpb24vb3JpZ2luYWwvbWV0cmljX3N1bW1hcnkucHkiOiAiOGM1ZWRlYTAwYzdmMGI0MjUwMWM3MWUzNGE5ODZkZTA1YzI4YzNjNTZlZGY0ZDYxODFkMmRkYWYyNDE0MmNmNCIsCiAgInNvbHV0aW9uL29yaWdpbmFsL21peGVkX3Jvd3MucHkiOiAiMDkxMTU3Y2RiOTA0YjBhMmZlZTA5ZDcyMDg2MmIzNzBkOGY3MmY5ZDgyNDAyM2QzN2VkYmRmOWM1ZTE3MjZiMiIsCiAgInNvbHV0aW9uL3ZlcmlmeV90aWxlLnB5IjogImY0ZjYyNTE5ODdiNjE5OTM2NWI4NTNkNDg5NzlhMTM2ZWMyZGMyNjI2YmVkNjhmMzc0YWUyOTQwZWU2MDQ3YTEiCn0K', 'driver.py': 'IiIiQm91bmRlZCBuYXRpdmUvY2h1bmsgZnJvemVuLWhlYWQgSEVBRF9DT01QT05FTlRfUFJFRkxJR0hUX09OTFkgb3JjaGVzdHJhdGlvbi4KCk5ldmVyIGluc3RhbGxzIGludG8gU09VUF9SRVBPIG9yIGNhbGxzIGEgdHJhaW5pbmcgaGFybmVzcy4gRXhpc3Rpbmcgc291cmNlL3J1bnRpbWUsCmxvY2ssIHBpZGZkL3N1YnJlYXBlciwgcXVhcmFudGluZSBhbmQgaWRsZS1jaGVjayBoZWxwZXJzIGFyZSBwcmVzZXJ2ZWQgdW5jaGFuZ2VkLgpUaGUgZGVwbG95ZWQgc2VydmljZSdzIGxpbWl0cyByZW1haW4gYXV0aG9yaXRhdGl2ZTsgZWFjaCBHUFUgY2hpbGQgZ2V0cyA8PTgwIHMuCiIiIgppbXBvcnQgZmNudGwsIGhhc2hsaWIsIGpzb24sIG9zLCBzeXMsIHRpbWUsIHV1aWQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBwcm9jZXNzX2d1YXJkCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQpmcm9tIHJldmlld2VkX3N1cHBvcnQgaW1wb3J0IHZlcmlmeV9vZmZpY2lhbCwgaWRsZV9jaGVjaywgRXhwb3J0ZXIsIGV4cG9ydF9zbWFsbCwgcmVhZCwgd3JpdGUsIHNoYQoKTE9DSyA9IFBhdGgoJy90bXAvc291cC1ncHUwLmxvY2snKQpNQVJLRVIgPSBQYXRoKCcvdG1wL3NvdXAtZ3B1MC5sb2NrLmJsb2NrZWQnKQoKZGVmIG1haW4oKToKICAgIGhlcmUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CiAgICBpZiBub3Qgb3MuZW52aXJvbi5nZXQoJ1NPVVBfUkVQTycpIG9yIG5vdCBvcy5lbnZpcm9uLmdldCgnU09VUF9SRVNVTFRTJyk6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgnUmVxdWlyZXMgZGVwbG95ZWQgU09VUF9SRVBPL1NPVVBfUkVTVUxUUywgbm8gbWFudWFsIGZhbGxiYWNrJykKICAgIHJlcG8sIHJlc3VsdHMgPSBQYXRoKG9zLmVudmlyb25bJ1NPVVBfUkVQTyddKS5hYnNvbHV0ZSgpLCBQYXRoKG9zLmVudmlyb25bJ1NPVVBfUkVTVUxUUyddKS5hYnNvbHV0ZSgpCiAgICBpZiByZXBvLmlzX3N5bWxpbmsoKSBvciByZXN1bHRzLmlzX3N5bWxpbmsoKSBvciByZXBvID09IHJlc3VsdHM6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgnUHJlcGFyZWQgY2hlY2tvdXQvcmVzdWx0cyBtdXN0IGJlIHNlcGFyYXRlIHJlYWwgZGlyZWN0b3JpZXMnKQogICAgYXNzZXJ0IFBhdGgoJy9rYWdnbGUvd29ya2luZycpLmlzX2RpcigpCiAgICByZXN1bHRzLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgaWRlbnRpdHkgPSAndG9rZW4tY2h1bmstY2UtaGVhZC1jb21wb25lbnQtb25seS0nICsgdXVpZC51dWlkNCgpLmhleFs6MTJdCiAgICBydW4sIGV4cG9ydCA9IFBhdGgoJy9rYWdnbGUvd29ya2luZycpL2lkZW50aXR5LCByZXN1bHRzL2lkZW50aXR5CiAgICBydW4ubWtkaXIoZXhpc3Rfb2s9RmFsc2UpOyBleHBvcnQubWtkaXIoZXhpc3Rfb2s9RmFsc2UpCiAgICBzdGF0dXMgPSBkaWN0KHN0YXR1cz0nUFJFUEFSSU5HX01JQ1JPQkVOQ0hfT05MWScsc2NvcGU9J0hFQURfQ09NUE9ORU5UX1BSRUZMSUdIVF9PTkxZJywKICAgICAgICAgICAgICAgICAgdHJhaW5pbmdfZXhlY3V0ZWQ9RmFsc2UsY29tcGxldGVfZml4ZWRfdGFzaz1GYWxzZSxzY29yZT1Ob25lLGFkbWlzc2lvbj0nVU5LTk9XTicsb3JpZ2luYWxfdHJpdG9uX2NyZWRpdD1GYWxzZSwKICAgICAgICAgICAgICAgICAgbWV0aG9kX3ZhcmlhbnRzPXsnbmF0aXZlJzonbmF0aXZlX2FtcF9saW5lYXJfbmF0aXZlX2xvc3NfbWV0cmljcycsJ29sZF9jaHVuayc6J3RhcmdldF9wYXNzZWRfbmF0aXZlX2NlX3Rva2VuX2NodW5rcycsJ2NlX2NodW5rJzondG9rZW5fY2h1bmtzX29yaWdpbmFsX2NlX3NhdmVkX25vcm1hbGl6ZXJzX21ldHJpY3MnfSwKICAgICAgICAgICAgICAgICAgcGVyX2dwdV9jaGlsZF93YWxsX2NhcF9zPTgwLGNhcF9ieXBhc3M9RmFsc2UsCiAgICAgICAgICAgICAgICAgIGNhbmRpZGF0ZV9zb3VyY2Vfc2hhMjU2PXJlYWQoaGVyZS8nc291cmNlX21hbmlmZXN0Lmpzb24nKSwKICAgICAgICAgICAgICAgICAgbWV0aG9kc19pbl9leGVjdXRpb25fb3JkZXI9WyduYXRpdmUnLCdvbGRfY2h1bmsnLCdjZV9jaHVuayddLHByb2Nlc3Nlcz17fSkKICAgIHdyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN0YXR1cykKICAgIGxvY2tmZCA9IE5vbmU7IHN0YXJ0ZWQgPSB0aW1lLm1vbm90b25pYygpCiAgICB0cnk6CiAgICAgICAgZm9yIG5hbWUsZGlnZXN0IGluIHJlYWQoaGVyZS8ncGF5bG9hZF9tYW5pZmVzdC5qc29uJykuaXRlbXMoKToKICAgICAgICAgICAgcCA9IGhlcmUvbmFtZQogICAgICAgICAgICBhc3NlcnQgbm90IHAuaXNfc3ltbGluaygpIGFuZCBwLmlzX2ZpbGUoKSBhbmQgc2hhKHApPT1kaWdlc3QsICdQYXlsb2FkIGNoYW5nZWQ6ICcrbmFtZQogICAgICAgIGlmIE1BUktFUi5leGlzdHMoKTogcmFpc2UgUnVudGltZUVycm9yKCdHUFUgcXVhcmFudGluZSByZXF1aXJlcyBvcGVyYXRvciByZWNvbmNpbGlhdGlvbicpCiAgICAgICAgbG9ja2ZkID0gb3Mub3BlbihMT0NLLG9zLk9fQ1JFQVR8b3MuT19SRFdSfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICAgICAgZmNudGwuZmxvY2sobG9ja2ZkLGZjbnRsLkxPQ0tfRVh8ZmNudGwuTE9DS19OQikKICAgICAgICBpZiBNQVJLRVIuZXhpc3RzKCk6IHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVIHF1YXJhbnRpbmUgYXBwZWFyZWQgd2hpbGUgbG9ja2luZycpCiAgICAgICAgdHJlZS5wcmVwYXJlKCk7IHRyZWUuYmVnaW5fcXVhcmFudGluZShNQVJLRVIpCiAgICAgICAgb2ZmaWNpYWwgPSByZWFkKGhlcmUvJ29mZmljaWFsX21hbmlmZXN0Lmpzb24nKQogICAgICAgIHZlcmlmeV9vZmZpY2lhbChyZXBvLG9mZmljaWFsKQogICAgICAgIGVudiA9IGRpY3Qob3MuZW52aXJvbixIRl9IVUJfT0ZGTElORT0nMScsSEZfREFUQVNFVFNfT0ZGTElORT0nMScsVFJBTlNGT1JNRVJTX09GRkxJTkU9JzEnLAogICAgICAgICAgICAgICAgICAgUFlUSE9ORE9OVFdSSVRFQllURUNPREU9JzEnLFBZVEhPTlVOQlVGRkVSRUQ9JzEnKQogICAgICAgIGVudi5wb3AoJ1NPVVBfUk9UQVJZX0NPREVHRU5fRElSJyxOb25lKQogICAgICAgIHdpdGggRXhwb3J0ZXIocnVuLGV4cG9ydCk6CiAgICAgICAgICAgIHByb2JlID0gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctQicsc3RyKGhlcmUvJ3J1bnRpbWVfcHJvYmUucHknKSxzdHIocnVuLydydW50aW1lX3Byb2JlLmpzb24nKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY3dkPXJlcG8sZW52PWVudixsb2c9cnVuLydydW50aW1lX3Byb2JlLmxvZycsdGltZW91dD00MCxsb2NrX2ZkPWxvY2tmZCkKICAgICAgICAgICAgc3RhdHVzWydwcm9jZXNzZXMnXVsncnVudGltZV9wcm9iZSddID0gcHJvYmUKICAgICAgICAgICAgaWYgcHJvYmVbJ2V4aXRfY29kZSddIG9yIG5vdCByZWFkKHJ1bi8ncnVudGltZV9wcm9iZS5qc29uJykuZ2V0KCdwYXNzZWQnKToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignRXhhY3QgZGVwbG95ZWQgcnVudGltZSBnYXRlIGZhaWxlZDsgbm8gYmVuY2htYXJrJykKICAgICAgICAgICAgZGVmIGlkbGUobGFiZWwpOgogICAgICAgICAgICAgICAgZm9sZGVyPXJ1bi8oJ2lkbGVfJytsYWJlbCk7Zm9sZGVyLm1rZGlyKGV4aXN0X29rPUZhbHNlKQogICAgICAgICAgICAgICAgcmVzdWx0PWlkbGVfY2hlY2soZW52LHJlcG8sZm9sZGVyLGxvY2tmZCkKICAgICAgICAgICAgICAgIHN0YXR1c1sncHJvY2Vzc2VzJ11bJ2lkbGVfJytsYWJlbF09cmVzdWx0CiAgICAgICAgICAgICAgICBpZiByZXN1bHRbJ2V4aXRfY29kZSddOnJhaXNlIFJ1bnRpbWVFcnJvcignSWRsZSBnYXRlIGZhaWxlZDogJytsYWJlbCkKICAgICAgICAgICAgaWRsZSgnYmVmb3JlX3ByZWZsaWdodCcpCiAgICAgICAgICAgICMgU291cmNlLWJvdW5kIG5hdGl2ZSBBTVAvZnVsbC12b2NhYnVsYXJ5IG51bWVyaWNhbCBnYXRlIGluIGEgc2VwYXJhdGUgcHJvY2Vzcy4KICAgICAgICAgICAgcHJlZW52ID0gZGljdChlbnYsVFJJVE9OX0NBQ0hFX0RJUj1zdHIocnVuLydwcmVmbGlnaHRfdHJpdG9uJyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgVE9SQ0hJTkRVQ1RPUl9DQUNIRV9ESVI9c3RyKHJ1bi8ncHJlZmxpZ2h0X2luZHVjdG9yJykpCiAgICAgICAgICAgIHByZSA9IHByb2Nlc3NfZ3VhcmQucnVuKFtzeXMuZXhlY3V0YWJsZSwnLUInLHN0cihoZXJlLyd3b3JrZXIucHknKSwnLS1tZXRob2QnLCdudW1lcmljYWwnLCctLW91dCcsc3RyKHJ1bi8ncHJlZmxpZ2h0Lmpzb24nKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGN3ZD1oZXJlLGVudj1wcmVlbnYsbG9nPXJ1bi8ncHJlZmxpZ2h0LmxvZycsdGltZW91dD04MCxsb2NrX2ZkPWxvY2tmZCkKICAgICAgICAgICAgc3RhdHVzWydwcm9jZXNzZXMnXVsncHJlZmxpZ2h0J10gPSBwcmUKICAgICAgICAgICAgaWYgcHJlWydleGl0X2NvZGUnXSBvciByZWFkKHJ1bi8ncHJlZmxpZ2h0Lmpzb24nKS5nZXQoJ3N0YXR1cycpIT0nUEFTU0VEX0hFQURfQ09NUE9ORU5UX1BSRUZMSUdIVF9PTkxZJzoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignTmF0aXZlIEFNUCBmdWxsLXZvY2FidWxhcnkgaGVhZCBudW1lcmljYWwgZ2F0ZSBmYWlsZWQ7IG5vIGJlbmNobWFyaycpCiAgICAgICAgICAgIHN0YXR1c1snc3RhdHVzJ109J1JVTk5JTkdfTUlDUk9CRU5DSF9PTkxZJzt3cml0ZShleHBvcnQvJ3N0YXR1cy5qc29uJyxzdGF0dXMpCiAgICAgICAgICAgIGlkbGUoJ3ByZWZsaWdodCcpCiAgICAgICAgICAgIGZvciBtZXRob2QgaW4gc3RhdHVzWydtZXRob2RzX2luX2V4ZWN1dGlvbl9vcmRlciddOgogICAgICAgICAgICAgICAgY2FjaGU9cnVuLyd0cml0b25fY2FjaGUnL21ldGhvZDtjYWNoZS5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9RmFsc2UpCiAgICAgICAgICAgICAgICBjaGlsZGVudj1kaWN0KGVudixUUklUT05fQ0FDSEVfRElSPXN0cihjYWNoZSksVE9SQ0hJTkRVQ1RPUl9DQUNIRV9ESVI9c3RyKHJ1bi8naW5kdWN0b3JfY2FjaGUnL21ldGhvZCkpCiAgICAgICAgICAgICAgICBwcm9jZXNzPXByb2Nlc3NfZ3VhcmQucnVuKFtzeXMuZXhlY3V0YWJsZSwnLUInLHN0cihoZXJlLyd3b3JrZXIucHknKSwnLS1tZXRob2QnLG1ldGhvZCwnLS1vdXQnLHN0cihydW4vKG1ldGhvZCsnLmpzb24nKSldLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjd2Q9aGVyZSxlbnY9Y2hpbGRlbnYsbG9nPXJ1bi8obWV0aG9kKycubG9nJyksdGltZW91dD04MCxsb2NrX2ZkPWxvY2tmZCkKICAgICAgICAgICAgICAgIHN0YXR1c1sncHJvY2Vzc2VzJ11bbWV0aG9kXT1wcm9jZXNzCiAgICAgICAgICAgICAgICBpZiBwcm9jZXNzWydleGl0X2NvZGUnXSBvciByZWFkKHJ1bi8obWV0aG9kKycuanNvbicpKS5nZXQoJ3N0YXR1cycpIT0nUEFTU0VEX0hFQURfQ09NUE9ORU5UX1BSRUZMSUdIVF9PTkxZJzoKICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0luY29tcGxldGUvZmFpbGVkL2NlbnNvcmVkIG1pY3JvYmVuY2htYXJrOiAnK21ldGhvZCkKICAgICAgICAgICAgICAgIGlkbGUobWV0aG9kKQogICAgICAgICAgICAgICAgd3JpdGUoZXhwb3J0LydzdGF0dXMuanNvbicsc3RhdHVzKQogICAgICAgICAgICB2ZXJpZnlfb2ZmaWNpYWwocmVwbyxvZmZpY2lhbCkKICAgICAgICAgICAgcmVwb3J0cz17bWV0aG9kOnJlYWQocnVuLyhtZXRob2QrJy5qc29uJykpIGZvciBtZXRob2QgaW4gc3RhdHVzWydtZXRob2RzX2luX2V4ZWN1dGlvbl9vcmRlciddfQogICAgICAgICAgICBhc3NlcnQgYWxsKGxlbihyWydyb3dzJ10pPT0yIGFuZCByWyd0cmFpbmluZ19leGVjdXRlZCddIGlzIEZhbHNlIGFuZCByWydjb21wbGV0ZV9maXhlZF90YXNrJ10gaXMgRmFsc2UgZm9yIHIgaW4gcmVwb3J0cy52YWx1ZXMoKSkKICAgICAgICAgICAgd3JpdGUocnVuLydjb21wYXJpc29uLmpzb24nLGRpY3Qoc2NvcGU9J0hFQURfQ09NUE9ORU5UX1BSRUZMSUdIVF9PTkxZJyxyZXBvcnRzPXJlcG9ydHMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2NvcmU9Tm9uZSxhZG1pc3Npb249J1VOS05PV04nLGNvbXBsZXRlX2ZpeGVkX3Rhc2s9RmFsc2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbm90ZT0nSXNvbGF0ZWQgc3ludGhldGljIGhlYWQrbmF0aXZlLXJlcXVpcmVkLW1ldHJpY3MrYWRqb2ludCBvbmx5LiBOYXRpdmUvY2h1bmsgYXJlIGZyZXNoIHByb2Nlc3NlcyB3aXRoIHNhbWUgc3ludGhldGljIGZyb3plbiBGUDMyIFBhcmFtZXRlci4gQWJzb2x1dGUgYWxsb2NhdG9yIHBlYWtzIGluY2x1ZGUgY29tcG9uZW50IFcvaW5wdXRzLCBub3QgYSBtb2RlbCBvciBmdWxsIHRyYWluaW5nIHBlYWsuIE5vIHRhc2sgc3BlZWR1cCwgM0dCLW1vZGVsIGNsYWltLCBhZG1pc3Npb24gY3JlZGl0LCBvciBvd25lcnNoaXAtc2hhcmUgaW5mZXJlbmNlLiBPcmlnaW5hbCBDRSBrZXJuZWxzIHVuY2hhbmdlZDsgdXNlZnVsIGZ1bGwtdGFzayBzaGFyZSBpcyB1bm1lYXN1cmVkLicpKQogICAgICAgICAgICBzdGF0dXNbJ3N0YXR1cyddPSdQQVNTRURfTUlDUk9CRU5DSF9PTkxZJwogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHN0YXR1cy51cGRhdGUoc3RhdHVzPSdCTE9DS0VEX0ZBSUxFRF9PUl9DRU5TT1JFRF9NSUNST0JFTkNIX09OTFknLGVycm9yPXJlcHIoZXhjKSkKICAgICAgICBpZiBsb2NrZmQgaXMgbm90IE5vbmU6dHJlZS5xdWFyYW50aW5lKCdUb2tlbi1jaHVuayBoZWFkIGNvbXBvbmVudCBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgZmluYWxseToKICAgICAgICBzdGF0dXNbJ29yY2hlc3RyYXRvcl93YWxsX3MnXT10aW1lLm1vbm90b25pYygpLXN0YXJ0ZWQKICAgICAgICBleHBvcnRfc21hbGwocnVuLGV4cG9ydCk7d3JpdGUoZXhwb3J0LydzdGF0dXMuanNvbicsc3RhdHVzKQogICAgICAgIGlmIGxvY2tmZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgdHJlZS5maW5pc2hfcXVhcmFudGluZSgpO29zLmNsb3NlKGxvY2tmZCkKICAgIHByaW50KGpzb24uZHVtcHMoeydzY29wZSc6J0hFQURfQ09NUE9ORU5UX1BSRUZMSUdIVF9PTkxZJywnc3RhdHVzJzpzdGF0dXNbJ3N0YXR1cyddLCdleHBvcnQnOnN0cihleHBvcnQpLAogICAgICAgICAgICAgICAgICAgICAgJ3Njb3JlJzpOb25lLCdhZG1pc3Npb24nOidVTktOT1dOJ30pLGZsdXNoPVRydWUpCiAgICByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgc3RhdHVzWydzdGF0dXMnXT09J1BBU1NFRF9NSUNST0JFTkNIX09OTFknIGVsc2UgMSkKCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOm1haW4oKQo=', 'source_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICJjZmZmNTE4YmU1YWNmNWY0OTNhYjg0NTAwMDJhMDMyMTc0MGI0Yzg3NDU2Nzk1NDk5ZTI4ODk4NTk1MjY2Zjk0IiwKICAic29sdXRpb24vY2VfdGlsZS5weSI6ICIyYmFlYTFmNGI3YzU2NzM3OWUzNjc0Yjc2MjA0ZDM5ZTlmYjEyOGViNmNmY2RhOWE4ZDhlZGViYmZkMjMwZTdiIiwKICAic29sdXRpb24vY29udHJhY3RzLnB5IjogIjU2MWRkM2Y3OTBmNjc0NGJmZjg3OGJlNjY4MGY5NTdhYmQyOTE4ZTFkMzUwZTdkMzc5NGM5M2ZkYmJlMTRjMGMiLAogICJzb2x1dGlvbi9vcmlnaW5hbC9fX2luaXRfXy5weSI6ICJlM2IwYzQ0Mjk4ZmMxYzE0OWFmYmY0Yzg5OTZmYjkyNDI3YWU0MWU0NjQ5YjkzNGNhNDk1OTkxYjc4NTJiODU1IiwKICAic29sdXRpb24vb3JpZ2luYWwvbWF0ZXJpYWxpemVkX2NlLnB5IjogIjY1MTljOGYxZGRjZTdkYjI3MGZiZTQyNWE4ZTg3NzFhZWNkYTNmMjk0ZWNjODM4MzlkNDc1OTUzMWZkMWRjZTEiLAogICJzb2x1dGlvbi9vcmlnaW5hbC9tZXRyaWNfc3VtbWFyeS5weSI6ICI4YzVlZGVhMDBjN2YwYjQyNTAxYzcxZTM0YTk4NmRlMDVjMjhjM2M1NmVkZjRkNjE4MWQyZGRhZjI0MTQyY2Y0IiwKICAic29sdXRpb24vb3JpZ2luYWwvbWl4ZWRfcm93cy5weSI6ICIwOTExNTdjZGI5MDRiMGEyZmVlMDlkNzIwODYyYjM3MGQ4ZjcyZjlkODI0MDIzZDM3ZWRiZGY5YzVlMTcyNmIyIiwKICAic29sdXRpb24vdG9rZW5fY2h1bmtfaGVhZC5weSI6ICI1OGVjZjViOWE4M2Y5MWYxMjg2MTZkOTU0ZjU5YTI3ZjE4OTk2OWQ2MmU0NGFkNjBlZDJjNjdkZjIwMTFmZGIzIiwKICAic29sdXRpb24vdmVyaWZ5X3RpbGUucHkiOiAiZjRmNjI1MTk4N2I2MTk5MzY1Yjg1M2Q0ODk3OWExMzZlYzJkYzI2MjZiZWQ2OGYzNzRhZTI5NDBlZTYwNDdhMSIsCiAgInNoYXJlZF9jb25zdW1lci9fX2luaXRfXy5weSI6ICJlM2IwYzQ0Mjk4ZmMxYzE0OWFmYmY0Yzg5OTZmYjkyNDI3YWU0MWU0NjQ5YjkzNGNhNDk1OTkxYjc4NTJiODU1IiwKICAic2hhcmVkX2NvbnN1bWVyL2JyaWRnZV9zdGF0ZS5weSI6ICJlM2JhNjQ0NDZhOGJiM2ViOGNkNmVjMzFiNGU1ZDAwMThhNDdjOGYzYjkwZDAwZGE5NTVkYzJlNDQ0ZWIxOGU5IiwKICAic2hhcmVkX2NvbnN1bWVyL2NvbnN1bWVyLnB5IjogImYwMThhOWI2ODA4ZDc2MzgxYTIwNTU4MGU4YTJmZjlkMmMwZWI3MDZmYTM0Yzg0MzRmZTQxYjBhZDk5ZGEyYmMiLAogICJvbGRfc29sdXRpb24vX19pbml0X18ucHkiOiAiOTQ3OTRkNGY0YWI0YmE3NmViNThmNzMzODMwMDhlYTYwNWUzYTZlMjVhZmI5MWY1NTA3MjNiNWM4MDFlMDNhYSIsCiAgIm9sZF9zb2x1dGlvbi90b2tlbl9jaHVua19oZWFkLnB5IjogImM3NWQzM2I3YWRmYTQyZTY1OWMwNDljZTk5MzNhNTc1MDEzMjRiYWI3ZDdhZjhiM2JiZGEwODMyYmNlNzNmYjUiCn0K', 'payload_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICJjZmZmNTE4YmU1YWNmNWY0OTNhYjg0NTAwMDJhMDMyMTc0MGI0Yzg3NDU2Nzk1NDk5ZTI4ODk4NTk1MjY2Zjk0IiwKICAic29sdXRpb24vY2VfdGlsZS5weSI6ICIyYmFlYTFmNGI3YzU2NzM3OWUzNjc0Yjc2MjA0ZDM5ZTlmYjEyOGViNmNmY2RhOWE4ZDhlZGViYmZkMjMwZTdiIiwKICAic29sdXRpb24vY29udHJhY3RzLnB5IjogIjU2MWRkM2Y3OTBmNjc0NGJmZjg3OGJlNjY4MGY5NTdhYmQyOTE4ZTFkMzUwZTdkMzc5NGM5M2ZkYmJlMTRjMGMiLAogICJzb2x1dGlvbi9vcmlnaW5hbC9fX2luaXRfXy5weSI6ICJlM2IwYzQ0Mjk4ZmMxYzE0OWFmYmY0Yzg5OTZmYjkyNDI3YWU0MWU0NjQ5YjkzNGNhNDk1OTkxYjc4NTJiODU1IiwKICAic29sdXRpb24vb3JpZ2luYWwvbWF0ZXJpYWxpemVkX2NlLnB5IjogIjY1MTljOGYxZGRjZTdkYjI3MGZiZTQyNWE4ZTg3NzFhZWNkYTNmMjk0ZWNjODM4MzlkNDc1OTUzMWZkMWRjZTEiLAogICJzb2x1dGlvbi9vcmlnaW5hbC9tZXRyaWNfc3VtbWFyeS5weSI6ICI4YzVlZGVhMDBjN2YwYjQyNTAxYzcxZTM0YTk4NmRlMDVjMjhjM2M1NmVkZjRkNjE4MWQyZGRhZjI0MTQyY2Y0IiwKICAic29sdXRpb24vb3JpZ2luYWwvbWl4ZWRfcm93cy5weSI6ICIwOTExNTdjZGI5MDRiMGEyZmVlMDlkNzIwODYyYjM3MGQ4ZjcyZjlkODI0MDIzZDM3ZWRiZGY5YzVlMTcyNmIyIiwKICAic29sdXRpb24vdG9rZW5fY2h1bmtfaGVhZC5weSI6ICI1OGVjZjViOWE4M2Y5MWYxMjg2MTZkOTU0ZjU5YTI3ZjE4OTk2OWQ2MmU0NGFkNjBlZDJjNjdkZjIwMTFmZGIzIiwKICAic29sdXRpb24vdmVyaWZ5X3RpbGUucHkiOiAiZjRmNjI1MTk4N2I2MTk5MzY1Yjg1M2Q0ODk3OWExMzZlYzJkYzI2MjZiZWQ2OGYzNzRhZTI5NDBlZTYwNDdhMSIsCiAgInNoYXJlZF9jb25zdW1lci9fX2luaXRfXy5weSI6ICJlM2IwYzQ0Mjk4ZmMxYzE0OWFmYmY0Yzg5OTZmYjkyNDI3YWU0MWU0NjQ5YjkzNGNhNDk1OTkxYjc4NTJiODU1IiwKICAic2hhcmVkX2NvbnN1bWVyL2JyaWRnZV9zdGF0ZS5weSI6ICJlM2JhNjQ0NDZhOGJiM2ViOGNkNmVjMzFiNGU1ZDAwMThhNDdjOGYzYjkwZDAwZGE5NTVkYzJlNDQ0ZWIxOGU5IiwKICAic2hhcmVkX2NvbnN1bWVyL2NvbnN1bWVyLnB5IjogImYwMThhOWI2ODA4ZDc2MzgxYTIwNTU4MGU4YTJmZjlkMmMwZWI3MDZmYTM0Yzg0MzRmZTQxYjBhZDk5ZGEyYmMiLAogICJvbGRfc29sdXRpb24vX19pbml0X18ucHkiOiAiOTQ3OTRkNGY0YWI0YmE3NmViNThmNzMzODMwMDhlYTYwNWUzYTZlMjVhZmI5MWY1NTA3MjNiNWM4MDFlMDNhYSIsCiAgIm9sZF9zb2x1dGlvbi90b2tlbl9jaHVua19oZWFkLnB5IjogImM3NWQzM2I3YWRmYTQyZTY1OWMwNDljZTk5MzNhNTc1MDEzMjRiYWI3ZDdhZjhiM2JiZGEwODMyYmNlNzNmYjUiLAogICJwcm9jZXNzX2d1YXJkLnB5IjogIjgyNDY1OTQzYTNhOWFmMzgzMWE1Mzk5YTNhNTVkMDlhYzI5MGJhNzM2NWM0ZTgwZDZiYzVlNzlmYWI2MmE2ZmYiLAogICJvd25lZF9wcm9jZXNzX3RyZWUucHkiOiAiNjkzZGRlNzA4ZGE1YzA2MDUzYWYyMzk0NjM5NTIxMTVlZjVlODg2NWY1MGJjYmM1MWEyMDM4MjM3Y2E1N2Q2OSIsCiAgInJ1bnRpbWVfcHJvYmUucHkiOiAiZWZlMWI3Y2YzZjNjZjFlMzdiYmMxNjlhODQ2ODg1OWM1MTI3ZWMzMjgxYTM3ZTM5MDY4NzAzYjJlYjNjMTQ4MCIsCiAgIm9mZmljaWFsX21hbmlmZXN0Lmpzb24iOiAiZDAxM2UxNjMzOTQwYzI3ZDcwOGUyNDc0YTgyODA5OTAwMDBlMzMwMThlODg5ZWQxNDY1NWM4NTYzMjlmZjFkOCIsCiAgInJldmlld2VkX3N1cHBvcnQucHkiOiAiN2M1ZDYwMmQwNzEzODNlZjA3ZmRkZDk5NjcwZWQxY2NlMzE0MWZlYWJkNWFmODM5NGYwNTNjNmI0ZTA3MTIyYyIsCiAgIndvcmtlci5weSI6ICI3MDk0NzMzN2IyZGMzYjJiMTk5ZDczOTRiZDJkNjUzMGI2OWNjM2UyYjAzMzM2MjkwYjMyZjA3OWZkMDc0ZjcyIiwKICAibmF0aXZlX3NvdXJjZV9tYW5pZmVzdC5qc29uIjogIjEyZGU2OTkxZmQ0MWYyZGVhNmNhODkzMDA3OWYxMGJhMmNkN2M3NTJmOTU1ODM2ZTM3NzRmMzk5NzY2NmE4MTYiLAogICJzaGFyZWRfY29uc3VtZXJfbWFuaWZlc3QuanNvbiI6ICI3YzMyODMzYmZlYzdmOGQxN2VjNjc3MWU5NjgzYjkyMjEwOWYyZTFjMDllN2U0NzM1OWE4NmMwM2FkMGZmMzk1IiwKICAidGlsZV9zb3VyY2VfbWFuaWZlc3QuanNvbiI6ICI1ZTliMjc2N2FmYTk0YTRlODViZDE4ZmMzYmRmNjEwYmY1ZmY1NjJkODM2YWY5Zjk1ZGI4ZWJhZDZjZTZkODk0IiwKICAiZHJpdmVyLnB5IjogImU2NGE1ODQ3NWI0NThlODFkYzg1NmM2Nzc0OTM0ZjQ0MGMzN2FmNjg1YTY2NjUyYjk0MDljMDg0MWY5ZWY2MjIiLAogICJzb3VyY2VfbWFuaWZlc3QuanNvbiI6ICJmMjU2NjJkYTU2YTZkZGE4OTQzYWE5ZWNiZDQ1ZWFkZWZkNmZlZGM3MjEzZTk2YWZkMjY1ODExZWIzMWU5NjJlIgp9Cg=='}
PAYLOAD_SHA256 = {'solution/__init__.py': 'cfff518be5acf5f493ab8450002a0321740b4c87456795499e28898595266f94', 'solution/ce_tile.py': '2baea1f4b7c567379e3674b76204d39e9fb128eb6cfcda9a8d8edebbfd230e7b', 'solution/contracts.py': '561dd3f790f6744bff878be6680f957abd2918e1d350e7d3794c93fdbbe14c0c', 'solution/original/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'solution/original/materialized_ce.py': '6519c8f1ddce7db270fbe425a8e8771aecda3f294ecc83839d4759531fd1dce1', 'solution/original/metric_summary.py': '8c5edea00c7f0b42501c71e34a986de05c28c3c56edf4d6181d2ddaf24142cf4', 'solution/original/mixed_rows.py': '091157cdb904b0a2fee09d720862b370d8f72f9d824023d37edbdf9c5e1726b2', 'solution/token_chunk_head.py': '58ecf5b9a83f91f128616d954f59a27f189969d62e44ad60ed2c67df2011fdb3', 'solution/verify_tile.py': 'f4f6251987b6199365b853d48979a136ec2dc2626bed68f374ae2940ee6047a1', 'shared_consumer/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'shared_consumer/bridge_state.py': 'e3ba64446a8bb3eb8cd6ec31b4e5d0018a47c8f3b90d00da955dc2e444eb18e9', 'shared_consumer/consumer.py': 'f018a9b6808d76381a205580e8a2ff9d2c0eb706fa34c8434fe41b0ad99da2bc', 'old_solution/__init__.py': '94794d4f4ab4ba76eb58f73383008ea605e3a6e25afb91f550723b5c801e03aa', 'old_solution/token_chunk_head.py': 'c75d33b7adfa42e659c049ce9933a57501324bab7d7af8b3bbda0832bce73fb5', 'process_guard.py': '82465943a3a9af3831a5399a3a55d09ac290ba7365c4e80d6bc5e79fab62a6ff', 'owned_process_tree.py': '693dde708da5c06053af239463952115ef5e8865f50bcbc51a2038237ca57d69', 'runtime_probe.py': 'efe1b7cf3f3cf1e37bbc169a8468859c5127ec3281a37e39068703b2eb3c1480', 'official_manifest.json': 'd013e1633940c27d708e2474a8280990000e33018e889ed14655c856329ff1d8', 'reviewed_support.py': '7c5d602d071383ef07fddd99670ed1cce3141feabd5af8394f053c6b4e07122c', 'worker.py': '70947337b2dc3b2b199d7394bd2d6530b69cc3e2b03336290b32f079fd074f72', 'native_source_manifest.json': '12de6991fd41f2dea6ca8930079f10ba2cd7c752f955836e3774f3997666a816', 'shared_consumer_manifest.json': '7c32833bfec7f8d17ec6771e9683b922109f2e1c09e7e47359a86c03ad0ff395', 'tile_source_manifest.json': '5e9b2767afa94a4e85bd18fc3bdf610bf5ff562d836af9f95db8ebad6ce6d894', 'driver.py': 'e64a58475b458e81dc856c6774934f440c37af685a66652b9409c0841f9ef622', 'source_manifest.json': 'f25662da56a6dda8943aa9ecbd45eadefd6fedc7213e96afd265811eb31e962e', 'payload_manifest.json': 'd992f90cea406d987d2d643d1a512fe9d95cad8fdb8d99904ff69c5733320b4a'}
STAGED=Path(tempfile.mkdtemp(prefix='token-chunk-ce-head-component-',dir='/kaggle/working'))
for name,encoded in PAYLOAD.items():
    parts=Path(name).parts
    assert not Path(name).is_absolute() and '..' not in parts
    raw=base64.b64decode(encoded,validate=True)
    assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA256[name]
    target=STAGED/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
print('Prepared MICROBENCH_ONLY payload:',STAGED)


In [ ]:
import subprocess,sys,signal
process=subprocess.Popen([sys.executable,'-B',str(STAGED/'driver.py')],start_new_session=True)
try:
    return_code=process.wait()
except BaseException:
    if process.poll() is None:process.send_signal(signal.SIGTERM)
    while process.poll() is None:
        try:process.wait()
        except KeyboardInterrupt:pass
    raise
if return_code:raise RuntimeError('MICROBENCH_ONLY failed or censored; inspect exported status. No automatic retry.')
